# SkyGuard AI — GPU Iteration 9

## Domain-invariant calibration and multi-seed confirmation stress test

Iteration 8 delivered a large DWD transfer gain, but it was correctly not promoted: only 8 of 14 gates passed. Confirmation precision, India point F1, India weather-event recognition, worst-climate weather F1 and the fault-to-weather cap still failed. Feature importance also exposed a likely domain shortcut: absolute rolling climate levels and station spacing were among the strongest predictors.

This notebook starts from the frozen Iteration 8 development checkpoint and directly addresses those failures:

1. Remove absolute climate-level and station-spacing shortcuts from a new residual-only LightGBM contract.
2. Preserve physical limit rules separately so extreme observations are still caught.
3. Calibrate full-tree, residual-tree, Isolation Forest and causal LSTM scores with an L2-regularized meta-model that never receives station ID, domain or climate labels.
4. Use a causal per-station score normalization based only on prior observations.
5. Split January–April tune data chronologically into calibration and policy halves.
6. Freeze thresholds before discovery, confirmation and multi-seed stress evaluation.
7. Add two independent DWD 2023 injection seeds; together with the Iteration 8 seed, each family/climate is tested three times without changing anomaly prevalence.
8. Report root-cause macro F1, per-family episode recall, seed confidence intervals and every rejected ablation.

DWD/NOAA 2024 and all 2025 observations or labels remain sealed.


## Run instructions

1. Keep the two existing bundles in `/content/drive/MyDrive/SkyGuard_AI_GPU/`:
   - `SkyGuard_GPU_Data_Bundle.zip`
   - `SkyGuard_Iteration8_Development_Data_Bundle.zip`
2. Keep the completed Iteration 8 experiment folder in Drive. The notebook reuses its cached features and models.
3. Do **not** upload or extract any locked 2024 bundle and do not add any 2025 file.
4. Select **Runtime → Change runtime type → T4 GPU**.
5. Leave `UNLOCK_FINAL_TESTS=False`, `REUSE_SAVED_MODELS=True` and `RUN_ITER9_STRESS=True`.
6. Run all cells in order. First execution is expected to take roughly 90–240 minutes. Each stress seed is cached, so a disconnected runtime can safely restart and reuse completed work.
7. Return the Iteration 9 JSON/CSV files printed by the final cell. Do not open a locked test unless every promotion gate passes and Codex audits the outputs first.


In [3]:
!pip -q install catboost==1.2.10 lightgbm==4.6.0 scikit-learn==1.7.2 pyarrow==21.0.0 psutil==7.0.0
from google.colab import drive
drive.mount('/content/drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 25.6 MB/s eta 0:00:00
Mounted at /content/drive


In [4]:
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive/SkyGuard_AI_GPU')
BUNDLE_ZIP=DRIVE_ROOT/'SkyGuard_GPU_Data_Bundle.zip'
DATA_ROOT=DRIVE_ROOT/'SkyGuard_GPU_Data_Bundle'
ITER1=DRIVE_ROOT/'experiments'/'iteration_01_detection'
ARTIFACT_ROOT=DRIVE_ROOT/'experiments'/'iteration_02_weak_fault_rescue'
ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)

UNLOCK_FINAL_TESTS=False
REUSE_SAVED_MODELS=True
SEEDS=[17,29,41,53,67]
SPECIALIST_SEEDS=[17,41,67]
print('Iteration 1:',ITER1)
print('Iteration 2:',ARTIFACT_ROOT)


Iteration 1: /content/drive/MyDrive/SkyGuard_AI_GPU/experiments/iteration_01_detection
Iteration 2: /content/drive/MyDrive/SkyGuard_AI_GPU/experiments/iteration_02_weak_fault_rescue


In [5]:
import os,json,time,math,hashlib,zipfile,warnings,platform
import joblib,numpy as np,pandas as pd,matplotlib.pyplot as plt,psutil
from IPython.display import display
from sklearn.metrics import average_precision_score,precision_score,recall_score,f1_score,confusion_matrix
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
import torch

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',50)
pd.set_option('display.float_format',lambda x:f'{x:,.5f}')
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print({'python':platform.python_version(),'device':DEVICE,
       'gpu':torch.cuda.get_device_name(0) if DEVICE=='cuda' else None,
       'ram_gb':round(psutil.virtual_memory().total/2**30,2)})
assert DEVICE=='cuda','Select a GPU runtime.'

if not DATA_ROOT.exists():
    assert BUNDLE_ZIP.exists(),f'Missing {BUNDLE_ZIP}'
    with zipfile.ZipFile(BUNDLE_ZIP) as archive: archive.extractall(DRIVE_ROOT)


{'python': '3.13.15', 'device': 'cuda', 'gpu': 'Tesla T4', 'ram_gb': 12.67}


## 1. Load exact development partitions and Phase 10 contract

In [6]:
FEATURE_DIR=DATA_ROOT/'data'/'features_phase10'
BASELINE_FILE=DATA_ROOT/'models'/'phase10_final.joblib'

def load_table(name):
    frame=pd.read_csv(FEATURE_DIR/f'{name}_features.csv.gz',low_memory=False)
    frame['station_id']=frame.station_id.astype(str)
    frame['emitted_timestamp_utc']=pd.to_datetime(frame.emitted_timestamp_utc,utc=True)
    frame['episode_id']=frame.episode_id.fillna('').astype(str)
    return frame.sort_values(['station_id','emitted_timestamp_utc','row_id']).reset_index(drop=True)

train=load_table('train'); validation=load_table('validation')
train['dev_split']='train'
ts=validation.emitted_timestamp_utc
validation['dev_split']=np.select(
    [ts<'2023-05-01',ts<'2023-07-01',ts<'2023-10-01'],
    ['tune_model','block_may_jun','block_jul_sep'],default='block_oct_dec')
episode_part=(validation.loc[validation.episode_id.ne('')].sort_values('emitted_timestamp_utc')
              .groupby('episode_id').dev_split.first())
mask=validation.episode_id.ne('')
validation.loc[mask,'dev_split']=validation.loc[mask,'episode_id'].map(episode_part)
dev=pd.concat([train,validation],ignore_index=True)
del train,validation
dev=dev.loc[dev.available_to_detector.eq(1)].copy().reset_index(drop=True)

bundle=joblib.load(BASELINE_FILE); FEATURES=list(bundle['event_features'])
assert len(FEATURES)==108 and not ({'temperature_dewpoint_spread_c','hour_sin','hour_cos','day_of_year_sin','day_of_year_cos'}&set(FEATURES))
assert dev.loc[dev.episode_id.ne('')].groupby('episode_id').dev_split.nunique().max()==1
display(dev.groupby('dev_split').agg(rows=('row_id','size'),fault_rows=('is_anomaly','sum'),
                                     episodes=('episode_id',lambda s:s[s.ne('')].nunique()),stations=('station_id','nunique')))


,rows,fault_rows,episodes,stations
dev_split,,,,
block_jul_sep,46803,375,28,20
block_may_jun,29012,243,23,20
block_oct_dec,47599,471,37,20
train,182122,2270,185,20
tune_model,57894,445,36,20


## 2. Exact Phase 10 and Iteration 1 CatBoost scores

In [7]:
event_model=bundle['event_model']; classes=list(event_model.classes_)
fault_index=classes.index('sensor_fault'); weather_index=classes.index('genuine_weather')
proba=event_model.predict_proba(dev[FEATURES].replace([np.inf,-np.inf],np.nan))
dev['phase10_fault']=proba[:,fault_index]; dev['phase10_weather']=proba[:,weather_index]
PHASE10_THRESHOLD=float(bundle['policy']['known_station']['threshold'])

def load_cat_models(prefix,seeds):
    models=[]
    for seed in seeds:
        path=ITER1/f'{prefix}_seed{seed}.cbm'
        assert path.exists(),f'Missing {path}. Run Iteration 1 CatBoost cell or restore Drive artifacts.'
        model=CatBoostClassifier(); model.load_model(path); models.append(model)
    return models

fault_models=load_cat_models('cat_fault',SEEDS)
weather_models=load_cat_models('cat_weather',SEEDS)
X=dev[FEATURES]
fault_seed_scores=np.column_stack([m.predict_proba(X)[:,1] for m in fault_models])
weather_seed_scores=np.column_stack([m.predict_proba(X)[:,1] for m in weather_models])
dev['cat_fault_mean']=fault_seed_scores.mean(1)
dev['cat_fault_median']=np.median(fault_seed_scores,axis=1)
dev['cat_weather_mean']=weather_seed_scores.mean(1)
dev['cat_weather_median']=np.median(weather_seed_scores,axis=1)
print('Loaded',len(fault_models),'fault and',len(weather_models),'weather models.')


Loaded 5 fault and 5 weather models.


## 3. Strict metrics and fast causal alert policies

In [8]:
def point_metrics(y,p,score):
    y=np.asarray(y,int); p=np.asarray(p,bool); score=np.asarray(score,float)
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    return {'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn),
            'precision':precision_score(y,p,zero_division=0),'recall':recall_score(y,p,zero_division=0),
            'f1':f1_score(y,p,zero_division=0),'auprc':average_precision_score(y,score)}

def predicted_events(group,pred_col):
    g=group.sort_values('emitted_timestamp_utc'); times=g.emitted_timestamp_utc.tolist(); pred=g[pred_col].to_numpy(bool)
    dt=g.emitted_timestamp_utc.diff().dt.total_seconds().div(60); positive=dt[dt.gt(0)]
    gap=max(60,2.5*(float(positive.median()) if len(positive) else 60))
    events=[]; start=None; previous=None
    for pos in np.flatnonzero(pred):
        separated=(previous is None or pos!=previous+1 or (times[pos]-times[previous]).total_seconds()/60>gap)
        if separated:
            if start is not None: events.append((times[start],times[previous]))
            start=pos
        previous=pos
    if start is not None: events.append((times[start],times[previous]))
    return events

def event_metrics(frame,pred_col):
    truth=[]
    labelled=frame.loc[frame.is_anomaly.eq(1)&frame.episode_id.ne('')]
    for (station,episode),g in labelled.groupby(['station_id','episode_id']):
        truth.append((station,episode,g.emitted_timestamp_utc.min(),g.emitted_timestamp_utc.max(),g.anomaly_type.mode().iloc[0]))
    predictions=[]; station_days=0
    for station,g in frame.groupby('station_id',sort=False):
        g=g.sort_values('emitted_timestamp_utc'); station_days+=max((g.emitted_timestamp_utc.iloc[-1]-g.emitted_timestamp_utc.iloc[0]).total_seconds()/86400,1/24)
        predictions.extend((station,a,b) for a,b in predicted_events(g,pred_col))
    used=set(); hits=[]; delays=[]; per_fault={}
    for station,episode,start,end,fault in truth:
        candidates=[(i,p) for i,p in enumerate(predictions) if i not in used and p[0]==station and p[1]<=end and p[2]>=start]
        hit=bool(candidates)
        if hit:
            i,p=min(candidates,key=lambda item:item[1][1]); used.add(i); delays.append(max(0,(max(start,p[1])-start).total_seconds()/60))
        hits.append(hit); per_fault.setdefault(fault,[]).append(hit)
    tp=sum(hits); fp=len(predictions)-len(used); fn=len(truth)-tp
    ep=tp/max(tp+fp,1); er=tp/max(tp+fn,1)
    return {'true_episodes':len(truth),'predicted_episodes':len(predictions),'event_precision':ep,'event_recall':er,
            'event_f1':2*ep*er/max(ep+er,1e-12),'false_alarm_episodes_per_station_day':fp/max(station_days,1e-12),
            'delay_median_min':float(np.median(delays)) if delays else None,'delay_p90_min':float(np.quantile(delays,.9)) if delays else None,
            'delay_mean_min':float(np.mean(delays)) if delays else None,
            'per_fault_episode_recall':{k:float(np.mean(v)) for k,v in sorted(per_fault.items())}}

def hysteresis(frame,score_col,start,cont):
    result=pd.Series(False,index=frame.index)
    for _,g in frame.groupby('station_id',sort=False):
        g=g.sort_values('emitted_timestamp_utc'); active=False; out=np.zeros(len(g),bool)
        for pos,score in enumerate(g[score_col].fillna(0).to_numpy(float)):
            if not active and score>=start: active=True
            elif active and score<cont: active=False
            out[pos]=active
        result.loc[g.index]=out
    return result

def persistent_signal(frame,score_col,threshold,min_points):
    result=pd.Series(False,index=frame.index)
    for _,g in frame.groupby('station_id',sort=False):
        g=g.sort_values('emitted_timestamp_utc'); run=0; out=np.zeros(len(g),bool)
        for pos,score in enumerate(g[score_col].fillna(0).to_numpy(float)):
            run=run+1 if score>=threshold else 0
            out[pos]=run>=min_points
        result.loc[g.index]=out
    return result

def evaluate(frame,score_col,pred_col): return {**point_metrics(frame.is_anomaly,frame[pred_col],frame[score_col]),**event_metrics(frame,pred_col)}


## 4. Train focused frozen, bias, and drift specialists

Each specialist uses a small causal feature family and three fixed seeds. Models train on 2022, use January–April 2023 only for early stopping, and never see the three robust policy blocks during fitting.


In [9]:
FROZEN_FEATURES=[c for c in FEATURES if any(token in c for token in [
    'frozen_run_length','rolling_mad_24h','rolling_median_24h','delta1','rate_per_hour','neighbor_'])]
BIAS_FEATURES=[c for c in FEATURES if any(token in c for token in [
    'cusum_','climatology_residual','ewma_residual','neighbor_residual','regional_agreement'])]
DRIFT_FEATURES=[c for c in FEATURES if any(token in c for token in [
    '_slope_','cusum_','monotonic_run','climatology_residual','neighbor_residual','regional_'])]
SPECIALIST_FEATURES={'frozen_sensor':FROZEN_FEATURES,'bias':BIAS_FEATURES,'drift':DRIFT_FEATURES}
print({k:len(v) for k,v in SPECIALIST_FEATURES.items()})
assert all(len(v)>=12 for v in SPECIALIST_FEATURES.values())

train_mask=dev.dev_split.eq('train'); tune_mask=dev.dev_split.eq('tune_model')
specialist_models={}; specialist_history={}
for fault,features in SPECIALIST_FEATURES.items():
    y_train=dev.loc[train_mask,'anomaly_type'].eq(fault).astype(int)
    y_tune=dev.loc[tune_mask,'anomaly_type'].eq(fault).astype(int)
    ratio=(len(y_train)-y_train.sum())/max(y_train.sum(),1)
    models=[]; history=[]
    for seed in SPECIALIST_SEEDS:
        path=ARTIFACT_ROOT/f'{fault}_specialist_seed{seed}.cbm'
        model=CatBoostClassifier(iterations=1000,depth=7,learning_rate=.035,loss_function='Logloss',eval_metric='PRAUC',
            scale_pos_weight=min(math.sqrt(ratio),25),l2_leaf_reg=8,random_strength=.4,random_seed=seed,
            task_type='GPU',devices='0',verbose=100,od_type='Iter',od_wait=100,allow_writing_files=False)
        if REUSE_SAVED_MODELS and path.exists(): model.load_model(path)
        else:
            model.fit(dev.loc[train_mask,features],y_train,eval_set=(dev.loc[tune_mask,features],y_tune),use_best_model=True)
            model.save_model(path)
        best=model.get_best_iteration()
        models.append(model); history.append({'seed':seed,'best_iteration':int(best if best is not None else model.tree_count_-1)})
    specialist_models[fault]=models; specialist_history[fault]=history
    raw=np.mean([m.predict_proba(dev[features])[:,1] for m in models],axis=0)
    dev[f'{fault}_raw']=raw
print(specialist_history)


{'frozen_sensor': 52, 'bias': 26, 'drift': 41}
{'frozen_sensor': [{'seed': 17, 'best_iteration': 17}, {'seed': 41, 'best_iteration': 65}, {'seed': 67, 'best_iteration': 8}], 'bias': [{'seed': 17, 'best_iteration': 4}, {'seed': 41, 'best_iteration': 12}, {'seed': 67, 'best_iteration': 3}], 'drift': [{'seed': 17, 'best_iteration': 9}, {'seed': 41, 'best_iteration': 18}, {'seed': 67, 'best_iteration': 4}]}


## 5. Rank-preserving Platt calibration for comparable rescue scores

In [10]:
fit_mask=dev.dev_split.eq('block_may_jun')
calibrators={}

def fit_platt(raw,y):
    raw=np.clip(np.asarray(raw,float),1e-6,1-1e-6); logit=np.log(raw/(1-raw)).reshape(-1,1)
    model=LogisticRegression(C=1,max_iter=2000).fit(logit,np.asarray(y,int)); return model

def apply_platt(model,raw):
    raw=np.clip(np.asarray(raw,float),1e-6,1-1e-6); return model.predict_proba(np.log(raw/(1-raw)).reshape(-1,1))[:,1]

for fault in SPECIALIST_FEATURES:
    col=f'{fault}_raw'; y=dev.anomaly_type.eq(fault).astype(int)
    calibrators[fault]=fit_platt(dev.loc[fit_mask,col],y.loc[fit_mask])
    dev[f'{fault}_score']=apply_platt(calibrators[fault],dev[col])

base_calibrator=fit_platt(dev.loc[fit_mask,'cat_fault_mean'],dev.loc[fit_mask,'is_anomaly'])
dev['base_score']=apply_platt(base_calibrator,dev.cat_fault_mean)
dev['rescue_score']=dev[[f'{f}_score' for f in SPECIALIST_FEATURES]].max(axis=1)

rows=[]
for fault in SPECIALIST_FEATURES:
    for block in ['block_jul_sep','block_oct_dec']:
        m=dev.dev_split.eq(block); y=dev.anomaly_type.eq(fault).astype(int)
        rows.append({'fault':fault,'block':block,'auprc':average_precision_score(y[m],dev.loc[m,f'{fault}_score'])})
specialist_validation=pd.DataFrame(rows)
display(specialist_validation)


,fault,block,auprc
0,frozen_sensor,block_jul_sep,0.45550
1,frozen_sensor,block_oct_dec,0.00294
2,bias,block_jul_sep,0.01461
3,bias,block_oct_dec,0.37917
4,drift,block_jul_sep,0.00473
5,drift,block_oct_dec,0.03752


## 6. Hard rules and two-tier rescue policy

The high-precision CatBoost channel creates ordinary alerts. Frozen/bias/drift specialists may rescue an incident only after their evidence persists for multiple consecutive readings. Hard packet/timestamp/physical errors remain deterministic overrides.


In [11]:
def add_hard_rules(frame):
    z=frame.copy()
    duplicate=z.duplicated(['station_id','emitted_timestamp_utc'],keep=False)
    timestamp=z.out_of_order_indicator.fillna(0).gt(0)
    physical=((z.temperature_value.notna()&~z.temperature_value.between(-60,60))|
              (z.pressure_value.notna()&~z.pressure_value.between(800,1100))|
              (z.humidity_value.notna()&~z.humidity_value.between(0,100)))
    z['hard_rule']=(duplicate|timestamp|physical)
    return z
dev=add_hard_rules(dev)

POLICY_BLOCKS=['block_may_jun','block_jul_sep','block_oct_dec']

def apply_two_tier(frame,base_start,base_continue,rescue_threshold,min_points):
    base=hysteresis(frame,'base_score',base_start,base_continue)
    rescue=persistent_signal(frame,'rescue_score',rescue_threshold,min_points)
    return base|rescue|frame.hard_rule

def search_robust_policy(frame):
    rows=[]
    for base_start in np.linspace(.30,.90,8):
      for rescue_threshold in [.50,.60,.70,.80,.90]:
       for min_points in [2,3,4]:
        block_metrics=[]
        for block in POLICY_BLOCKS:
            part=frame.loc[frame.dev_split.eq(block)].copy()
            part['pred']=apply_two_tier(part,base_start,max(0,base_start-.08),rescue_threshold,min_points)
            block_metrics.append((block,evaluate(part,'base_score','pred')))
        summary={'base_start':base_start,'base_continue':max(0,base_start-.08),
                 'rescue_threshold':rescue_threshold,'min_points':min_points}
        for block,m in block_metrics:
            for key in ['precision','recall','f1','auprc','event_precision','event_recall','event_f1',
                        'false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']:
                summary[f'{block}_{key}']=m[key]
        summary['min_precision']=min(m['precision'] for _,m in block_metrics)
        summary['max_false_alarm']=max(m['false_alarm_episodes_per_station_day'] for _,m in block_metrics)
        summary['min_event_recall']=min(m['event_recall'] for _,m in block_metrics)
        summary['mean_event_f1']=np.mean([m['event_f1'] for _,m in block_metrics])
        summary['mean_point_f1']=np.mean([m['f1'] for _,m in block_metrics])
        rows.append(summary)
    frontier=pd.DataFrame(rows)
    feasible=frontier.loc[(frontier.min_precision>=.75)&(frontier.max_false_alarm<=.02)]
    if len(feasible):
        selected=feasible.sort_values(['min_event_recall','mean_event_f1','mean_point_f1'],ascending=False).iloc[0]
        status='constraints_met_all_blocks'
    else:
        frontier['violation']=np.maximum(0,.75-frontier.min_precision)/.75+np.maximum(0,frontier.max_false_alarm-.02)/.02
        selected=frontier.sort_values(['violation','min_event_recall','mean_event_f1'],ascending=[True,False,False]).iloc[0]
        status='pareto_fallback'
    return selected,frontier,status

selected,frontier,POLICY_STATUS=search_robust_policy(dev)
frontier.to_csv(ARTIFACT_ROOT/'iteration2_policy_frontier.csv',index=False)
display(selected.to_frame('selected')); print(POLICY_STATUS)


,selected
base_start,0.30000
base_continue,0.22000
rescue_threshold,0.50000
min_points,2.00000
block_may_jun_precision,0.80000
block_may_jun_recall,0.44444
block_may_jun_f1,0.57143
block_may_jun_auprc,0.52547
block_may_jun_event_precision,0.51613
block_may_jun_event_recall,0.72727


constraints_met_all_blocks


## 7. Multi-block ablation and weak-fault recall

In [12]:
def robust_base_policy(frame,score_col):
    candidates=[]
    for threshold in np.linspace(.10,.95,25):
        block_results=[]
        for block in POLICY_BLOCKS:
            part=frame.loc[frame.dev_split.eq(block)].copy()
            part['pred']=hysteresis(part,score_col,threshold,max(0,threshold-.08))
            block_results.append(evaluate(part,score_col,'pred'))
        candidates.append({'threshold':threshold,
            'min_precision':min(m['precision'] for m in block_results),
            'max_false_alarm':max(m['false_alarm_episodes_per_station_day'] for m in block_results),
            'min_event_recall':min(m['event_recall'] for m in block_results),
            'mean_event_f1':np.mean([m['event_f1'] for m in block_results])})
    feasible=[x for x in candidates if x['min_precision']>=.75 and x['max_false_alarm']<=.02]
    return max(feasible or candidates,key=lambda x:(x['min_event_recall'],x['mean_event_f1']))

rows=[]; combined=[]
base_policy=robust_base_policy(dev,'base_score')
print('Robust CatBoost base policy:',base_policy)
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['phase10_pred']=part.phase10_fault.ge(PHASE10_THRESHOLD)
    rows.append({'block':block,'variant':'Phase10 fixed',**evaluate(part,'phase10_fault','phase10_pred')})
    part['cat_pred']=hysteresis(part,'base_score',base_policy['threshold'],max(0,base_policy['threshold']-.08))
    rows.append({'block':block,'variant':'CatBoost base',**evaluate(part,'base_score','cat_pred')})
    part['rescue_pred']=apply_two_tier(part,selected.base_start,selected.base_continue,selected.rescue_threshold,int(selected.min_points))
    rows.append({'block':block,'variant':'CatBoost + weak-fault rescue',**evaluate(part,'base_score','rescue_pred')})
    combined.append(part)
ablation=pd.DataFrame(rows)
ablation.to_csv(ARTIFACT_ROOT/'iteration2_multiblock_ablation.csv',index=False)
display(ablation[['block','variant','precision','recall','f1','auprc','event_precision','event_recall','event_f1',
                  'false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']])

combined=pd.concat(combined,ignore_index=True); combined['pred']=combined.rescue_pred
fault_recall=pd.Series(event_metrics(combined,'pred')['per_fault_episode_recall'],name='episode_recall').sort_values().to_frame()
fault_recall.to_csv(ARTIFACT_ROOT/'iteration2_fault_episode_recall.csv')
display(fault_recall)


Robust CatBoost base policy: {'threshold': np.float64(0.27708333333333335), 'min_precision': 0.8515625, 'max_false_alarm': 0.01796346068791889, 'min_event_recall': 0.6296296296296297, 'mean_event_f1': np.float64(0.5463963278555489)}


,block,variant,precision,recall,f1,auprc,event_precision,event_recall,event_f1,false_alarm_episodes_per_station_day,delay_mean_min,delay_p90_min
0,block_may_jun,Phase10 fixed,0.61453,0.45267,0.52133,0.48686,0.27273,0.81818,0.40909,0.03945,290.00000,702.00000
1,block_may_jun,CatBoost base,0.85156,0.44856,0.58760,0.52547,0.45161,0.63636,0.52830,0.01397,171.42857,477.00000
2,block_may_jun,CatBoost + weak-fault rescue,0.80000,0.44444,0.57143,0.52547,0.51613,0.72727,0.60377,0.01233,86.25000,300.00000
3,block_jul_sep,Phase10 fixed,0.76238,0.20533,0.32353,0.23993,0.35088,0.74074,0.47619,0.02014,327.00000,954.00000
4,block_jul_sep,CatBoost base,0.86250,0.18400,0.30330,0.24411,0.48571,0.62963,0.54839,0.00980,201.17647,720.00000
5,block_jul_sep,CatBoost + weak-fault rescue,0.88608,0.18667,0.30837,0.24411,0.54545,0.66667,0.60000,0.00817,190.00000,630.00000
6,block_oct_dec,Phase10 fixed,0.73646,0.43312,0.54545,0.46925,0.30851,0.80556,0.44615,0.03538,104.48276,444.00000
7,block_oct_dec,CatBoost base,0.89573,0.40127,0.55425,0.49516,0.45000,0.75000,0.56250,0.01796,100.00000,468.00000
8,block_oct_dec,CatBoost + weak-fault rescue,0.83262,0.41189,0.55114,0.49516,0.47541,0.80556,0.59794,0.01742,93.10345,444.00000


,episode_recall
duplicate_packet,0.00000
frozen_sensor,0.16667
drift,0.25000
bias,0.42857
noise,0.85714
sudden_drop,0.87500
multi_sensor_failure,1.00000
communication_corruption,1.00000
scaling_error,1.00000
spike,1.00000


## 8. Repair the weather channel by selecting the robust existing score

In [13]:
weather_candidates=['phase10_weather','cat_weather_mean','cat_weather_median']
weather_rows=[]
for score_col in weather_candidates:
 for threshold in np.linspace(.02,.90,45):
    metrics=[]
    for block in POLICY_BLOCKS:
        part=dev.loc[dev.dev_split.eq(block)]; pred=part[score_col].ge(threshold)
        metrics.append({'block':block,'precision':precision_score(part.is_weather_event,pred,zero_division=0),
                        'recall':recall_score(part.is_weather_event,pred,zero_division=0),
                        'f1':f1_score(part.is_weather_event,pred,zero_division=0)})
    weather_rows.append({'score':score_col,'threshold':threshold,'min_f1':min(m['f1'] for m in metrics),
                         'mean_f1':np.mean([m['f1'] for m in metrics]),'blocks':metrics})
weather_frontier=pd.DataFrame(weather_rows)
weather_selected=weather_frontier.sort_values(['min_f1','mean_f1'],ascending=False).iloc[0]
weather_frontier.drop(columns='blocks').to_csv(ARTIFACT_ROOT/'iteration2_weather_frontier.csv',index=False)
display(weather_selected.to_frame('selected'))


,selected
score,cat_weather_mean
threshold,0.24000
min_f1,0.43697
mean_f1,0.56113
blocks,"[{'block': 'block_may_jun', 'precision': 0.470..."


## 9. Save Iteration 2 result package

In [14]:
result={
 'iteration':'02_weak_fault_rescue','device':DEVICE,'gpu':torch.cuda.get_device_name(0),
 'final_tests_opened':bool(UNLOCK_FINAL_TESTS),'tcn_promoted':False,'isotonic_fusion_promoted':False,
 'specialist_history':specialist_history,'specialist_validation':specialist_validation.to_dict('records'),
 'policy_status':POLICY_STATUS,'selected_policy':selected.to_dict(),'robust_base_policy':base_policy,
 'multiblock_ablation':ablation.drop(columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
 'fault_episode_recall':fault_recall.episode_recall.to_dict(),
 'weather_selected':{'score':weather_selected.score,'threshold':float(weather_selected.threshold),
                     'min_f1':float(weather_selected.min_f1),'mean_f1':float(weather_selected.mean_f1)},
}
(ARTIFACT_ROOT/'iteration2_result_block.json').write_text(json.dumps(result,indent=2,default=float))
joblib.dump({'specialist_calibrators':calibrators,'base_calibrator':base_calibrator,
             'specialist_features':SPECIALIST_FEATURES,'selected_policy':selected.to_dict(),
             'weather_score':weather_selected.score,'weather_threshold':float(weather_selected.threshold)},
            ARTIFACT_ROOT/'iteration2_policy.joblib')
print(json.dumps(result,indent=2,default=float))
print('\nHISTORICAL CHECKPOINT FILES - continue through Iteration 8; do not return these yet:')
for name in ['iteration2_result_block.json','iteration2_multiblock_ablation.csv','iteration2_fault_episode_recall.csv','iteration2_weather_frontier.csv']:
    print(ARTIFACT_ROOT/name)


{
  "iteration": "02_weak_fault_rescue",
  "device": "cuda",
  "gpu": "Tesla T4",
  "final_tests_opened": false,
  "tcn_promoted": false,
  "isotonic_fusion_promoted": false,
  "specialist_history": {
    "frozen_sensor": [
      {
        "seed": 17,
        "best_iteration": 17
      },
      {
        "seed": 41,
        "best_iteration": 65
      },
      {
        "seed": 67,
        "best_iteration": 8
      }
    ],
    "bias": [
      {
        "seed": 17,
        "best_iteration": 4
      },
      {
        "seed": 41,
        "best_iteration": 12
      },
      {
        "seed": 67,
        "best_iteration": 3
      }
    ],
    "drift": [
      {
        "seed": 17,
        "best_iteration": 9
      },
      {
        "seed": 41,
        "best_iteration": 18
      },
      {
        "seed": 67,
        "best_iteration": 4
      }
    ]
  },
  "specialist_validation": [
    {
      "fault": "frozen_sensor",
      "block": "block_jul_sep",
      "auprc": 0.4555048717987588
   

## 10. Final-test seal

Iteration 2 intentionally contains no final-test scoring cell. Even if `UNLOCK_FINAL_TESTS` is changed accidentally, no 2024 file is loaded. After reviewing these outputs, we will either retain CatBoost alone or freeze the rescue policy, and only a separate finalization notebook will open the tests once.


## Historical checkpoint - continue running

Send the four files listed above. We will accept the rescue policy only if it improves weak-fault/event recall consistently across the three 2023 blocks while preserving precision and the 0.02 false-alarm budget. Otherwise CatBoost alone remains the winner.


# Iteration 3 controlled experiment

Everything above reconstructs Iteration 2 and reuses its saved models. The cells below write only to `iteration_03_frozen_communication`.


In [15]:
ITER3_ROOT=DRIVE_ROOT/'experiments'/'iteration_03_frozen_communication'
ITER3_ROOT.mkdir(parents=True,exist_ok=True)
print('Iteration 3 artifacts:',ITER3_ROOT)


Iteration 3 artifacts: /content/drive/MyDrive/SkyGuard_AI_GPU/experiments/iteration_03_frozen_communication


## 11. Sensor-specific frozen rules

Natural quantization differs strongly by sensor. Therefore a single frozen threshold is inappropriate. Candidate rules require both a constant-value run and disagreement from currently available neighbouring stations.


In [16]:
FROZEN_CANDIDATES={
 'temperature':[None,(4,5.0),(8,4.0),(12,3.0)],
 'pressure':[None,(12,5.0),(16,3.0),(20,2.0)],
 'humidity':[None,(16,10.0),(12,10.0),(20,5.0)],
}

def sensor_frozen_rule(frame,sensor,config):
    if config is None: return pd.Series(False,index=frame.index)
    run_threshold,residual_threshold=config
    return (frame[f'{sensor}_frozen_run_length'].ge(run_threshold)&
            frame[f'neighbor_{sensor}_count'].ge(1)&
            frame[f'neighbor_{sensor}_residual'].abs().ge(residual_threshold))

def frozen_rule(frame,configs):
    result=pd.Series(False,index=frame.index)
    for sensor,config in configs.items(): result|=sensor_frozen_rule(frame,sensor,config)
    return result

def evaluate_frozen_configuration(configs):
    rows=[]
    for block in POLICY_BLOCKS:
        part=dev.loc[dev.dev_split.eq(block)].copy()
        part['iteration2_pred']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                               selected.rescue_threshold,int(selected.min_points))
        part['frozen_rule']=frozen_rule(part,configs)
        part['candidate_pred']=part.iteration2_pred|part.frozen_rule
        reference=evaluate(part,'base_score','iteration2_pred')
        metric=evaluate(part,'base_score','candidate_pred')
        rows.append({'block':block,**metric,
                     'event_f1_delta':metric['event_f1']-reference['event_f1'],
                     'point_f1_delta':metric['f1']-reference['f1']})
    valid_frozen=[row['per_fault_episode_recall'].get('frozen_sensor') for row in rows
                  if row['per_fault_episode_recall'].get('frozen_sensor') is not None]
    return {
        'temperature':str(configs['temperature']),'pressure':str(configs['pressure']),
        'humidity':str(configs['humidity']),'rows':rows,
        'min_precision':min(row['precision'] for row in rows),
        'max_false_alarm':max(row['false_alarm_episodes_per_station_day'] for row in rows),
        'min_event_recall':min(row['event_recall'] for row in rows),
        'mean_event_f1':float(np.mean([row['event_f1'] for row in rows])),
        'min_event_f1_delta':min(row['event_f1_delta'] for row in rows),
        'mean_event_f1_delta':float(np.mean([row['event_f1_delta'] for row in rows])),
        'min_frozen_recall':min(valid_frozen) if valid_frozen else 0.0,
        'mean_frozen_recall':float(np.mean(valid_frozen)) if valid_frozen else 0.0,
    }

candidate_rows=[]
for temperature in FROZEN_CANDIDATES['temperature']:
 for pressure in FROZEN_CANDIDATES['pressure']:
  for humidity in FROZEN_CANDIDATES['humidity']:
    candidate_rows.append(evaluate_frozen_configuration({
        'temperature':temperature,'pressure':pressure,'humidity':humidity}))

frontier=pd.DataFrame([{k:v for k,v in row.items() if k!='rows'} for row in candidate_rows])
baseline_frozen=float(frontier.loc[
    frontier[['temperature','pressure','humidity']].eq('None').all(axis=1),'mean_frozen_recall'].iloc[0])
feasible=frontier.loc[(frontier.min_precision>=.75)&(frontier.max_false_alarm<=.02)&
                      (frontier.min_event_f1_delta>=-.01)&(frontier.mean_event_f1_delta>=0)]
if len(feasible):
    frozen_selected=feasible.sort_values(
        ['min_frozen_recall','mean_frozen_recall','mean_event_f1_delta','min_event_recall'],ascending=False).iloc[0]
    FROZEN_STATUS='constraints_met_all_blocks'
else:
    frontier['violation']=np.maximum(0,.75-frontier.min_precision)/.75+np.maximum(0,frontier.max_false_alarm-.02)/.02
    frozen_selected=frontier.sort_values(['violation','min_frozen_recall','mean_event_f1'],ascending=[True,False,False]).iloc[0]
    FROZEN_STATUS='pareto_fallback'

def parse_config(value):
    if value=='None': return None
    left,right=value.strip('()').split(','); return (int(left),float(right))

SELECTED_FROZEN_CONFIG={sensor:parse_config(frozen_selected[sensor]) for sensor in ['temperature','pressure','humidity']}
if float(frozen_selected.mean_frozen_recall)<=baseline_frozen:
    SELECTED_FROZEN_CONFIG={sensor:None for sensor in ['temperature','pressure','humidity']}
    FROZEN_STATUS='no_generalizable_gain_keep_iteration2'
frontier.to_csv(ITER3_ROOT/'iteration3_frozen_frontier.csv',index=False)
display(frozen_selected.to_frame('selected')); print(FROZEN_STATUS,SELECTED_FROZEN_CONFIG)


,selected
temperature,None
pressure,None
humidity,None
min_precision,0.80000
max_false_alarm,0.01742
min_event_recall,0.66667
mean_event_f1,0.60057
min_event_f1_delta,0.00000
mean_event_f1_delta,0.00000
min_frozen_recall,0.00000


no_generalizable_gain_keep_iteration2 {'temperature': None, 'pressure': None, 'humidity': None}


## 12. Clean contribution ablation

In [17]:
rows=[]; combined=[]
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['cat_base']=hysteresis(part,'base_score',base_policy['threshold'],max(0,base_policy['threshold']-.08))
    part['cat_hard']=part.cat_base|part.hard_rule
    part['iteration2']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                      selected.rescue_threshold,int(selected.min_points))
    part['frozen_only']=part.cat_hard|frozen_rule(part,SELECTED_FROZEN_CONFIG)
    part['iteration3']=part.iteration2|frozen_rule(part,SELECTED_FROZEN_CONFIG)
    for variant,pred in [('CatBoost base','cat_base'),('CatBoost + hard rules','cat_hard'),
                         ('Iteration2 learned rescue','iteration2'),('CatBoost + frozen rule','frozen_only'),
                         ('Iteration3 combined','iteration3')]:
        rows.append({'block':block,'variant':variant,**evaluate(part,'base_score',pred)})
    combined.append(part)

ablation3=pd.DataFrame(rows)
ablation3.to_csv(ITER3_ROOT/'iteration3_multiblock_ablation.csv',index=False)
display(ablation3[['block','variant','precision','recall','f1','event_precision','event_recall','event_f1',
                   'false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']])

combined=pd.concat(combined,ignore_index=True); combined['pred']=combined.iteration3
fault_recall3=pd.Series(event_metrics(combined,'pred')['per_fault_episode_recall'],name='episode_recall').sort_values().to_frame()
fault_recall3.to_csv(ITER3_ROOT/'iteration3_fault_episode_recall.csv')
display(fault_recall3)


,block,variant,precision,recall,f1,event_precision,event_recall,event_f1,false_alarm_episodes_per_station_day,delay_mean_min,delay_p90_min
0,block_may_jun,CatBoost base,0.85156,0.44856,0.58760,0.45161,0.63636,0.52830,0.01397,171.42857,477.00000
1,block_may_jun,CatBoost + hard rules,0.78472,0.46502,0.58398,0.50000,0.77273,0.60714,0.01397,141.17647,414.00000
2,block_may_jun,Iteration2 learned rescue,0.80000,0.44444,0.57143,0.51613,0.72727,0.60377,0.01233,86.25000,300.00000
3,block_may_jun,CatBoost + frozen rule,0.78472,0.46502,0.58398,0.50000,0.77273,0.60714,0.01397,141.17647,414.00000
4,block_may_jun,Iteration3 combined,0.80000,0.44444,0.57143,0.51613,0.72727,0.60377,0.01233,86.25000,300.00000
5,block_jul_sep,CatBoost base,0.86250,0.18400,0.30330,0.48571,0.62963,0.54839,0.00980,201.17647,720.00000
6,block_jul_sep,CatBoost + hard rules,0.84337,0.18667,0.30568,0.50000,0.66667,0.57143,0.00980,190.00000,630.00000
7,block_jul_sep,Iteration2 learned rescue,0.88608,0.18667,0.30837,0.54545,0.66667,0.60000,0.00817,190.00000,630.00000
8,block_jul_sep,CatBoost + frozen rule,0.84337,0.18667,0.30568,0.50000,0.66667,0.57143,0.00980,190.00000,630.00000
9,block_jul_sep,Iteration3 combined,0.88608,0.18667,0.30837,0.54545,0.66667,0.60000,0.00817,190.00000,630.00000


,episode_recall
duplicate_packet,0.00000
frozen_sensor,0.16667
drift,0.25000
bias,0.42857
noise,0.85714
sudden_drop,0.87500
multi_sensor_failure,1.00000
communication_corruption,1.00000
scaling_error,1.00000
spike,1.00000


## 13. Correct duplicate-packet evaluation through replay

The detector never reads `stream_action`. The simulator reads it and emits an identical packet twice. The operational rule then detects the second packet by a station/timestamp/value fingerprint. This mirrors the actual streaming engine.


In [18]:
validation_full=load_table('validation')
replay_packets=[]
for _,row in validation_full.iterrows():
    packet={'station_id':row.station_id,'timestamp':row.emitted_timestamp_utc,
            'temperature':row.temperature_value,'pressure':row.pressure_value,'humidity':row.humidity_value,
            'episode_id':row.episode_id,'injected_duplicate':False}
    replay_packets.append(packet)
    if row.stream_action=='duplicate':
        duplicate=packet.copy(); duplicate['injected_duplicate']=True; replay_packets.append(duplicate)

def packet_fingerprint(packet):
    clean=lambda value: None if pd.isna(value) else value
    return (packet['station_id'],packet['timestamp'],clean(packet['temperature']),
            clean(packet['pressure']),clean(packet['humidity']))

seen=set(); tp=fp=fn=0; detected_episodes=set(); true_episodes=set()
for packet in replay_packets:
    fingerprint=packet_fingerprint(packet)
    prediction=fingerprint in seen; seen.add(fingerprint)
    truth=bool(packet['injected_duplicate'])
    tp+=int(prediction and truth); fp+=int(prediction and not truth); fn+=int(not prediction and truth)
    if truth and packet['episode_id']: true_episodes.add(packet['episode_id'])
    if prediction and truth and packet['episode_id']: detected_episodes.add(packet['episode_id'])

communication_result={
    'simulated_input_rows':int(len(validation_full)),
    'emitted_packets':int(len(replay_packets)),
    'duplicate_packet_tp':tp,'duplicate_packet_fp':fp,'duplicate_packet_fn':fn,
    'duplicate_packet_precision':tp/max(tp+fp,1),
    'duplicate_packet_recall':tp/max(tp+fn,1),
    'true_duplicate_episodes':len(true_episodes),
    'detected_duplicate_episodes':len(true_episodes&detected_episodes),
    'duplicate_episode_recall':len(true_episodes&detected_episodes)/max(len(true_episodes),1),
    'detector_inputs':['station_id','timestamp','temperature','pressure','humidity'],
    'stream_action_used_by_detector':False,
}
pd.DataFrame([communication_result]).to_csv(ITER3_ROOT/'iteration3_communication_replay.csv',index=False)
display(pd.Series(communication_result,name='replay'))


,replay
simulated_input_rows,181470
emitted_packets,181486
duplicate_packet_tp,16
duplicate_packet_fp,0
duplicate_packet_fn,0
duplicate_packet_precision,1.00000
duplicate_packet_recall,1.00000
true_duplicate_episodes,10
detected_duplicate_episodes,10
duplicate_episode_recall,1.00000


## 14. Save Iteration 3 result block

In [19]:
iteration3_rows=ablation3.loc[ablation3.variant.eq('Iteration3 combined')]
result3={
 'iteration':'03_frozen_communication','device':DEVICE,'gpu':torch.cuda.get_device_name(0),
 'final_tests_opened':False,'frozen_policy_status':FROZEN_STATUS,
 'selected_frozen_config':SELECTED_FROZEN_CONFIG,
 'iteration3_blocks':iteration3_rows.drop(columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
 'fault_episode_recall':fault_recall3.episode_recall.to_dict(),
 'communication_replay':communication_result,
 'weather_policy_unchanged':{'score':weather_selected.score,'threshold':float(weather_selected.threshold)},
}
(ITER3_ROOT/'iteration3_result_block.json').write_text(json.dumps(result3,indent=2,default=float))
print(json.dumps(result3,indent=2,default=float))
print('\nHISTORICAL CHECKPOINT FILES - continue through Iteration 8; do not return these yet:')
for name in ['iteration3_result_block.json','iteration3_multiblock_ablation.csv',
             'iteration3_fault_episode_recall.csv','iteration3_communication_replay.csv']:
    print(ITER3_ROOT/name)


{
  "iteration": "03_frozen_communication",
  "device": "cuda",
  "gpu": "Tesla T4",
  "final_tests_opened": false,
  "frozen_policy_status": "no_generalizable_gain_keep_iteration2",
  "selected_frozen_config": {
    "temperature": null,
    "pressure": null,
    "humidity": null
  },
  "iteration3_blocks": [
    {
      "block": "block_may_jun",
      "variant": "Iteration3 combined",
      "tp": 108,
      "fp": 27,
      "fn": 135,
      "tn": 28742,
      "precision": 0.8,
      "recall": 0.4444444444444444,
      "f1": 0.5714285714285714,
      "auprc": 0.5254728087516216,
      "true_episodes": 22,
      "predicted_episodes": 31,
      "event_precision": 0.5161290322580645,
      "event_recall": 0.7272727272727273,
      "event_f1": 0.6037735849056604,
      "false_alarm_episodes_per_station_day": 0.012328344919694532,
      "delay_median_min": 0.0,
      "delay_p90_min": 300.0,
      "delay_mean_min": 86.25
    },
    {
      "block": "block_jul_sep",
      "variant": "Iteration

## Decision rule

Promote the frozen rule only if it raises frozen episode recall while all three blocks retain precision ≥75% and false-alarm episodes ≤0.02/station-day. Duplicate-packet replay must reach 100% episode recall. The final 2024 evaluation remains a separate, one-time notebook after this result is reviewed.


# Iteration 4 controlled experiment

Everything above reconstructs Iterations 2 and 3 from saved Drive checkpoints. The cells below write only to `iteration_04_causal_incident_state`.


In [20]:
ITER4_ROOT=DRIVE_ROOT/'experiments'/'iteration_04_causal_incident_state'
ITER4_ROOT.mkdir(parents=True,exist_ok=True)
print('Iteration 4 artifacts:',ITER4_ROOT)


Iteration 4 artifacts: /content/drive/MyDrive/SkyGuard_AI_GPU/experiments/iteration_04_causal_incident_state


## 15. Causal incident-state controller

`grace_minutes` allows a live alert to remain active briefly after its latest strong or supporting observation. `support_threshold` is applied to the maximum calibrated CatBoost/specialist score and cannot start an incident. `max_duration_minutes` prevents an alert from remaining active indefinitely.


In [21]:
def causal_incident_state(frame,trigger_col,grace_minutes,support_threshold,max_duration_minutes):
    result=pd.Series(False,index=frame.index)
    for _,g in frame.groupby('station_id',sort=False):
        g=g.sort_values('emitted_timestamp_utc')
        times=g.emitted_timestamp_utc.tolist()
        triggers=g[trigger_col].fillna(False).to_numpy(bool)
        support=np.maximum(g.base_score.fillna(0).to_numpy(float),g.rescue_score.fillna(0).to_numpy(float))
        out=np.zeros(len(g),bool); active=False; start=None; last_evidence=None
        for pos,(timestamp,trigger,score) in enumerate(zip(times,triggers,support)):
            if trigger:
                if not active: start=timestamp
                active=True; last_evidence=timestamp; out[pos]=True
                continue
            if not active: continue
            elapsed=(timestamp-start).total_seconds()/60
            since_evidence=(timestamp-last_evidence).total_seconds()/60
            if elapsed>max_duration_minutes:
                active=False; start=None; last_evidence=None; continue
            if score>=support_threshold:
                last_evidence=timestamp; out[pos]=True
            elif since_evidence<=grace_minutes:
                out[pos]=True
            else:
                active=False; start=None; last_evidence=None
        result.loc[g.index]=out
    return result

STATE_CANDIDATES={
    'grace_minutes':[0,60,180,360],
    'support_threshold':[.10,.20,.30,1.10],
    'max_duration_minutes':[720,1440],
}

STATE_REFERENCE={}
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['iteration3_trigger']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                              selected.rescue_threshold,int(selected.min_points))
    part['iteration3_trigger']|=frozen_rule(part,SELECTED_FROZEN_CONFIG)
    STATE_REFERENCE[block]=evaluate(part,'base_score','iteration3_trigger')

def evaluate_state_configuration(grace_minutes,support_threshold,max_duration_minutes):
    rows=[]
    for block in POLICY_BLOCKS:
        part=dev.loc[dev.dev_split.eq(block)].copy()
        part['iteration3_trigger']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                                  selected.rescue_threshold,int(selected.min_points))
        part['iteration3_trigger']|=frozen_rule(part,SELECTED_FROZEN_CONFIG)
        part['candidate_pred']=causal_incident_state(part,'iteration3_trigger',grace_minutes,
                                                     support_threshold,max_duration_minutes)
        metric=evaluate(part,'base_score','candidate_pred'); reference=STATE_REFERENCE[block]
        rows.append({'block':block,**metric,
                     'point_f1_delta':metric['f1']-reference['f1'],
                     'event_f1_delta':metric['event_f1']-reference['event_f1']})
    return {
        'grace_minutes':grace_minutes,'support_threshold':support_threshold,
        'max_duration_minutes':max_duration_minutes,'rows':rows,
        'min_precision':min(row['precision'] for row in rows),
        'max_false_alarm':max(row['false_alarm_episodes_per_station_day'] for row in rows),
        'min_point_f1':min(row['f1'] for row in rows),
        'mean_point_f1':float(np.mean([row['f1'] for row in rows])),
        'min_point_f1_delta':min(row['point_f1_delta'] for row in rows),
        'mean_point_f1_delta':float(np.mean([row['point_f1_delta'] for row in rows])),
        'min_event_recall':min(row['event_recall'] for row in rows),
        'min_event_f1_delta':min(row['event_f1_delta'] for row in rows),
        'mean_event_f1_delta':float(np.mean([row['event_f1_delta'] for row in rows])),
    }

state_candidates=[]
for grace in STATE_CANDIDATES['grace_minutes']:
 for support in STATE_CANDIDATES['support_threshold']:
  for maximum in STATE_CANDIDATES['max_duration_minutes']:
    state_candidates.append(evaluate_state_configuration(grace,support,maximum))

state_frontier=pd.DataFrame([{k:v for k,v in row.items() if k!='rows'} for row in state_candidates])
feasible=state_frontier.loc[(state_frontier.min_precision>=.75)&
                            (state_frontier.max_false_alarm<=.02)&
                            (state_frontier.min_point_f1_delta>=0)&
                            (state_frontier.min_event_f1_delta>=-.01)&
                            (state_frontier.mean_event_f1_delta>=0)]
if len(feasible):
    state_selected=feasible.sort_values(
        ['min_point_f1','mean_point_f1','min_event_recall','mean_event_f1_delta'],ascending=False).iloc[0]
    STATE_STATUS='constraints_met_all_blocks'
    safe_selection=True
else:
    state_selected=state_frontier.sort_values(
        ['min_point_f1_delta','mean_point_f1_delta','mean_event_f1_delta'],ascending=False).iloc[0]
    STATE_STATUS='no_safe_candidate_keep_iteration3'
    safe_selection=False

if not safe_selection:
    SELECTED_STATE={'grace_minutes':0,'support_threshold':1.10,'max_duration_minutes':720}
elif float(state_selected.mean_point_f1_delta)<=0:
    STATE_STATUS='no_generalizable_gain_keep_iteration3'
    SELECTED_STATE={'grace_minutes':0,'support_threshold':1.10,'max_duration_minutes':720}
else:
    SELECTED_STATE={k:(int(state_selected[k]) if k!='support_threshold' else float(state_selected[k]))
                    for k in ['grace_minutes','support_threshold','max_duration_minutes']}

state_frontier.to_csv(ITER4_ROOT/'iteration4_state_frontier.csv',index=False)
display(state_selected.to_frame('selected')); print(STATE_STATUS,SELECTED_STATE)


,selected
grace_minutes,0.00000
support_threshold,0.30000
max_duration_minutes,720.00000
min_precision,0.80000
max_false_alarm,0.01742
min_point_f1,0.30837
mean_point_f1,0.47698
min_point_f1_delta,0.00000
mean_point_f1_delta,0.00000
min_event_recall,0.66667


no_generalizable_gain_keep_iteration3 {'grace_minutes': 0, 'support_threshold': 1.1, 'max_duration_minutes': 720}


## 16. Multiblock ablation and fault coverage

In [22]:
rows=[]; combined=[]
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['iteration3']=apply_two_tier(part,selected.base_start,selected.base_continue,
                                      selected.rescue_threshold,int(selected.min_points))
    part['iteration3']|=frozen_rule(part,SELECTED_FROZEN_CONFIG)
    part['iteration4']=causal_incident_state(part,'iteration3',**SELECTED_STATE)
    for variant,pred in [('Iteration3 trigger','iteration3'),('Iteration4 causal state','iteration4')]:
        rows.append({'block':block,'variant':variant,**evaluate(part,'base_score',pred)})
    combined.append(part)

ablation4=pd.DataFrame(rows)
ablation4.to_csv(ITER4_ROOT/'iteration4_multiblock_ablation.csv',index=False)
display(ablation4[['block','variant','precision','recall','f1','event_precision','event_recall','event_f1',
                   'false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']])

combined=pd.concat(combined,ignore_index=True); combined['pred']=combined.iteration4
fault_recall4=pd.Series(event_metrics(combined,'pred')['per_fault_episode_recall'],name='episode_recall').sort_values().to_frame()
fault_recall4.to_csv(ITER4_ROOT/'iteration4_fault_episode_recall.csv')

fault_rows=combined.loc[combined.is_anomaly.eq(1)]
point_fault_recall4=(fault_rows.groupby('anomaly_type').pred.mean().rename('point_recall').sort_values().to_frame())
point_fault_recall4.to_csv(ITER4_ROOT/'iteration4_point_fault_recall.csv')
display(fault_recall4); display(point_fault_recall4)


,block,variant,precision,recall,f1,event_precision,event_recall,event_f1,false_alarm_episodes_per_station_day,delay_mean_min,delay_p90_min
0,block_may_jun,Iteration3 trigger,0.80000,0.44444,0.57143,0.51613,0.72727,0.60377,0.01233,86.25000,300.00000
1,block_may_jun,Iteration4 causal state,0.80000,0.44444,0.57143,0.51613,0.72727,0.60377,0.01233,86.25000,300.00000
2,block_jul_sep,Iteration3 trigger,0.88608,0.18667,0.30837,0.54545,0.66667,0.60000,0.00817,190.00000,630.00000
3,block_jul_sep,Iteration4 causal state,0.88608,0.18667,0.30837,0.54545,0.66667,0.60000,0.00817,190.00000,630.00000
4,block_oct_dec,Iteration3 trigger,0.83262,0.41189,0.55114,0.47541,0.80556,0.59794,0.01742,93.10345,444.00000
5,block_oct_dec,Iteration4 causal state,0.83262,0.41189,0.55114,0.47541,0.80556,0.59794,0.01742,93.10345,444.00000


,episode_recall
duplicate_packet,0.00000
frozen_sensor,0.16667
drift,0.25000
bias,0.42857
noise,0.85714
sudden_drop,0.87500
multi_sensor_failure,1.00000
communication_corruption,1.00000
scaling_error,1.00000
spike,1.00000


,point_recall
anomaly_type,
duplicate_packet,0.00000
drift,0.05556
frozen_sensor,0.23077
bias,0.35622
noise,0.46789
multi_sensor_failure,0.72251
sudden_drop,0.87500
communication_corruption,1.00000
scaling_error,1.00000


## 17. Operational communication coverage

Duplicate packets are evaluated through replay, not static feature rows. Therefore the operational coverage table replaces the static duplicate value with the validated replay result while retaining both values for auditability.


In [23]:
operational_fault_recall=fault_recall4.episode_recall.to_dict()
static_duplicate_recall=float(operational_fault_recall.get('duplicate_packet',0.0))
operational_fault_recall['duplicate_packet']=float(communication_result['duplicate_episode_recall'])
communication_coverage={
    'static_duplicate_recall_not_operational':static_duplicate_recall,
    'replay_duplicate_packet_precision':float(communication_result['duplicate_packet_precision']),
    'replay_duplicate_packet_recall':float(communication_result['duplicate_packet_recall']),
    'replay_duplicate_episode_recall':float(communication_result['duplicate_episode_recall']),
}
display(pd.Series(communication_coverage,name='communication'))


,communication
static_duplicate_recall_not_operational,0.00000
replay_duplicate_packet_precision,1.00000
replay_duplicate_packet_recall,1.00000
replay_duplicate_episode_recall,1.00000


## 18. Save Iteration 4 result block

In [24]:
iteration4_rows=ablation4.loc[ablation4.variant.eq('Iteration4 causal state')]
result4={
 'iteration':'04_causal_incident_state','device':DEVICE,'gpu':torch.cuda.get_device_name(0),
 'final_tests_opened':False,'state_policy_status':STATE_STATUS,'selected_state':SELECTED_STATE,
 'iteration4_blocks':iteration4_rows.drop(columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
 'fault_episode_recall_static':fault_recall4.episode_recall.to_dict(),
 'fault_episode_recall_operational':operational_fault_recall,
 'point_fault_recall':point_fault_recall4.point_recall.to_dict(),
 'communication_coverage':communication_coverage,
 'frozen_rule_promoted':any(value is not None for value in SELECTED_FROZEN_CONFIG.values()),
}
(ITER4_ROOT/'iteration4_result_block.json').write_text(json.dumps(result4,indent=2,default=float))
print(json.dumps(result4,indent=2,default=float))
print('\nHISTORICAL CHECKPOINT FILES - continue through Iteration 8; do not return these yet:')
for name in ['iteration4_result_block.json','iteration4_state_frontier.csv',
             'iteration4_multiblock_ablation.csv','iteration4_fault_episode_recall.csv',
             'iteration4_point_fault_recall.csv']:
    print(ITER4_ROOT/name)


{
  "iteration": "04_causal_incident_state",
  "device": "cuda",
  "gpu": "Tesla T4",
  "final_tests_opened": false,
  "state_policy_status": "no_generalizable_gain_keep_iteration3",
  "selected_state": {
    "grace_minutes": 0,
    "support_threshold": 1.1,
    "max_duration_minutes": 720
  },
  "iteration4_blocks": [
    {
      "block": "block_may_jun",
      "variant": "Iteration4 causal state",
      "tp": 108,
      "fp": 27,
      "fn": 135,
      "tn": 28742,
      "precision": 0.8,
      "recall": 0.4444444444444444,
      "f1": 0.5714285714285714,
      "auprc": 0.5254728087516216,
      "true_episodes": 22,
      "predicted_episodes": 31,
      "event_precision": 0.5161290322580645,
      "event_recall": 0.7272727272727273,
      "event_f1": 0.6037735849056604,
      "false_alarm_episodes_per_station_day": 0.012328344919694532,
      "delay_median_min": 0.0,
      "delay_p90_min": 300.0,
      "delay_mean_min": 86.25
    },
    {
      "block": "block_jul_sep",
      "varian

## Decision rule

Promote the causal state only if every development block preserves at least 75% precision, at most 0.02 false-alarm episodes per station-day, non-decreasing point F1, no more than 0.01 event-F1 loss in any block, non-decreasing mean event F1, and a positive mean point-F1 gain. Otherwise keep the Iteration 3/2 detector unchanged.

This experiment changes alert persistence only. It does not claim to improve previously missed frozen, drift, or bias episodes, and it does not access the 2024 final tests.


# Iteration 5 controlled experiment

Everything above reconstructs Iterations 2–4 from the saved Drive checkpoints. Iteration 4 selected a zero-grace fallback, so the reference detector below is exactly the validated Iteration 3 trigger. These cells write only to `iteration_05_weak_fault_consensus`.


In [25]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

ITER5_ROOT=DRIVE_ROOT/'experiments'/'iteration_05_weak_fault_consensus'
ITER5_ROOT.mkdir(parents=True,exist_ok=True)
assert UNLOCK_FINAL_TESTS is False, 'Iteration 5 must not open final tests.'

WEAK_TYPES=('frozen_sensor','bias','drift')
WEAK_FEATURES=list(FEATURES)
assert len(WEAK_FEATURES)==108
assert not ({'hour_sin','hour_cos','day_of_year_sin','day_of_year_cos','temperature_dewpoint_spread_c'}&set(WEAK_FEATURES))
print('Iteration 5 artifacts:',ITER5_ROOT)
print('Weak fault families:',WEAK_TYPES,'| features:',len(WEAK_FEATURES))


Iteration 5 artifacts: /content/drive/MyDrive/SkyGuard_AI_GPU/experiments/iteration_05_weak_fault_consensus
Weak fault families: ('frozen_sensor', 'bias', 'drift') | features: 108


## 19. Episode-balanced weak-fault experts

The former per-fault specialists trained with row-level imbalance. A 100-reading drift incident could therefore influence fitting far more than a 5-reading frozen incident. Here every positive episode receives equal total weight before the usual class-imbalance adjustment. The model still sees only the permitted, causal feature pipeline.

A shared weak-fault target is intentionally used instead of nine fault/sensor neural networks: the 2022 training partition contains only 2 frozen-humidity episodes and 3 temperature-bias episodes, which is not enough support for a trustworthy separate model.


In [26]:
def make_episode_balanced_weights(frame,positive_mask):
    # Each weak-fault episode receives equal total positive weight. Labels are not used at inference.
    positive=np.asarray(positive_mask,bool)
    weights=np.ones(len(frame),dtype=float)
    positions=np.flatnonzero(positive)
    assert len(positions)>0
    positives=frame.iloc[positions][['episode_id','row_id']].copy()
    episode_key=positives.episode_id.fillna('').astype(str)
    episode_key=episode_key.where(episode_key.ne(''),'single_row_'+positives.row_id.astype(str))
    episode_length=episode_key.value_counts()
    per_row=episode_key.map(lambda key:1.0/episode_length.loc[key]).to_numpy(float)
    # Preserve the original total positive mass, while distributing it evenly between episodes.
    per_row*=len(per_row)/max(per_row.sum(),1e-12)
    weights[positions]=per_row
    imbalance=(len(frame)-len(positions))/max(weights[positions].sum(),1e-12)
    weights[positions]*=min(math.sqrt(imbalance),15.0)
    audit={
        'positive_rows':int(len(positions)),
        'positive_episodes':int(episode_key.nunique()),
        'median_episode_rows':float(episode_length.median()),
        'max_episode_rows':int(episode_length.max()),
        'positive_weight_sum':float(weights[positions].sum()),
        'negative_weight_sum':float(weights[~positive].sum()),
    }
    return weights,audit

weak_train_mask=dev.dev_split.eq('train')
weak_tune_mask=dev.dev_split.eq('tune_model')
weak_y=dev.anomaly_type.isin(WEAK_TYPES).astype(int)
weak_train_y=weak_y.loc[weak_train_mask].to_numpy(int)
weak_tune_y=weak_y.loc[weak_tune_mask].to_numpy(int)
weak_train_weights,weak_weight_audit=make_episode_balanced_weights(dev.loc[weak_train_mask],weak_train_y.astype(bool))

assert weak_train_y.sum()>100 and weak_tune_y.sum()>30
print('Episode-balanced training audit:',weak_weight_audit)
print('Train positives:',int(weak_train_y.sum()),'Tune positives:',int(weak_tune_y.sum()))


Episode-balanced training audit: {'positive_rows': 1397, 'positive_episodes': 45, 'median_episode_rows': 20.0, 'max_episode_rows': 95, 'positive_weight_sum': 15889.393474893872, 'negative_weight_sum': 180725.0}
Train positives: 1397 Tune positives: 272


In [27]:
WEAK_SEEDS=[17,41,67]
weak_cat_models=[]; weak_lgb_models=[]; weak_model_history=[]
X_train=dev.loc[weak_train_mask,WEAK_FEATURES]
X_tune=dev.loc[weak_tune_mask,WEAK_FEATURES]

for seed in WEAK_SEEDS:
    cat_path=ITER5_ROOT/f'weak_union_catboost_seed{seed}.cbm'
    cat=CatBoostClassifier(
        iterations=1400,depth=7,learning_rate=.03,loss_function='Logloss',eval_metric='PRAUC',
        l2_leaf_reg=10,random_strength=.35,random_seed=seed,task_type='GPU',devices='0',
        verbose=150,od_type='Iter',od_wait=120,allow_writing_files=False,
    )
    if REUSE_SAVED_MODELS and cat_path.exists():
        cat.load_model(cat_path)
    else:
        cat.fit(X_train,weak_train_y,sample_weight=weak_train_weights,
                eval_set=(X_tune,weak_tune_y),use_best_model=True)
        cat.save_model(cat_path)
    weak_cat_models.append(cat)

    lgb_path=ITER5_ROOT/f'weak_union_lightgbm_seed{seed}.joblib'
    if REUSE_SAVED_MODELS and lgb_path.exists():
        lgb=joblib.load(lgb_path)
    else:
        lgb=LGBMClassifier(
            objective='binary',n_estimators=1600,learning_rate=.025,num_leaves=31,
            max_depth=-1,min_child_samples=40,subsample=.85,colsample_bytree=.85,
            reg_alpha=1.0,reg_lambda=10.0,random_state=seed,n_jobs=-1,
            verbosity=-1,force_col_wise=True,
        )
        lgb.fit(X_train,weak_train_y,sample_weight=weak_train_weights,
                eval_set=[(X_tune,weak_tune_y)],eval_metric='average_precision',
                callbacks=[early_stopping(120,verbose=False),log_evaluation(0)])
        joblib.dump(lgb,lgb_path)
    weak_lgb_models.append(lgb)

    cat_best=cat.get_best_iteration()
    lgb_best=getattr(lgb,'best_iteration_',None)
    weak_model_history.append({
        'seed':seed,
        'catboost_best_iteration':int(cat_best if cat_best is not None and cat_best>=0 else cat.tree_count_-1),
        'lightgbm_best_iteration':int(lgb_best if lgb_best else lgb.n_estimators),
    })

print('Weak expert histories:',weak_model_history)


Weak expert histories: [{'seed': 17, 'catboost_best_iteration': 6, 'lightgbm_best_iteration': 30}, {'seed': 41, 'catboost_best_iteration': 277, 'lightgbm_best_iteration': 48}, {'seed': 67, 'catboost_best_iteration': 0, 'lightgbm_best_iteration': 45}]


In [28]:
dev['weak_cat_score']=np.mean([model.predict_proba(dev[WEAK_FEATURES])[:,1] for model in weak_cat_models],axis=0)
dev['weak_lgb_score']=np.mean([model.predict_proba(dev[WEAK_FEATURES])[:,1] for model in weak_lgb_models],axis=0)
dev['weak_consensus_min']=np.minimum(dev.weak_cat_score,dev.weak_lgb_score)
dev['weak_consensus_geom']=np.sqrt(np.clip(dev.weak_cat_score,0,1)*np.clip(dev.weak_lgb_score,0,1))

weak_score_columns=['weak_cat_score','weak_lgb_score','weak_consensus_min','weak_consensus_geom']
weak_model_validation=[]
for split in ['tune_model',*POLICY_BLOCKS]:
    part=dev.loc[dev.dev_split.eq(split)]
    union_y=part.anomaly_type.isin(WEAK_TYPES).astype(int)
    for score_col in weak_score_columns:
        weak_model_validation.append({
            'split':split,'target':'weak_union','score':score_col,
            'auprc':float(average_precision_score(union_y,part[score_col])),
        })
    for fault in WEAK_TYPES:
        fault_y=part.anomaly_type.eq(fault).astype(int)
        for score_col in weak_score_columns:
            weak_model_validation.append({
                'split':split,'target':fault,'score':score_col,
                'auprc':float(average_precision_score(fault_y,part[score_col])),
            })
weak_model_validation=pd.DataFrame(weak_model_validation)
weak_model_validation.to_csv(ITER5_ROOT/'iteration5_weak_model_validation.csv',index=False)
display(weak_model_validation.pivot(index=['split','target'],columns='score',values='auprc').round(4))


score                        weak_cat_score  weak_consensus_geom  \
split         target                                               
block_jul_sep bias                  0.02590              0.02630   
              drift                 0.01160              0.01350   
              frozen_sensor         0.28580              0.36020   
              weak_union            0.11240              0.13500   
block_may_jun bias                  0.00230              0.00540   
              drift                 0.01040              0.00770   
              frozen_sensor         0.01650              0.01130   
              weak_union            0.01410              0.01400   
block_oct_dec bias                  0.38420              0.33390   
              drift                 0.04330              0.04580   
              frozen_sensor         0.00310              0.00410   
              weak_union            0.29830              0.26470   
tune_model    bias                  0.00530              0.00620   
              drift                 0.00280              0.00380   
              frozen_sensor         0.02450              0.01900   
              weak_union            0.02220              0.02180   

score                        weak_consensus_min  weak_lgb_score  
split         target                                             
block_jul_sep bias                      0.02730         0.02300  
              drift                     0.01890         0.02020  
              frozen_sensor             0.28870         0.29240  
              weak_union                0.13110         0.13280  
block_may_jun bias                      0.00600         0.00600  
              drift                     0.00680         0.00640  
              frozen_sensor             0.01010         0.01000  
              weak_union                0.01330         0.01310  
block_oct_dec bias                      0.27040         0.26090  
              drift                     0.05600         0.05080  
              frozen_sensor             0.00410         0.00410  
              weak_union                0.21570         0.20500  
tune_model    bias                      0.00520         0.01150  
              drift                     0.00400         0.00400  
              frozen_sensor             0.01640         0.01630  
              weak_union                0.02000         0.02100

## 20. Causal consensus gate and two-block policy selection

An Iteration 5 rescue can begin only when both model families agree, the score persists across consecutive emitted readings, and the observation is not classified by the existing weather channel as a likely genuine regional event. A temporal gap resets the persistence counter.

Thresholds are selected using May–September 2023. October–December 2023 is an internal confirmation block and must independently pass every safeguard. This makes the experiment stricter than reusing a single block for both calibration and reporting.


In [29]:
def gap_aware_run_length(frame,score_col,threshold,max_gap_minutes):
    result=pd.Series(0,index=frame.index,dtype=int)
    for _,group in frame.groupby('station_id',sort=False):
        group=group.sort_values('emitted_timestamp_utc')
        scores=group[score_col].fillna(0).to_numpy(float)
        timestamps=group.emitted_timestamp_utc.tolist()
        runs=np.zeros(len(group),dtype=int); run=0; previous=None
        for position,(timestamp,score) in enumerate(zip(timestamps,scores)):
            if previous is not None and (timestamp-previous).total_seconds()/60>max_gap_minutes:
                run=0
            run=run+1 if score>=threshold else 0
            runs[position]=run
            previous=timestamp
        result.loc[group.index]=runs
    return result

def weak_episode_mean(metric):
    values=[metric['per_fault_episode_recall'].get(fault,np.nan) for fault in WEAK_TYPES]
    values=[value for value in values if not pd.isna(value)]
    return float(np.mean(values)) if values else float('nan')

WEAK_REFERENCE={}
dev['iteration3_reference']=False
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    baseline=apply_two_tier(part,selected.base_start,selected.base_continue,
                            selected.rescue_threshold,int(selected.min_points))
    baseline|=frozen_rule(part,SELECTED_FROZEN_CONFIG)
    dev.loc[part.index,'iteration3_reference']=baseline.to_numpy(bool)
    metrics=evaluate(part.assign(iteration3_reference=baseline),'base_score','iteration3_reference')
    WEAK_REFERENCE[block]=metrics

WEAK_SCORE_OPTIONS=['weak_consensus_min','weak_consensus_geom']
WEAK_THRESHOLDS=[.15,.25,.35,.45,.55,.65,.75]
WEAK_MIN_POINTS=[2,3,4,6]
WEAK_MAX_GAPS=[90,180]
WEAK_WEATHER_GUARDS=[True,False]
WEATHER_GUARD_THRESHOLD=float(weather_selected.threshold)
dev['weak_weather_safe']=dev.cat_weather_mean.fillna(0).lt(WEATHER_GUARD_THRESHOLD)

run_cache={}
for score_col in WEAK_SCORE_OPTIONS:
    for threshold in WEAK_THRESHOLDS:
        for max_gap in WEAK_MAX_GAPS:
            run_cache[(score_col,threshold,max_gap)]=gap_aware_run_length(dev,score_col,threshold,max_gap)

def evaluate_weak_policy(score_col,threshold,min_points,max_gap_minutes,weather_guard):
    rescue=run_cache[(score_col,threshold,max_gap_minutes)].ge(min_points)
    if weather_guard:
        rescue&=dev.weak_weather_safe
    candidate=dev.iteration3_reference|rescue
    rows=[]
    for block in POLICY_BLOCKS:
        part=dev.loc[dev.dev_split.eq(block)].copy()
        part['candidate']=candidate.loc[part.index].to_numpy(bool)
        metric=evaluate(part,'base_score','candidate')
        reference=WEAK_REFERENCE[block]
        weak_recall=weak_episode_mean(metric)
        base_weak_recall=weak_episode_mean(reference)
        rows.append({
            'block':block,**metric,
            'weak_episode_recall':weak_recall,
            'point_f1_delta':metric['f1']-reference['f1'],
            'event_f1_delta':metric['event_f1']-reference['event_f1'],
            'weak_episode_recall_delta':weak_recall-base_weak_recall,
        })
    summary={
        'score_col':score_col,'threshold':float(threshold),'min_points':int(min_points),
        'max_gap_minutes':int(max_gap_minutes),'weather_guard':bool(weather_guard),'rows':rows,
    }
    for scope,blocks in {'tune':['block_may_jun','block_jul_sep'],'all':POLICY_BLOCKS}.items():
        scoped=[row for row in rows if row['block'] in blocks]
        summary[f'{scope}_min_precision']=float(min(row['precision'] for row in scoped))
        summary[f'{scope}_max_false_alarm']=float(max(row['false_alarm_episodes_per_station_day'] for row in scoped))
        summary[f'{scope}_min_point_f1_delta']=float(min(row['point_f1_delta'] for row in scoped))
        summary[f'{scope}_mean_point_f1_delta']=float(np.mean([row['point_f1_delta'] for row in scoped]))
        summary[f'{scope}_min_event_f1_delta']=float(min(row['event_f1_delta'] for row in scoped))
        summary[f'{scope}_mean_event_f1_delta']=float(np.mean([row['event_f1_delta'] for row in scoped]))
        summary[f'{scope}_mean_weak_episode_recall_delta']=float(np.mean([row['weak_episode_recall_delta'] for row in scoped]))
    return summary

def passes_safety(summary,scope):
    return (
        summary[f'{scope}_min_precision']>=.75 and
        summary[f'{scope}_max_false_alarm']<=.02 and
        summary[f'{scope}_min_point_f1_delta']>=0 and
        summary[f'{scope}_min_event_f1_delta']>=-.01 and
        summary[f'{scope}_mean_event_f1_delta']>=0 and
        summary[f'{scope}_mean_point_f1_delta']>0 and
        summary[f'{scope}_mean_weak_episode_recall_delta']>0
    )

policy_candidates=[]
for score_col in WEAK_SCORE_OPTIONS:
    for threshold in WEAK_THRESHOLDS:
        for min_points in WEAK_MIN_POINTS:
            for max_gap in WEAK_MAX_GAPS:
                for weather_guard in WEAK_WEATHER_GUARDS:
                    policy_candidates.append(evaluate_weak_policy(
                        score_col,threshold,min_points,max_gap,weather_guard))

policy_frontier=pd.DataFrame([{key:value for key,value in row.items() if key!='rows'} for row in policy_candidates])
policy_frontier['passes_tune_gates']=policy_frontier.apply(lambda row:passes_safety(row.to_dict(),'tune'),axis=1)
policy_frontier['passes_all_gates']=policy_frontier.apply(lambda row:passes_safety(row.to_dict(),'all'),axis=1)
policy_frontier.to_csv(ITER5_ROOT/'iteration5_policy_frontier.csv',index=False)

tune_feasible=[candidate for candidate in policy_candidates if passes_safety(candidate,'tune')]
if tune_feasible:
    proposed=sorted(tune_feasible,key=lambda row:(
        row['tune_mean_weak_episode_recall_delta'],row['tune_mean_point_f1_delta'],
        row['tune_mean_event_f1_delta'],row['tune_min_precision']),reverse=True)[0]
    if passes_safety(proposed,'all'):
        ITER5_STATUS='constraints_met_with_confirmation'
        SELECTED_WEAK_POLICY={key:proposed[key] for key in ['score_col','threshold','min_points','max_gap_minutes','weather_guard']}
    else:
        ITER5_STATUS='failed_october_confirmation_keep_iteration3'
        SELECTED_WEAK_POLICY={'score_col':'weak_consensus_min','threshold':1.10,'min_points':999,'max_gap_minutes':90,'weather_guard':True}
else:
    proposed=sorted(policy_candidates,key=lambda row:(
        row['all_min_point_f1_delta'],row['all_mean_point_f1_delta'],row['all_mean_weak_episode_recall_delta']),reverse=True)[0]
    ITER5_STATUS='no_safe_development_gain_keep_iteration3'
    SELECTED_WEAK_POLICY={'score_col':'weak_consensus_min','threshold':1.10,'min_points':999,'max_gap_minutes':90,'weather_guard':True}

print('Tune-feasible candidates:',len(tune_feasible),'of',len(policy_candidates))
print('Iteration 5 status:',ITER5_STATUS)
display(pd.Series(SELECTED_WEAK_POLICY,name='selected_policy').to_frame())
display(policy_frontier.sort_values(['passes_all_gates','all_mean_weak_episode_recall_delta','all_mean_point_f1_delta'],ascending=False).head(12))


Tune-feasible candidates: 4 of 224
Iteration 5 status: constraints_met_with_confirmation


,selected_policy
score_col,weak_consensus_min
threshold,0.35000
min_points,4
max_gap_minutes,180
weather_guard,True


,score_col,threshold,min_points,max_gap_minutes,weather_guard,tune_min_precision,tune_max_false_alarm,tune_min_point_f1_delta,tune_mean_point_f1_delta,tune_min_event_f1_delta,tune_mean_event_f1_delta,tune_mean_weak_episode_recall_delta,all_min_precision,all_max_false_alarm,all_min_point_f1_delta,all_mean_point_f1_delta,all_min_event_f1_delta,all_mean_event_f1_delta,all_mean_weak_episode_recall_delta,passes_tune_gates,passes_all_gates
42,weak_consensus_min,0.35000,4,180,True,0.80147,0.01233,0.00377,0.00559,0.01017,0.01801,0.08333,0.80147,0.01742,0.00000,0.00373,0.00000,0.01201,0.05556,True,True
43,weak_consensus_min,0.35000,4,180,False,0.80147,0.01233,0.00377,0.00559,0.01017,0.01801,0.08333,0.80147,0.01742,0.00000,0.00373,0.00000,0.01201,0.05556,True,True
115,weak_consensus_geom,0.15000,2,180,False,0.03080,0.42820,-0.43989,-0.34670,-0.54821,-0.54047,0.54167,0.03080,0.42820,-0.43989,-0.36964,-0.54821,-0.53138,0.47222,False,False
114,weak_consensus_geom,0.15000,2,180,True,0.02990,0.42820,-0.43903,-0.34708,-0.54834,-0.54053,0.54167,0.02990,0.42820,-0.43903,-0.36978,-0.54834,-0.53143,0.47222,False,False
123,weak_consensus_geom,0.15000,4,180,False,0.04940,0.21067,-0.36401,-0.29546,-0.51224,-0.49039,0.41667,0.04940,0.21067,-0.36401,-0.31629,-0.51224,-0.47746,0.38889,False,False
122,weak_consensus_geom,0.15000,4,180,True,0.04838,0.21176,-0.36278,-0.29570,-0.51264,-0.49083,0.41667,0.04838,0.21176,-0.36278,-0.31630,-0.51264,-0.47775,0.38889,False,False
119,weak_consensus_geom,0.15000,3,180,False,0.03875,0.29614,-0.40432,-0.32304,-0.53559,-0.51981,0.41667,0.03875,0.29614,-0.40432,-0.34449,-0.53559,-0.50766,0.38889,False,False
118,weak_consensus_geom,0.15000,3,180,True,0.03755,0.29723,-0.40332,-0.32357,-0.53581,-0.51992,0.41667,0.03755,0.29723,-0.40332,-0.34472,-0.53581,-0.50773,0.38889,False,False
2,weak_consensus_min,0.15000,2,180,True,0.13465,0.15025,-0.26414,-0.20292,-0.48199,-0.45729,0.37500,0.13465,0.15025,-0.26414,-0.20118,-0.48199,-0.42648,0.32407,False,False
3,weak_consensus_min,0.15000,2,180,False,0.13584,0.15079,-0.26496,-0.20248,-0.48235,-0.45786,0.37500,0.13584,0.15079,-0.26496,-0.20140,-0.48235,-0.42657,0.32407,False,False


## 21. Full three-block ablation and fault coverage

In [30]:
def materialize_weak_rescue(frame,policy):
    if int(policy['min_points'])>100:
        return pd.Series(False,index=frame.index)
    run=gap_aware_run_length(frame,policy['score_col'],float(policy['threshold']),int(policy['max_gap_minutes']))
    rescue=run.ge(int(policy['min_points']))
    if bool(policy['weather_guard']):
        rescue&=frame.cat_weather_mean.fillna(0).lt(WEATHER_GUARD_THRESHOLD)
    return rescue

rows=[]; combined=[]
dev['iteration5_rescue']=materialize_weak_rescue(dev,SELECTED_WEAK_POLICY)
for block in POLICY_BLOCKS:
    part=dev.loc[dev.dev_split.eq(block)].copy()
    part['iteration3']=part.iteration3_reference.to_numpy(bool)
    # Slice the full causal run so persistence before a block boundary is preserved exactly as evaluated.
    part['weak_rescue']=dev.loc[part.index,'iteration5_rescue'].to_numpy(bool)
    part['iteration5']=part.iteration3|part.weak_rescue
    for variant,pred_col in [('Iteration3 reference','iteration3'),('Iteration5 weak consensus','iteration5')]:
        metric=evaluate(part,'base_score',pred_col)
        metric['weak_episode_recall']=weak_episode_mean(metric)
        rows.append({'block':block,'variant':variant,**metric})
    combined.append(part)

ablation5=pd.DataFrame(rows)
ablation5.to_csv(ITER5_ROOT/'iteration5_multiblock_ablation.csv',index=False)
display(ablation5[['block','variant','precision','recall','f1','event_precision','event_recall','event_f1',
                   'weak_episode_recall','false_alarm_episodes_per_station_day','delay_mean_min','delay_p90_min']])

combined=pd.concat(combined,ignore_index=True)
combined['pred']=combined.iteration5
fault_recall5=(pd.Series(event_metrics(combined,'pred')['per_fault_episode_recall'],name='episode_recall')
               .sort_values().rename_axis('anomaly_type').reset_index())
fault_recall5.to_csv(ITER5_ROOT/'iteration5_fault_episode_recall.csv',index=False)
point_recall5=(combined.loc[combined.is_anomaly.eq(1)].groupby('anomaly_type').pred.mean()
               .rename('point_recall').sort_values().rename_axis('anomaly_type').reset_index())
point_recall5.to_csv(ITER5_ROOT/'iteration5_point_fault_recall.csv',index=False)
display(fault_recall5); display(point_recall5)


,block,variant,precision,recall,f1,event_precision,event_recall,event_f1,weak_episode_recall,false_alarm_episodes_per_station_day,delay_mean_min,delay_p90_min
0,block_may_jun,Iteration3 reference,0.80000,0.44444,0.57143,0.51613,0.72727,0.60377,0.00000,0.01233,86.25000,300.00000
1,block_may_jun,Iteration5 weak consensus,0.80147,0.44856,0.57520,0.53125,0.77273,0.62963,0.16667,0.01233,176.47059,432.00000
2,block_jul_sep,Iteration3 reference,0.88608,0.18667,0.30837,0.54545,0.66667,0.60000,0.41667,0.00817,190.00000,630.00000
3,block_jul_sep,Iteration5 weak consensus,0.88889,0.19200,0.31579,0.56250,0.66667,0.61017,0.41667,0.00762,190.00000,630.00000
4,block_oct_dec,Iteration3 reference,0.83262,0.41189,0.55114,0.47541,0.80556,0.59794,0.33333,0.01742,93.10345,444.00000
5,block_oct_dec,Iteration5 weak consensus,0.83262,0.41189,0.55114,0.47541,0.80556,0.59794,0.33333,0.01742,93.10345,444.00000


,anomaly_type,episode_recall
0,duplicate_packet,0.00000
1,frozen_sensor,0.16667
2,drift,0.37500
3,bias,0.42857
4,noise,0.85714
5,sudden_drop,0.87500
6,multi_sensor_failure,1.00000
7,communication_corruption,1.00000
8,scaling_error,1.00000
9,spike,1.00000


,anomaly_type,point_recall
0,duplicate_packet,0.00000
1,drift,0.06085
2,frozen_sensor,0.23077
3,bias,0.36052
4,noise,0.46789
5,multi_sensor_failure,0.72251
6,sudden_drop,0.87500
7,communication_corruption,1.00000
8,scaling_error,1.00000
9,spike,1.00000


## 22. Save result package and stop rule

The selected policy is a deliberately impossible threshold whenever no candidate passes all safeguards. In that case the reported Iteration 5 rows are the unmodified Iteration 3 reference, not a disguised lower-quality model.


In [31]:
iteration5_rows=ablation5.loc[ablation5.variant.eq('Iteration5 weak consensus')]
operational_fault_recall=dict(zip(fault_recall5.anomaly_type,fault_recall5.episode_recall))
operational_fault_recall['duplicate_packet']=float(communication_result['duplicate_episode_recall'])
result5={
    'iteration':'05_episode_balanced_weak_fault_consensus',
    'device':DEVICE,'gpu':torch.cuda.get_device_name(0),
    'final_tests_opened':False,
    'status':ITER5_STATUS,
    'source_baseline':'Iteration3 trigger; Iteration4 was an identical zero-grace fallback',
    'model_families':['episode-balanced CatBoost weak-union ensemble','episode-balanced LightGBM weak-union ensemble'],
    'feature_count':len(WEAK_FEATURES),'weak_fault_types':list(WEAK_TYPES),
    'episode_weight_audit':weak_weight_audit,
    'model_history':weak_model_history,
    'selected_policy':SELECTED_WEAK_POLICY,
    'policy_candidates':int(len(policy_candidates)),
    'tune_feasible_candidates':int(len(tune_feasible)),
    'promotion_gates':{
        'min_precision':.75,'max_false_alarm_episodes_per_station_day':.02,
        'minimum_point_f1_delta':0.0,'minimum_event_f1_delta':-.01,
        'minimum_mean_event_f1_delta':0.0,'positive_mean_point_f1_delta':True,
        'positive_mean_weak_episode_recall_delta':True,
        'october_december_confirmation_required':True,
    },
    'iteration5_blocks':iteration5_rows.drop(columns=['per_fault_episode_recall'],errors='ignore').to_dict('records'),
    'fault_episode_recall_static':dict(zip(fault_recall5.anomaly_type,fault_recall5.episode_recall)),
    'fault_episode_recall_operational':operational_fault_recall,
    'point_fault_recall':dict(zip(point_recall5.anomaly_type,point_recall5.point_recall)),
    'communication_coverage':{
        'replay_duplicate_packet_precision':float(communication_result['duplicate_packet_precision']),
        'replay_duplicate_packet_recall':float(communication_result['duplicate_packet_recall']),
        'replay_duplicate_episode_recall':float(communication_result['duplicate_episode_recall']),
    },
}
(ITER5_ROOT/'iteration5_result_block.json').write_text(json.dumps(result5,indent=2,default=float))
(ITER5_ROOT/'iteration5_feature_contract.json').write_text(json.dumps({
    'features':WEAK_FEATURES,'forbidden_features':['hour_sin','hour_cos','day_of_year_sin','day_of_year_cos','temperature_dewpoint_spread_c'],
    'causal':True,'detector_inputs':['temperature','pressure','relative_humidity']
},indent=2))
print(json.dumps(result5,indent=2,default=float))
print('\nHISTORICAL CHECKPOINT FILES - continue through Iteration 8; do not return these yet:')
for name in ['iteration5_result_block.json','iteration5_policy_frontier.csv','iteration5_multiblock_ablation.csv',
             'iteration5_fault_episode_recall.csv','iteration5_point_fault_recall.csv','iteration5_weak_model_validation.csv']:
    print(ITER5_ROOT/name)


{
  "iteration": "05_episode_balanced_weak_fault_consensus",
  "device": "cuda",
  "gpu": "Tesla T4",
  "final_tests_opened": false,
  "status": "constraints_met_with_confirmation",
  "source_baseline": "Iteration3 trigger; Iteration4 was an identical zero-grace fallback",
  "model_families": [
    "episode-balanced CatBoost weak-union ensemble",
    "episode-balanced LightGBM weak-union ensemble"
  ],
  "feature_count": 108,
  "weak_fault_types": [
    "frozen_sensor",
    "bias",
    "drift"
  ],
  "episode_weight_audit": {
    "positive_rows": 1397,
    "positive_episodes": 45,
    "median_episode_rows": 20.0,
    "max_episode_rows": 95,
    "positive_weight_sum": 15889.393474893872,
    "negative_weight_sum": 180725.0
  },
  "model_history": [
    {
      "seed": 17,
      "catboost_best_iteration": 6,
      "lightgbm_best_iteration": 30
    },
    {
      "seed": 41,
      "catboost_best_iteration": 277,
      "lightgbm_best_iteration": 48
    },
    {
      "seed": 67,
      "cat

## Promotion decision

Promote Iteration 5 only when a policy selected without October–December labels also passes October–December confirmation and every one of these all-block requirements:

- point precision at least 75%;
- false-alert episodes at most 0.02 per station-day;
- no point-F1 regression in any block;
- no event-F1 regression larger than one percentage point in any block;
- positive mean point-F1 gain;
- positive mean weak-fault episode-recall gain.

Otherwise keep Iteration 3/2 unchanged. Regardless of the result, do not open the previously inspected 2024 benchmark. The next trustworthy promotion step is a newly created blind time/station benchmark.


# Iteration 8 controlled new-data phase

The frozen Iteration 5 development reference and former blind benchmark are immutable. This section opens only the new DWD 2022–2023 development bundle. It does not load DWD 2024, NOAA 2024, or any 2025 observation/label file.


In [32]:
import sys,shutil

ITER8_ROOT=DRIVE_ROOT/'experiments'/'iteration_08_multiclimate_data_curriculum'
ITER8_ROOT.mkdir(parents=True,exist_ok=True)
ITER8_BUNDLE_ZIP=DRIVE_ROOT/'SkyGuard_Iteration8_Development_Data_Bundle.zip'
ITER8_EXTRACT_ROOT=Path('/content/skyguard_iteration8')
ITER8_DATA_ROOT=ITER8_EXTRACT_ROOT/'SkyGuard_Iteration8_Development_Data_Bundle'
ITER8_EXPECTED_SHA='7f47466805309faf528d6abc8f04a588c4accbaf454989e36a2b4ff5b31681d4'
OLD_EXPECTED_SHA='9329f02c1a05241b109c50b0ed3ba5bc59761eb92652f77509bda7409faf182a'

def sha256_file(path):
    digest=hashlib.sha256()
    with open(path,'rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()

assert UNLOCK_FINAL_TESTS is False
assert ITER8_BUNDLE_ZIP.exists(),f'Missing {ITER8_BUNDLE_ZIP}'
assert sha256_file(ITER8_BUNDLE_ZIP)==ITER8_EXPECTED_SHA,'Iteration 8 development bundle hash mismatch.'
assert sha256_file(BUNDLE_ZIP)==OLD_EXPECTED_SHA,'Original GPU data bundle hash mismatch.'
ITER8_EXTRACT_ROOT.mkdir(parents=True,exist_ok=True)
if not ITER8_DATA_ROOT.exists():
    with zipfile.ZipFile(ITER8_BUNDLE_ZIP) as archive: archive.extractall(ITER8_EXTRACT_ROOT)
members=[str(path.relative_to(ITER8_DATA_ROOT)).lower() for path in ITER8_DATA_ROOT.rglob('*') if path.is_file()]
assert not any('2024' in name or '2025' in name for name in members),'Locked year leaked into development bundle.'
sys.path.insert(0,str(ITER8_DATA_ROOT/'src'))
print({'iteration8_root':str(ITER8_ROOT),'bundle_sha256':ITER8_EXPECTED_SHA,'files':len(members)})


{'iteration8_root': '/content/drive/MyDrive/SkyGuard_AI_GPU/experiments/iteration_08_multiclimate_data_curriculum', 'bundle_sha256': '7f47466805309faf528d6abc8f04a588c4accbaf454989e36a2b4ff5b31681d4', 'files': 20}


## 32. Load and audit the new official DWD development corpus

The model sees temperature, sea-level-reduced pressure and relative humidity. Raw station pressure, source quality code, station elevation and transformation metadata remain audit fields and are not model features. Dew point is blanked before feature generation.


In [33]:
from skyguard.faults.curriculum import (
    FAULT_FAMILIES,WEATHER_FAMILIES,MultiClimateCurriculum)
from skyguard.features.builder import FeatureBuilder
from skyguard.features.contracts import FeatureConfig,OUTPUT_COLUMNS
from skyguard.features.neighbors import NeighborIndex
from skyguard.features.temporal import TemporalFeatureBuilder
from skyguard.features.phase10 import PHASE10_FEATURES,add_phase10_features,fit_climatology

I8_DATA=ITER8_DATA_ROOT/'data'
I8_CONFIG=ITER8_DATA_ROOT/'config'/'iteration8_dwd_stations.csv'
i8_source_report=json.loads((ITER8_DATA_ROOT/'reports'/'iteration8_dwd_data_validation.json').read_text())
assert i8_source_report['status']=='PASS'
assert i8_source_report['split_safety']['2025_loaded'] is False

def load_dwd(year):
    frame=pd.read_csv(I8_DATA/f'dwd_aws_10min_{year}.csv.gz',low_memory=False)
    frame['station_id']=frame.station_id.astype(str)
    timestamp=pd.to_datetime(frame.timestamp_utc,utc=True)
    frame=frame.loc[timestamp.dt.minute.eq(0)].copy().reset_index(drop=True)
    frame['timestamp_utc']=pd.to_datetime(frame.timestamp_utc,utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    frame['dew_point_c']=''
    return frame

dwd22_raw=load_dwd(2022); dwd23_raw=load_dwd(2023)
assert dwd22_raw.station_id.nunique()==16 and dwd23_raw.station_id.nunique()==16
assert dwd22_raw.cluster.nunique()==4 and dwd23_raw.cluster.nunique()==4
assert dwd22_raw.year.eq(2022).all() and dwd23_raw.year.eq(2023).all()
assert not any('2024' in value or '2025' in value for value in dwd22_raw.source.astype(str).unique())
display(pd.DataFrame([
    {'year':2022,'rows_10min':i8_source_report['yearly'][0]['rows'],'hourly_rows':len(dwd22_raw),'stations':dwd22_raw.station_id.nunique()},
    {'year':2023,'rows_10min':i8_source_report['yearly'][1]['rows'],'hourly_rows':len(dwd23_raw),'stations':dwd23_raw.station_id.nunique()},
]))


,year,rows_10min,hourly_rows,stations
0,2022,838126,139684,16
1,2023,838307,139716,16


## 33. Build balanced physical weather and fault curricula

Every 2023 scope contains all four climates, all six weather families and all five fault families. Events never cross scope boundaries or overlap. Genuine weather is a negative class for the fault detector.


In [34]:
i8_train_injector=MultiClimateCurriculum(dwd22_raw,'dwd_train',seed=8017)
i8_train_injector.build_training('2022-01-08','2022-12-22',weather_repetitions=4,fault_repetitions=8)
i8_validation_scopes={
    'tune':('2023-01-08','2023-04-24'),
    'discovery':('2023-05-05','2023-08-25'),
    'confirmation':('2023-09-05','2023-12-22'),
}
i8_val_injector=MultiClimateCurriculum(dwd23_raw,'dwd_validation',seed=8023)
i8_val_injector.build_validation(i8_validation_scopes)
i8_train_audit=i8_train_injector.validate(); i8_val_audit=i8_val_injector.validate()
assert i8_train_audit['status']=='PASS' and i8_val_audit['status']=='PASS'
assert i8_train_audit['weather_events']==96 and i8_train_audit['fault_events']==160
assert i8_val_audit['weather_events']==72 and i8_val_audit['fault_events']==60

i8_train_events=i8_train_injector.event_frame(); i8_val_events=i8_val_injector.event_frame()
coverage=(i8_val_events.groupby(['scope','cluster','label_category','anomaly_type']).size()
          .rename('events').reset_index())
assert coverage.events.min()>=1
i8_events=pd.concat([i8_train_events,i8_val_events],ignore_index=True)
i8_events.to_csv(ITER8_ROOT/'iteration8_curriculum_events.csv',index=False)
dwd22=i8_train_injector.frame; dwd23=i8_val_injector.frame
display(pd.DataFrame([i8_train_audit,i8_val_audit],index=['train','validation']))
display(coverage)
del dwd22_raw,dwd23_raw


,status,rows,event_rows,event_row_fraction,weather_events,fault_events,weather_families,fault_families,errors
train,PASS,139684,11498,0.08231,96,160,"[dry_intrusion, humidity_surge, multivariate_s...","[bias, drift, frozen_sensor, noise, spike]",[]
validation,PASS,139716,7241,0.05183,72,60,"[dry_intrusion, humidity_surge, multivariate_s...","[bias, drift, frozen_sensor, noise, spike]",[]


,scope,cluster,label_category,anomaly_type,events
0,confirmation,dwd_east_continental,genuine_weather_scenario,dry_intrusion,1
1,confirmation,dwd_east_continental,genuine_weather_scenario,humidity_surge,1
2,confirmation,dwd_east_continental,genuine_weather_scenario,multivariate_storm,1
3,confirmation,dwd_east_continental,genuine_weather_scenario,pressure_front,1
4,confirmation,dwd_east_continental,genuine_weather_scenario,regional_cold_surge,1
...,...,...,...,...,...
127,tune,dwd_west_lowland,sensor_fault,bias,1
128,tune,dwd_west_lowland,sensor_fault,drift,1
129,tune,dwd_west_lowland,sensor_fault,frozen_sensor,1
130,tune,dwd_west_lowland,sensor_fault,noise,1


## 34. Generate the exact 108-feature causal contract

The same production feature code is applied to the DWD domain. Neighbour lookup uses observations at or before the current timestamp only; station trends use current and prior observations only. The full feature tables are cached in Drive.


In [35]:
I8_DWD_TRAIN_FEATURES=ITER8_ROOT/'dwd_train_phase10_features.csv.gz'
I8_DWD_VAL_FEATURES=ITER8_ROOT/'dwd_validation_phase10_features.csv.gz'
I8_DWD_PROFILE=ITER8_ROOT/'dwd_phase10_climatology.joblib'
i8_station_frame=pd.read_csv(I8_CONFIG,dtype={'station_id':str,'dwd_numeric_id':str})
i8_stations={str(row['station_id']):{key:str(value) for key,value in row.items()}
              for row in i8_station_frame.to_dict('records')}
i8_intervals={station:60.0 for station in i8_stations}
i8_feature_config=FeatureConfig(max_neighbors=3,neighbor_tolerance_minutes=90.0)

def i8_enforce_numeric_features(frame,label):
    missing=[feature for feature in FEATURES if feature not in frame.columns]
    if missing:
        raise ValueError(f'{label} is missing {len(missing)} contracted features: {missing[:8]}')
    for feature in FEATURES:
        numeric=pd.to_numeric(frame[feature],errors='coerce')
        frame[feature]=numeric.replace([np.inf,-np.inf],np.nan).astype(np.float32)
    bad=[feature for feature in FEATURES
         if not (pd.api.types.is_integer_dtype(frame[feature])
                 or pd.api.types.is_float_dtype(frame[feature])
                 or pd.api.types.is_bool_dtype(frame[feature]))]
    if bad:
        raise TypeError(f'{label} has non-numeric contracted features: {bad}')
    print(label,'numeric feature contract: PASS | features:',len(FEATURES))
    return frame

def i8_base_features(frame,label):
    source=frame.sort_values(['station_id','timestamp_utc'],kind='stable').reset_index(drop=True)
    rows=source.to_dict('records')
    builder=FeatureBuilder(
        TemporalFeatureBuilder(i8_intervals,i8_feature_config),
        NeighborIndex(rows,i8_stations,i8_feature_config))
    output=[]
    for position,row in enumerate(rows,1):
        output.append(builder.transform(row))
        if position%50000==0: print(label,position,'/',len(rows))
    return pd.DataFrame(output,columns=OUTPUT_COLUMNS)

if REUSE_SAVED_MODELS and I8_DWD_TRAIN_FEATURES.exists() and I8_DWD_VAL_FEATURES.exists() and I8_DWD_PROFILE.exists():
    dwd_train=pd.read_csv(I8_DWD_TRAIN_FEATURES,low_memory=False)
    dwd_val=pd.read_csv(I8_DWD_VAL_FEATURES,low_memory=False)
    i8_profiles=joblib.load(I8_DWD_PROFILE)
    print('Reused cached DWD Phase 10 features.')
else:
    dwd_train_base=i8_base_features(dwd22,'DWD 2022')
    i8_profiles=fit_climatology(dwd_train_base)
    joblib.dump(i8_profiles,I8_DWD_PROFILE,compress=3)
    dwd_train=add_phase10_features(dwd_train_base,i8_profiles)
    dwd_train.to_csv(I8_DWD_TRAIN_FEATURES,index=False,compression={'method':'gzip','compresslevel':6})
    del dwd_train_base
    dwd_val_base=i8_base_features(dwd23,'DWD 2023')
    dwd_val=add_phase10_features(dwd_val_base,i8_profiles)
    dwd_val.to_csv(I8_DWD_VAL_FEATURES,index=False,compression={'method':'gzip','compresslevel':6})
    del dwd_val_base

dwd_train=i8_enforce_numeric_features(dwd_train,'DWD train')
dwd_val=i8_enforce_numeric_features(dwd_val,'DWD validation')
assert list(PHASE10_FEATURES)==FEATURES
assert set(FEATURES)<=set(dwd_train.columns) and set(FEATURES)<=set(dwd_val.columns)
assert not ({'temperature_dewpoint_spread_c','hour_sin','hour_cos','day_of_year_sin','day_of_year_cos'}&set(FEATURES))
for frame in (dwd_train,dwd_val):
    frame['station_id']=frame.station_id.astype(str)
    frame['emitted_timestamp_utc']=pd.to_datetime(frame.emitted_timestamp_utc,utc=True)
    frame['episode_id']=frame.episode_id.fillna('').astype(str)
    frame['available_to_detector']=pd.to_numeric(frame.available_to_detector).fillna(0).astype(int)
print({'dwd_train_features':dwd_train.shape,'dwd_validation_features':dwd_val.shape})
del dwd22,dwd23


Reused cached DWD Phase 10 features.
DWD train numeric feature contract: PASS | features: 108
DWD validation numeric feature contract: PASS | features: 108
{'dwd_train_features': (139684, 136), 'dwd_validation_features': (139716, 136)}


## 35. Freeze DWD pseudo-unseen stations and reconstruct the accepted baseline

One station per climate is excluded from all candidate fitting. The existing India-trained detector is evaluated on DWD without adapting its thresholds; this measures genuine domain transfer.


In [36]:
# Recovery guard: a completed Cell 34 must provide both feature tables.
i8_missing_tables=[name for name in ('dwd_train','dwd_val') if name not in globals()]
if i8_missing_tables:
    if (I8_DWD_TRAIN_FEATURES.exists() and I8_DWD_VAL_FEATURES.exists()
            and I8_DWD_PROFILE.exists()):
        dwd_train=pd.read_csv(I8_DWD_TRAIN_FEATURES,low_memory=False)
        dwd_val=pd.read_csv(I8_DWD_VAL_FEATURES,low_memory=False)
        i8_profiles=joblib.load(I8_DWD_PROFILE)
        for frame in (dwd_train,dwd_val):
            frame['station_id']=frame.station_id.astype(str)
            frame['emitted_timestamp_utc']=pd.to_datetime(frame.emitted_timestamp_utc,utc=True)
            frame['episode_id']=frame.episode_id.fillna('').astype(str)
            frame['available_to_detector']=pd.to_numeric(frame.available_to_detector).fillna(0).astype(int)
            i8_enforce_numeric_features(frame,'Recovered DWD feature table')
        print('Recovered cached DWD feature tables after an interrupted/out-of-order run.')
    else:
        raise RuntimeError(
            'Cell 34 did not complete, so dwd_train/dwd_val do not exist and no cache is available. '
            'Restore the complete corrected Cell 34, run it to completion, then rerun Cell 35.'
        )
del i8_missing_tables

I8_DWD_HOLDOUTS=['DWD-05839','DWD-01684','DWD-04336','DWD-00232']
dwd_train['i8_domain']='dwd'; dwd_val['i8_domain']='dwd'
dwd_train['i8_scope']='train'
dwd_val['i8_scope']=np.select(
    [dwd_val.emitted_timestamp_utc<'2023-05-01',dwd_val.emitted_timestamp_utc<'2023-09-01'],
    ['tune','discovery'],default='confirmation')
i8_scope_map=i8_val_events.set_index('episode_id').scope.to_dict()
event_mask=dwd_val.episode_id.ne('')
dwd_val.loc[event_mask,'i8_scope']=dwd_val.loc[event_mask,'episode_id'].map(i8_scope_map)
assert dwd_val.loc[event_mask].groupby('episode_id').i8_scope.nunique().max()==1

def i8_old_scores(frame):
    X=frame[FEATURES]
    frame['cat_fault_mean']=np.mean([model.predict_proba(X)[:,1] for model in fault_models],axis=0)
    frame['cat_weather_mean']=np.mean([model.predict_proba(X)[:,1] for model in weather_models],axis=0)
    frame['base_score']=apply_platt(base_calibrator,frame.cat_fault_mean)
    for fault,models in specialist_models.items():
        raw=np.mean([model.predict_proba(frame[SPECIALIST_FEATURES[fault]])[:,1] for model in models],axis=0)
        frame[f'{fault}_raw']=raw
        frame[f'{fault}_score']=apply_platt(calibrators[fault],raw)
    frame['rescue_score']=frame[[f'{fault}_score' for fault in SPECIALIST_FEATURES]].max(axis=1)
    scored=add_hard_rules(frame)
    frame['hard_rule']=scored.hard_rule.to_numpy(bool)
    baseline=apply_two_tier(frame,selected.base_start,selected.base_continue,
                            selected.rescue_threshold,int(selected.min_points))
    baseline|=frozen_rule(frame,SELECTED_FROZEN_CONFIG)
    frame['i8_baseline_pred']=baseline.to_numpy(bool)
    frame['i8_baseline_weather']=frame.cat_weather_mean.ge(WEATHER_GUARD_THRESHOLD)
    return frame

dwd_val=i8_old_scores(dwd_val)
print('DWD pseudo-unseen rows:',int(dwd_val.station_id.isin(I8_DWD_HOLDOUTS).sum()))


DWD pseudo-unseen rows: 34920


## 36. Train domain-balanced CatBoost + LightGBM challengers

India and DWD receive equal aggregate training weight. Positive episode mass is balanced so long incidents cannot dominate. Thresholds are not fitted here; early stopping uses only January–April 2023 tune rows.


In [37]:
i8_old_train=dev.loc[dev.dev_split.eq('train')&dev.available_to_detector.eq(1)].copy()
i8_old_tune=dev.loc[dev.dev_split.eq('tune_model')&dev.available_to_detector.eq(1)].copy()
i8_old_train['i8_domain']='india'; i8_old_tune['i8_domain']='india'
i8_dwd_fit=dwd_train.loc[~dwd_train.station_id.isin(I8_DWD_HOLDOUTS)&dwd_train.available_to_detector.eq(1)].copy()
i8_dwd_tune=dwd_val.loc[dwd_val.i8_scope.eq('tune')&~dwd_val.station_id.isin(I8_DWD_HOLDOUTS)&dwd_val.available_to_detector.eq(1)].copy()
i8_fit=pd.concat([i8_old_train,i8_dwd_fit],ignore_index=True,sort=False)
i8_tune=pd.concat([i8_old_tune,i8_dwd_tune],ignore_index=True,sort=False)
i8_fit=i8_enforce_numeric_features(i8_fit,'Combined training table')
i8_tune=i8_enforce_numeric_features(i8_tune,'Combined tuning table')

def i8_target(frame,target):
    return frame.is_anomaly.astype(int).to_numpy() if target=='fault' else frame.is_weather_event.astype(int).to_numpy()

def i8_domain_episode_weights(frame,y):
    y=np.asarray(y,bool); weights=np.ones(len(frame),dtype=float)
    positions=np.flatnonzero(y)
    if len(positions):
        positive=frame.iloc[positions][['episode_id','row_id']].copy()
        keys=positive.episode_id.fillna('').astype(str)
        keys=keys.where(keys.ne(''),'single_'+positive.row_id.astype(str))
        lengths=keys.value_counts(); mass=keys.map(lambda key:1.0/lengths.loc[key]).to_numpy(float)
        mass*=len(mass)/max(mass.sum(),1e-12)
        weights[positions]=mass
        imbalance=(len(frame)-len(positions))/max(weights[positions].sum(),1e-12)
        weights[positions]*=min(math.sqrt(imbalance),18.0)
    for _,indices in frame.groupby('i8_domain').indices.items():
        indices=np.asarray(indices,dtype=int)
        weights[indices]*=len(frame)/(len(frame.i8_domain.unique())*max(weights[indices].sum(),1e-12))
    return weights

I8_LGB_SEEDS=[17,41]
i8_models={}; i8_training_history=[]
for target in ['fault','weather']:
    y_fit=i8_target(i8_fit,target); y_tune=i8_target(i8_tune,target)
    weights=i8_domain_episode_weights(i8_fit,y_fit)
    cat_path=ITER8_ROOT/f'iteration8_{target}_catboost.cbm'
    cat=CatBoostClassifier(
        iterations=1500,depth=7,learning_rate=.03,loss_function='Logloss',eval_metric='PRAUC',
        l2_leaf_reg=12,random_strength=.35,random_seed=81,task_type='GPU',devices='0',
        verbose=150,od_type='Iter',od_wait=140,allow_writing_files=False)
    if REUSE_SAVED_MODELS and cat_path.exists(): cat.load_model(cat_path)
    else:
        cat.fit(i8_fit[FEATURES],y_fit,sample_weight=weights,
                eval_set=(i8_tune[FEATURES],y_tune),use_best_model=True)
        cat.save_model(cat_path)
    lgb_models=[]
    for seed in I8_LGB_SEEDS:
        path=ITER8_ROOT/f'iteration8_{target}_lightgbm_seed{seed}.joblib'
        if REUSE_SAVED_MODELS and path.exists(): lgb=joblib.load(path)
        else:
            lgb=LGBMClassifier(
                objective='binary',n_estimators=1800,learning_rate=.025,num_leaves=31,
                min_child_samples=60,subsample=.85,colsample_bytree=.80,
                reg_alpha=1.0,reg_lambda=12.0,random_state=seed,n_jobs=-1,
                verbosity=-1,force_col_wise=True)
            lgb.fit(i8_fit[FEATURES],y_fit,sample_weight=weights,
                    eval_set=[(i8_tune[FEATURES],y_tune)],eval_metric='average_precision',
                    callbacks=[early_stopping(140,verbose=False),log_evaluation(0)])
            joblib.dump(lgb,path)
        lgb_models.append(lgb)
    i8_models[target]={'cat':cat,'lgb':lgb_models}
    i8_training_history.append({
        'target':target,'fit_rows':len(i8_fit),'fit_positives':int(y_fit.sum()),
        'tune_rows':len(i8_tune),'tune_positives':int(y_tune.sum()),
        'catboost_best_iteration':int(max(cat.get_best_iteration(),0)),
        'lightgbm_best_iterations':','.join(str(getattr(model,'best_iteration_',0)) for model in lgb_models)})

pd.DataFrame(i8_training_history).to_csv(ITER8_ROOT/'iteration8_training_history.csv',index=False)
display(pd.DataFrame(i8_training_history))


Combined training table numeric feature contract: PASS | features: 108
Combined tuning table numeric feature contract: PASS | features: 108


,target,fit_rows,fit_positives,tune_rows,tune_positives,catboost_best_iteration,lightgbm_best_iterations
0,fault,286952,5228,92327,845,1490,"857,959"
1,weather,286952,6341,92327,1679,1498,"1371,1779"


In [38]:
def i8_add_candidate_scores(frame):
    frame=i8_enforce_numeric_features(frame,'Candidate scoring table')
    X=frame[FEATURES]
    for target,bundle8 in i8_models.items():
        cat=bundle8['cat'].predict_proba(X)[:,1]
        lgb=np.mean([model.predict_proba(X)[:,1] for model in bundle8['lgb']],axis=0)
        frame[f'i8_{target}_cat']=cat
        frame[f'i8_{target}_lgb']=lgb
        frame[f'i8_{target}_score']=np.sqrt(np.clip(cat,0,1)*np.clip(lgb,0,1))
    return frame

i8_old_eval=dev.loc[~dev.dev_split.eq('train')&dev.available_to_detector.eq(1)].copy()
i8_old_eval['i8_domain']='india'
i8_old_eval['i8_scope']=i8_old_eval.dev_split.map({
    'tune_model':'tune','block_may_jun':'discovery','block_jul_sep':'discovery','block_oct_dec':'confirmation'})
i8_reference_columns={'iteration3_reference','iteration5_rescue'}
assert i8_reference_columns.issubset(i8_old_eval.columns),(
    'Accepted Iteration 5 reference components are missing: '
    f'{sorted(i8_reference_columns-set(i8_old_eval.columns))}')
i8_old_eval['i8_baseline_pred']=(
    i8_old_eval['iteration3_reference'].astype(bool)
    |i8_old_eval['iteration5_rescue'].astype(bool))
i8_old_eval['i8_baseline_weather']=i8_old_eval.cat_weather_mean.ge(WEATHER_GUARD_THRESHOLD)
i8_old_eval=i8_add_candidate_scores(i8_old_eval)
dwd_val=i8_add_candidate_scores(dwd_val)
i8_validation=pd.concat([i8_old_eval,dwd_val],ignore_index=True,sort=False)
i8_validation['row_id']=i8_validation.row_id.astype(str)
print(i8_validation.groupby(['i8_domain','i8_scope']).size())


Candidate scoring table numeric feature contract: PASS | features: 108
Candidate scoring table numeric feature contract: PASS | features: 108
i8_domain  i8_scope    
dwd        confirmation    46667
           discovery       47121
           tune            45928
india      confirmation    47599
           discovery       75815
           tune            57894
dtype: int64


## 37. Clean-only unsupervised challengers

The supplied research note correctly suggested Isolation Forest and reconstruction autoencoders, but its example fitted on the corrupted evaluation stream. Here both novelty models are fitted only on 2022 rows labelled as non-fault. Genuine regional-weather rows remain in the normal class so the models are not rewarded for calling weather a sensor failure.

The recurrent model is a unidirectional causal LSTM autoencoder. Every score at time *t* uses only a 24-step window ending at *t*. Raw absolute T/P/RH, dew point, calendar variables, labels and future values are excluded; the inputs are robust temporal and same-parameter neighbour residuals derived from the three permitted observations.


In [39]:
from sklearn.ensemble import IsolationForest
from torch import nn
from torch.utils.data import DataLoader,TensorDataset

I8_UNSUP_FEATURE_CANDIDATES=[
    'primary_missing_count','time_since_previous_minutes','gap_ratio',
    'regional_agreement_mean','regional_agreement_min',
    'regional_standardized_disagreement_max','regional_trend_disagreement_mean',
]
for sensor in ['temperature','pressure','humidity']:
    I8_UNSUP_FEATURE_CANDIDATES += [
        f'{sensor}_robust_z_24h',f'{sensor}_ewma_residual',f'{sensor}_frozen_run_length',
        f'neighbor_{sensor}_residual',f'neighbor_{sensor}_agreement_fraction',
        f'{sensor}_slope_6h',f'{sensor}_neighbor_residual_slope_6h',
        f'{sensor}_cusum_positive',f'{sensor}_cusum_negative',
    ]
I8_UNSUP_FEATURES=[feature for feature in I8_UNSUP_FEATURE_CANDIDATES if feature in FEATURES]
assert len(I8_UNSUP_FEATURES)>=30
assert not ({'temperature_value','pressure_value','humidity_value','temperature_dewpoint_spread_c',
             'hour_sin','hour_cos','day_of_year_sin','day_of_year_cos'}&set(I8_UNSUP_FEATURES))

I8_UNSUP_PACKAGE=ITER8_ROOT/'iteration8_isolation_forest.joblib'
i8_clean_fit=i8_fit.loc[i8_fit.is_anomaly.eq(0)].copy()

def i8_balanced_sample(frame,max_per_domain,seed):
    pieces=[]
    for domain,group in frame.groupby('i8_domain',sort=True):
        count=min(len(group),int(max_per_domain))
        pieces.append(group.sample(count,random_state=int(seed)+sum(map(ord,str(domain)))))
    return pd.concat(pieces,ignore_index=True)

if REUSE_SAVED_MODELS and I8_UNSUP_PACKAGE.exists():
    i8_unsup_package=joblib.load(I8_UNSUP_PACKAGE)
    i8_unsup_median=np.asarray(i8_unsup_package['median'],dtype=np.float32)
    i8_unsup_scale=np.asarray(i8_unsup_package['scale'],dtype=np.float32)
    i8_iforest=i8_unsup_package['model']
    i8_if_reference=np.asarray(i8_unsup_package['reference'],dtype=np.float32)
else:
    scaler_rows=i8_balanced_sample(i8_clean_fit,90000,8101)
    scaler_values=scaler_rows[I8_UNSUP_FEATURES].apply(pd.to_numeric,errors='coerce').to_numpy(np.float32)
    i8_unsup_median=np.nanmedian(scaler_values,axis=0).astype(np.float32)
    i8_unsup_median=np.where(np.isfinite(i8_unsup_median),i8_unsup_median,0.0).astype(np.float32)
    q25=np.nanpercentile(scaler_values,25,axis=0); q75=np.nanpercentile(scaler_values,75,axis=0)
    iqr=q75-q25
    i8_unsup_scale=np.where(np.isfinite(iqr)&(iqr>=1e-3),iqr,1.0).astype(np.float32)
    scaler_values=np.where(np.isfinite(scaler_values),scaler_values,i8_unsup_median)
    scaler_values=np.clip((scaler_values-i8_unsup_median)/i8_unsup_scale,-12,12)
    i8_iforest=IsolationForest(
        n_estimators=400,max_samples=min(4096,len(scaler_values)),max_features=.85,
        contamination='auto',bootstrap=False,n_jobs=-1,random_state=8101)
    i8_iforest.fit(scaler_values)
    i8_if_reference=np.sort(-i8_iforest.score_samples(scaler_values)).astype(np.float32)
    i8_unsup_package={'features':I8_UNSUP_FEATURES,'median':i8_unsup_median,'scale':i8_unsup_scale,
                      'model':i8_iforest,'reference':i8_if_reference}
    joblib.dump(i8_unsup_package,I8_UNSUP_PACKAGE,compress=3)

assert list(i8_unsup_package['features'])==I8_UNSUP_FEATURES

def i8_scaled_matrix(frame):
    values=frame[I8_UNSUP_FEATURES].apply(pd.to_numeric,errors='coerce').to_numpy(np.float32)
    values=np.where(np.isfinite(values),values,i8_unsup_median)
    return np.clip((values-i8_unsup_median)/i8_unsup_scale,-12,12).astype(np.float32)

def i8_tail_probability(raw,reference):
    reference=np.asarray(reference,dtype=np.float32)
    return np.searchsorted(reference,np.asarray(raw,dtype=np.float32),side='right')/max(len(reference),1)

def i8_isolation_scores(frame):
    raw=-i8_iforest.score_samples(i8_scaled_matrix(frame))
    return np.clip(i8_tail_probability(raw,i8_if_reference),0,1).astype(np.float32)

print({'unsupervised_features':len(I8_UNSUP_FEATURES),'clean_fit_rows':len(i8_clean_fit),
       'iforest_reference_rows':len(i8_if_reference)})


{'unsupervised_features': 34, 'clean_fit_rows': 281724, 'iforest_reference_rows': 180000}


In [40]:
I8_SEQUENCE_WINDOW=24
I8_LSTM_PATH=ITER8_ROOT/'iteration8_causal_lstm_autoencoder.pt'

class I8CausalLSTMAutoencoder(nn.Module):
    def __init__(self,input_dim,hidden_dim=24):
        super().__init__()
        self.encoder=nn.LSTM(input_dim,hidden_dim,num_layers=1,batch_first=True,bidirectional=False)
        self.decoder=nn.Sequential(nn.Linear(hidden_dim,16),nn.GELU(),nn.Linear(16,input_dim))
    def forward(self,x):
        encoded,_=self.encoder(x)
        return self.decoder(encoded)

def i8_clean_sequences(frame,max_per_domain,seed):
    rng=np.random.default_rng(seed); domain_arrays=[]
    for domain,domain_frame in frame.groupby('i8_domain',sort=True):
        station_pieces=[]; station_groups=list(domain_frame.groupby('station_id',sort=True))
        per_station=max(1,int(np.ceil(max_per_domain/max(len(station_groups),1))))
        for _,group in station_groups:
            group=group.sort_values('emitted_timestamp_utc',kind='stable')
            values=i8_scaled_matrix(group)
            clean=group.is_anomaly.eq(0).to_numpy(bool)
            times=pd.to_datetime(group.emitted_timestamp_utc,utc=True)
            delta=times.diff().dt.total_seconds().div(60).to_numpy(float)
            bad_step=np.zeros(len(group),dtype=np.int32)
            bad_step[1:]=((delta[1:]<=0)|(delta[1:]>90)|~np.isfinite(delta[1:])).astype(np.int32)
            if len(group)<I8_SEQUENCE_WINDOW: continue
            clean_count=np.convolve(clean.astype(np.int32),np.ones(I8_SEQUENCE_WINDOW,dtype=np.int32),'valid')
            prefix=np.cumsum(bad_step); starts=np.arange(len(group)-I8_SEQUENCE_WINDOW+1)
            ends=starts+I8_SEQUENCE_WINDOW-1
            contiguous=(prefix[ends]-prefix[starts])==0
            eligible=starts[(clean_count==I8_SEQUENCE_WINDOW)&contiguous]
            if len(eligible)>per_station: eligible=np.sort(rng.choice(eligible,per_station,replace=False))
            if len(eligible): station_pieces.append(np.stack([values[s:s+I8_SEQUENCE_WINDOW] for s in eligible]))
        if station_pieces:
            domain_values=np.concatenate(station_pieces).astype(np.float32)
            if len(domain_values)>max_per_domain:
                domain_values=domain_values[rng.choice(len(domain_values),max_per_domain,replace=False)]
            domain_arrays.append(domain_values)
    if not domain_arrays: raise RuntimeError('No clean causal sequences were generated.')
    return np.concatenate(domain_arrays).astype(np.float32)

def i8_sequence_errors(model,sequences,batch_size=1024):
    loader=DataLoader(TensorDataset(torch.from_numpy(sequences)),batch_size=batch_size,shuffle=False,num_workers=0)
    values=[]; model.eval()
    with torch.no_grad():
        for (batch,) in loader:
            batch=batch.to(DEVICE)
            reconstruction=model(batch)
            values.append(torch.mean(torch.abs(reconstruction[:,-1]-batch[:,-1]),dim=1).cpu().numpy())
    return np.concatenate(values).astype(np.float32)

torch.manual_seed(8107); torch.cuda.manual_seed_all(8107)
i8_lstm=I8CausalLSTMAutoencoder(len(I8_UNSUP_FEATURES)).to(DEVICE)
i8_lstm_checkpoint=None
if REUSE_SAVED_MODELS and I8_LSTM_PATH.exists():
    i8_lstm_checkpoint=torch.load(I8_LSTM_PATH,map_location=DEVICE,weights_only=False)
    assert list(i8_lstm_checkpoint['features'])==I8_UNSUP_FEATURES
    assert int(i8_lstm_checkpoint['window'])==I8_SEQUENCE_WINDOW
    i8_lstm.load_state_dict(i8_lstm_checkpoint['state_dict'])
    i8_lstm_reference=np.asarray(i8_lstm_checkpoint['reference_errors'],dtype=np.float32)
    i8_lstm_history=i8_lstm_checkpoint['history']
    i8_lstm_train_sequences=int(i8_lstm_checkpoint['train_sequences'])
    i8_lstm_tune_sequences=int(i8_lstm_checkpoint['tune_sequences'])
else:
    i8_train_sequences=i8_clean_sequences(i8_fit,36000,8107)
    i8_tune_sequences=i8_clean_sequences(i8_tune,9000,8111)
    train_loader=DataLoader(TensorDataset(torch.from_numpy(i8_train_sequences)),batch_size=512,
                            shuffle=True,num_workers=0,pin_memory=(DEVICE=='cuda'))
    tune_tensor=torch.from_numpy(i8_tune_sequences).to(DEVICE)
    optimizer=torch.optim.AdamW(i8_lstm.parameters(),lr=8e-4,weight_decay=2e-4)
    loss_fn=nn.SmoothL1Loss(beta=.5)
    best_loss=float('inf'); best_state=None; patience=0; i8_lstm_history=[]
    for epoch in range(1,21):
        i8_lstm.train(); train_losses=[]
        for (batch,) in train_loader:
            batch=batch.to(DEVICE,non_blocking=True); optimizer.zero_grad(set_to_none=True)
            noisy=batch+.025*torch.randn_like(batch)
            reconstruction=i8_lstm(noisy)
            loss=.30*loss_fn(reconstruction,batch)+.70*loss_fn(reconstruction[:,-1],batch[:,-1])
            loss.backward(); torch.nn.utils.clip_grad_norm_(i8_lstm.parameters(),2.0); optimizer.step()
            train_losses.append(float(loss.detach().cpu()))
        i8_lstm.eval()
        with torch.no_grad():
            tune_reconstruction=i8_lstm(tune_tensor)
            tune_loss=float((.30*loss_fn(tune_reconstruction,tune_tensor)+
                             .70*loss_fn(tune_reconstruction[:,-1],tune_tensor[:,-1])).cpu())
        i8_lstm_history.append({'epoch':epoch,'train_loss':float(np.mean(train_losses)),'normal_tune_loss':tune_loss})
        print('LSTM-AE',i8_lstm_history[-1])
        if tune_loss<best_loss-1e-5:
            best_loss=tune_loss; patience=0
            best_state={key:value.detach().cpu().clone() for key,value in i8_lstm.state_dict().items()}
        else:
            patience+=1
            if patience>=4: break
    i8_lstm.load_state_dict(best_state); i8_lstm=i8_lstm.to(DEVICE)
    i8_lstm_reference=np.sort(i8_sequence_errors(i8_lstm,i8_train_sequences)).astype(np.float32)
    i8_lstm_train_sequences=len(i8_train_sequences); i8_lstm_tune_sequences=len(i8_tune_sequences)
    torch.save({'state_dict':i8_lstm.state_dict(),'reference_errors':i8_lstm_reference,
                'history':i8_lstm_history,'features':I8_UNSUP_FEATURES,'window':I8_SEQUENCE_WINDOW,
                'train_sequences':i8_lstm_train_sequences,'tune_sequences':i8_lstm_tune_sequences},I8_LSTM_PATH)
    del i8_train_sequences,i8_tune_sequences,tune_tensor



In [41]:
def i8_lstm_scores(frame):
    result=pd.Series(0.0,index=frame.index,dtype=float)
    for _,group in frame.groupby('station_id',sort=False):
        group=group.sort_values('emitted_timestamp_utc',kind='stable')
        values=i8_scaled_matrix(group); times=pd.to_datetime(group.emitted_timestamp_utc,utc=True)
        delta=times.diff().dt.total_seconds().div(60).to_numpy(float)
        breaks=np.flatnonzero((delta<=0)|(delta>90)|~np.isfinite(delta))
        starts=np.r_[0,breaks]; ends=np.r_[breaks,len(group)]
        for start,end in zip(starts,ends):
            if end-start<I8_SEQUENCE_WINDOW: continue
            segment=values[start:end]
            sequences=np.stack([segment[pos-I8_SEQUENCE_WINDOW+1:pos+1]
                                for pos in range(I8_SEQUENCE_WINDOW-1,len(segment))]).astype(np.float32)
            errors=i8_sequence_errors(i8_lstm,sequences)
            scores=np.clip(i8_tail_probability(errors,i8_lstm_reference),0,1)
            target_indices=group.index.to_numpy()[start+I8_SEQUENCE_WINDOW-1:end]
            result.loc[target_indices]=scores
    return result.to_numpy(np.float32)

i8_validation['i8_if_score']=i8_isolation_scores(i8_validation)
i8_validation['i8_lstm_ae_score']=i8_lstm_scores(i8_validation)
i8_validation['i8_tree_if_score']=np.maximum(
    i8_validation.i8_fault_score,np.sqrt(i8_validation.i8_fault_score*i8_validation.i8_if_score))
i8_validation['i8_tree_lstm_score']=np.maximum(
    i8_validation.i8_fault_score,np.sqrt(i8_validation.i8_fault_score*i8_validation.i8_lstm_ae_score))
i8_validation['i8_three_model_score']=np.maximum(
    i8_validation.i8_fault_score,
    np.cbrt(i8_validation.i8_fault_score*i8_validation.i8_if_score*i8_validation.i8_lstm_ae_score))
I8_SCORE_VARIANTS={
    'tree_only':'i8_fault_score','tree_plus_isolation_forest':'i8_tree_if_score',
    'tree_plus_lstm_autoencoder':'i8_tree_lstm_score','tree_if_lstm_consensus':'i8_three_model_score'}

i8_unsupervised_history=pd.DataFrame([
    {'model':'IsolationForest','fit_rows':len(i8_if_reference),'features':len(I8_UNSUP_FEATURES),
     'sequence_window':1,'epochs':0},
    {'model':'causal_LSTM_autoencoder','fit_rows':i8_lstm_train_sequences,'features':len(I8_UNSUP_FEATURES),
     'sequence_window':I8_SEQUENCE_WINDOW,'epochs':len(i8_lstm_history)},
])
i8_unsupervised_history.to_csv(ITER8_ROOT/'iteration8_unsupervised_training_history.csv',index=False)
display(i8_unsupervised_history)
display(i8_validation.groupby(['i8_domain','i8_scope'])[['i8_if_score','i8_lstm_ae_score']].quantile(.99))


,model,fit_rows,features,sequence_window,epochs
0,IsolationForest,180000,34,1,0
1,causal_LSTM_autoencoder,54015,34,24,20


i8_if_score  i8_lstm_ae_score
i8_domain i8_scope                                   
dwd       confirmation      0.98521           0.99126
          discovery         0.97668           0.97621
          tune              0.98747           0.99056
india     confirmation      0.99377           0.99645
          discovery         0.99331           0.99574
          tune              0.99481           0.97645

## 38. Tune once, then freeze thresholds

Weather threshold is selected first on the tune scope with a strict fault-to-weather cap. The fault threshold is then selected with incident false-alarm and precision constraints. Discovery and confirmation are never searched.


In [42]:
from sklearn.metrics import average_precision_score

def i8_weather_prediction(frame,threshold):
    agreement=pd.to_numeric(frame.regional_agreement_mean,errors='coerce').fillna(0)
    return frame.i8_weather_score.ge(float(threshold))&agreement.ge(.50)

def i8_weather_metrics(frame,pred):
    y=frame.is_weather_event.astype(int)
    return {
        'precision':float(precision_score(y,pred,zero_division=0)),
        'recall':float(recall_score(y,pred,zero_division=0)),
        'f1':float(f1_score(y,pred,zero_division=0)),
        'auprc':float(average_precision_score(y,frame.i8_weather_score)) if y.sum()>0 else float('nan'),
        'weather_event_rows':int(y.sum()),
        'weather_event_episodes':int(frame.loc[y.eq(1),'episode_id'].replace('',np.nan).nunique()),
        'fault_to_weather_rate':float(pred.loc[frame.is_anomaly.eq(1)].mean()) if frame.is_anomaly.eq(1).any() else 0.0,
    }

i8_weather_frontier=[]
for threshold in np.linspace(.03,.95,47):
    rows=[]
    for domain in ['india','dwd']:
        part=i8_validation.loc[i8_validation.i8_scope.eq('tune')&i8_validation.i8_domain.eq(domain)].copy()
        rows.append({'domain':domain,**i8_weather_metrics(part,i8_weather_prediction(part,threshold))})
    i8_weather_frontier.append({
        'threshold':float(threshold),'min_precision':min(row['precision'] for row in rows),
        'min_f1':min(row['f1'] for row in rows),'mean_f1':float(np.mean([row['f1'] for row in rows])),
        'max_fault_to_weather':max(row['fault_to_weather_rate'] for row in rows)})
i8_weather_frontier=pd.DataFrame(i8_weather_frontier)
i8_weather_feasible=i8_weather_frontier.loc[(i8_weather_frontier.min_precision>=.75)&(i8_weather_frontier.max_fault_to_weather<=.01)]
if len(i8_weather_feasible):
    i8_weather_selected=i8_weather_feasible.sort_values(['min_f1','mean_f1'],ascending=False).iloc[0]
    I8_WEATHER_TUNE_STATUS='constraints_met'
else:
    i8_weather_frontier['violation']=np.maximum(0,.75-i8_weather_frontier.min_precision)+10*np.maximum(0,i8_weather_frontier.max_fault_to_weather-.01)
    i8_weather_selected=i8_weather_frontier.sort_values(['violation','min_f1','mean_f1'],ascending=[True,False,False]).iloc[0]
    I8_WEATHER_TUNE_STATUS='pareto_fallback_not_promotable'
I8_WEATHER_THRESHOLD=float(i8_weather_selected.threshold)
i8_weather_frontier.to_csv(ITER8_ROOT/'iteration8_weather_policy_frontier.csv',index=False)
display(i8_weather_selected.to_frame('selected_weather'))


,selected_weather
threshold,0.29000
min_precision,0.91667
min_f1,0.60440
mean_f1,0.75885
max_fault_to_weather,0.00866


In [43]:
def i8_candidate_fault_prediction(frame,score_col,threshold):
    raw=hysteresis(frame,score_col,float(threshold),max(0,float(threshold)-.07))
    weather=i8_weather_prediction(frame,I8_WEATHER_THRESHOLD)
    override=frame[score_col].ge(min(.99,float(threshold)+.18))
    return (raw&(~weather|override))|frame.hard_rule.astype(bool)

# Recompute deterministic hard-rule flags consistently on the combined validation table.
i8_validation['hard_rule']=add_hard_rules(i8_validation).hard_rule.to_numpy(bool)
i8_fault_frontier=[]
for variant,score_col in I8_SCORE_VARIANTS.items():
  for threshold in np.linspace(.08,.95,45):
      rows=[]
      for domain in ['india','dwd']:
          part=i8_validation.loc[i8_validation.i8_scope.eq('tune')&i8_validation.i8_domain.eq(domain)].copy()
          part['candidate']=i8_candidate_fault_prediction(part,score_col,threshold)
          metric=evaluate(part,score_col,'candidate')
          rows.append({'domain':domain,**metric})
      i8_fault_frontier.append({
          'variant':variant,'score_col':score_col,'threshold':float(threshold),
          'min_precision':min(row['precision'] for row in rows),
          'max_false_alarm':max(row['false_alarm_episodes_per_station_day'] for row in rows),
          'min_point_f1':min(row['f1'] for row in rows),'mean_point_f1':float(np.mean([row['f1'] for row in rows])),
          'min_event_f1':min(row['event_f1'] for row in rows),'mean_event_f1':float(np.mean([row['event_f1'] for row in rows])),
          'min_event_recall':min(row['event_recall'] for row in rows)})
i8_fault_frontier=pd.DataFrame(i8_fault_frontier)
i8_fault_feasible=i8_fault_frontier.loc[(i8_fault_frontier.min_precision>=.80)&(i8_fault_frontier.max_false_alarm<=.02)]
if len(i8_fault_feasible):
    i8_fault_selected=i8_fault_feasible.sort_values(['min_event_recall','min_event_f1','mean_point_f1'],ascending=False).iloc[0]
    I8_FAULT_TUNE_STATUS='constraints_met'
else:
    i8_fault_frontier['violation']=np.maximum(0,.80-i8_fault_frontier.min_precision)+10*np.maximum(0,i8_fault_frontier.max_false_alarm-.02)
    i8_fault_selected=i8_fault_frontier.sort_values(['violation','min_event_recall','mean_event_f1'],ascending=[True,False,False]).iloc[0]
    I8_FAULT_TUNE_STATUS='pareto_fallback_not_promotable'
I8_FAULT_THRESHOLD=float(i8_fault_selected.threshold)
I8_FAULT_VARIANT=str(i8_fault_selected.variant)
I8_FAULT_SCORE_COL=str(i8_fault_selected.score_col)
i8_fault_frontier.to_csv(ITER8_ROOT/'iteration8_fault_policy_frontier.csv',index=False)
display(i8_fault_selected.to_frame('selected_fault'))


,selected_fault
variant,tree_plus_lstm_autoencoder
score_col,i8_tree_lstm_score
threshold,0.75227
min_precision,0.80208
max_false_alarm,0.01042
min_point_f1,0.27599
mean_point_f1,0.35830
min_event_f1,0.33333
mean_event_f1,0.51333
min_event_recall,0.40000


## 39. Discovery and confirmation comparison

The frozen candidate is compared against the Iteration 5 development reference in India and its unadapted transfer baseline in DWD. Results are reported separately by domain, scope, climate and fault family.


In [44]:
def i8_weak_mean(metric):
    values=[metric['per_fault_episode_recall'].get(name,np.nan) for name in ['bias','drift','frozen_sensor']]
    values=[value for value in values if not pd.isna(value)]
    return float(np.mean(values)) if values else float('nan')

i8_comparison=[]; i8_fault_rows=[]; i8_weather_rows=[]
for scope in ['discovery','confirmation']:
  domain_masks={
      'india':i8_validation.i8_domain.eq('india'),
      'dwd_all':i8_validation.i8_domain.eq('dwd'),
      'dwd_holdout':i8_validation.i8_domain.eq('dwd')&i8_validation.station_id.isin(I8_DWD_HOLDOUTS),
  }
  for domain,domain_mask in domain_masks.items():
    part=i8_validation.loc[i8_validation.i8_scope.eq(scope)&domain_mask].copy()
    assert len(part)>0,f'Empty Iteration 8 evaluation slice: {scope}/{domain}'
    part['baseline']=part.i8_baseline_pred.astype(bool)
    part['candidate']=i8_candidate_fault_prediction(part,I8_FAULT_SCORE_COL,I8_FAULT_THRESHOLD)
    baseline=evaluate(part,'base_score','baseline')
    candidate=evaluate(part,I8_FAULT_SCORE_COL,'candidate')
    candidate_weather=i8_weather_metrics(part,i8_weather_prediction(part,I8_WEATHER_THRESHOLD))
    baseline_weather=i8_weather_metrics(part,part.i8_baseline_weather.astype(bool))
    i8_comparison.append({
        'scope':scope,'domain':domain,'rows':len(part),'candidate_variant':I8_FAULT_VARIANT,
        'baseline_precision':baseline['precision'],'candidate_precision':candidate['precision'],
        'baseline_recall':baseline['recall'],'candidate_recall':candidate['recall'],
        'baseline_point_f1':baseline['f1'],'candidate_point_f1':candidate['f1'],
        'point_f1_delta':candidate['f1']-baseline['f1'],
        'baseline_event_f1':baseline['event_f1'],'candidate_event_f1':candidate['event_f1'],
        'event_f1_delta':candidate['event_f1']-baseline['event_f1'],
        'baseline_weak_recall':i8_weak_mean(baseline),'candidate_weak_recall':i8_weak_mean(candidate),
        'weak_recall_delta':i8_weak_mean(candidate)-i8_weak_mean(baseline),
        'candidate_false_alarm':candidate['false_alarm_episodes_per_station_day'],
        'baseline_weather_f1':baseline_weather['f1'],'candidate_weather_f1':candidate_weather['f1'],
        'candidate_fault_to_weather':candidate_weather['fault_to_weather_rate']})
    for family,group in part.loc[part.is_anomaly.eq(1)&part.episode_id.ne('')].groupby('anomaly_type'):
        ids=group.episode_id.unique(); base_hits=[]; candidate_hits=[]
        for episode_id in ids:
            event=part.episode_id.eq(episode_id)
            base_hits.append(bool(part.loc[event,'baseline'].any()))
            candidate_hits.append(bool(part.loc[event,'candidate'].any()))
        i8_fault_rows.append({'scope':scope,'domain':domain,'anomaly_type':family,'episodes':len(ids),
                              'baseline_episode_recall':float(np.mean(base_hits)),
                              'candidate_episode_recall':float(np.mean(candidate_hits))})
    for cluster,group in part.groupby('cluster'):
        result=i8_weather_metrics(group,i8_weather_prediction(group,I8_WEATHER_THRESHOLD))
        i8_weather_rows.append({'scope':scope,'domain':domain,'cluster':cluster,**result})

i8_comparison=pd.DataFrame(i8_comparison)
i8_fault_recall=pd.DataFrame(i8_fault_rows)
i8_weather_by_cluster=pd.DataFrame(i8_weather_rows)
i8_comparison.to_csv(ITER8_ROOT/'iteration8_multidomain_confirmation.csv',index=False)
i8_fault_recall.to_csv(ITER8_ROOT/'iteration8_fault_episode_recall.csv',index=False)
i8_weather_by_cluster.to_csv(ITER8_ROOT/'iteration8_weather_by_cluster.csv',index=False)
display(i8_comparison); display(i8_fault_recall); display(i8_weather_by_cluster)


,scope,domain,rows,candidate_variant,baseline_precision,candidate_precision,baseline_recall,candidate_recall,baseline_point_f1,candidate_point_f1,point_f1_delta,baseline_event_f1,candidate_event_f1,event_f1_delta,baseline_weak_recall,candidate_weak_recall,weak_recall_delta,candidate_false_alarm,baseline_weather_f1,candidate_weather_f1,candidate_fault_to_weather
0,discovery,india,75815,tree_plus_lstm_autoencoder,0.83410,0.86957,0.29288,0.25890,0.43353,0.39900,-0.03453,0.61947,0.64000,0.02053,0.33333,0.16667,-0.16667,0.00622,0.62000,0.78195,0.00000
1,discovery,dwd_all,47121,tree_plus_lstm_autoencoder,0.09036,0.80000,0.06711,0.20582,0.07702,0.32740,0.25038,0.09756,0.37037,0.27281,0.33333,0.41667,0.08333,0.01220,0.10327,0.85714,0.00895
2,discovery,dwd_holdout,11808,tree_plus_lstm_autoencoder,0.19318,0.87500,0.10241,0.25301,0.13386,0.39252,0.25867,0.06897,0.35294,0.28398,0.16667,0.33333,0.16667,0.01830,0.10337,0.86359,0.01205
3,confirmation,india,47599,tree_plus_lstm_autoencoder,0.83262,0.84211,0.41189,0.40764,0.55114,0.54936,-0.00178,0.59794,0.65263,0.05469,0.33333,0.55556,0.22222,0.01524,0.43697,0.34000,0.00637
4,confirmation,dwd_all,46667,tree_plus_lstm_autoencoder,0.03776,0.66667,0.05112,0.13497,0.04344,0.22449,0.18105,0.06867,0.30508,0.23642,0.33333,0.41667,0.08333,0.01537,0.07397,0.90836,0.01227
5,confirmation,dwd_holdout,11617,tree_plus_lstm_autoencoder,0.03587,0.76562,0.05229,0.32026,0.04255,0.45161,0.40906,0.07895,0.42105,0.34211,0.44444,0.55556,0.11111,0.01845,0.07063,0.90385,0.00654


,scope,domain,anomaly_type,episodes,baseline_episode_recall,candidate_episode_recall
0,discovery,india,bias,3,0.33333,0.00000
1,discovery,india,communication_corruption,2,1.00000,1.00000
2,discovery,india,drift,6,0.33333,0.16667
3,discovery,india,duplicate_packet,5,0.00000,0.00000
4,discovery,india,frozen_sensor,3,0.33333,0.33333
5,discovery,india,multi_sensor_failure,3,1.00000,1.00000
6,discovery,india,noise,4,1.00000,0.75000
7,discovery,india,scaling_error,5,1.00000,1.00000
8,discovery,india,spike,8,1.00000,1.00000
9,discovery,india,sudden_drop,3,0.66667,1.00000


,scope,domain,cluster,precision,recall,f1,auprc,weather_event_rows,weather_event_episodes,fault_to_weather_rate
0,discovery,india,bengaluru,0.00000,0.00000,0.00000,NaN,0,0,0.00000
1,discovery,india,chennai,1.00000,0.42222,0.59375,0.73910,90,1,0.00000
2,discovery,india,delhi,0.97059,0.94286,0.95652,0.98998,70,1,0.00000
3,discovery,india,hyderabad,0.00000,0.00000,0.00000,NaN,0,0,0.00000
4,discovery,dwd_all,dwd_east_continental,0.88154,0.85561,0.86839,0.94198,374,6,0.01613
5,discovery,dwd_all,dwd_north_coastal,0.87898,0.81176,0.84404,0.93719,510,6,0.00000
6,discovery,dwd_all,dwd_south_upland,0.81409,0.81250,0.81329,0.92081,512,6,0.02020
7,discovery,dwd_all,dwd_west_lowland,0.89800,0.91633,0.90707,0.96942,490,6,0.00000
8,discovery,dwd_holdout,dwd_east_continental,0.92045,0.88043,0.90000,0.96830,92,6,0.02020
9,discovery,dwd_holdout,dwd_north_coastal,0.93636,0.80469,0.86555,0.94317,128,6,0.00000


## 40. Root-cause model and explainability artifact

Root-cause fitting uses only labelled training fault rows. Feature importance is exported from the LightGBM challenger so every alert can expose the strongest causal signals; no label appears in inference features.


In [45]:
i8_root_train=i8_fit.loc[i8_fit.is_anomaly.eq(1)].copy()
i8_root_model=CatBoostClassifier(
    iterations=900,depth=7,learning_rate=.04,loss_function='MultiClass',eval_metric='TotalF1',
    l2_leaf_reg=10,random_seed=81,task_type='GPU',devices='0',verbose=150,
    allow_writing_files=False)
i8_root_path=ITER8_ROOT/'iteration8_root_cause_catboost.cbm'
if REUSE_SAVED_MODELS and i8_root_path.exists(): i8_root_model.load_model(i8_root_path)
else:
    i8_root_model.fit(i8_root_train[FEATURES],i8_root_train.anomaly_type.astype(str))
    i8_root_model.save_model(i8_root_path)

i8_confirmation=i8_validation.loc[i8_validation.i8_scope.eq('confirmation')].copy()
i8_confirmation['candidate']=i8_candidate_fault_prediction(
    i8_confirmation,I8_FAULT_SCORE_COL,I8_FAULT_THRESHOLD)
i8_root_eval=i8_confirmation.loc[i8_confirmation.is_anomaly.eq(1)&i8_confirmation.candidate].copy()
if len(i8_root_eval):
    root_prediction=np.asarray(i8_root_model.predict(i8_root_eval[FEATURES])).reshape(-1).astype(str)
    I8_ROOT_ACCURACY=float(np.mean(root_prediction==i8_root_eval.anomaly_type.astype(str).to_numpy()))
else: I8_ROOT_ACCURACY=float('nan')

importance=np.mean([model.feature_importances_ for model in i8_models['fault']['lgb']],axis=0)
i8_importance=(pd.DataFrame({'feature':FEATURES,'importance':importance})
               .sort_values('importance',ascending=False).reset_index(drop=True))
i8_importance.to_csv(ITER8_ROOT/'iteration8_feature_importance.csv',index=False)
display(i8_importance.head(25)); print('Detected-fault root-cause accuracy:',I8_ROOT_ACCURACY)


,feature,importance
0,temperature_rolling_median_24h,"1,246.50000"
1,humidity_rolling_median_24h,"1,163.50000"
2,nearest_neighbor_km,"1,017.50000"
3,pressure_rolling_median_24h,936.50000
4,neighbor_pressure_residual,818.50000
5,pressure_climatology_residual,711.00000
6,pressure_neighbor_residual_slope_24h,627.50000
7,humidity_rolling_mad_24h,619.00000
8,temperature_climatology_residual,603.00000
9,temperature_rolling_mad_24h,587.50000


Detected-fault root-cause accuracy: 0.7403100775193798


## 41. Promotion decision and integrity receipt

Failure of any gate produces a no-op decision. Passing every gate makes the challenger eligible for one locked DWD 2024 confirmation; it does not replace the deployed Phase 10 model by itself.


In [46]:
confirmation_rows=i8_comparison.loc[i8_comparison.scope.eq('confirmation')]
discovery_rows=i8_comparison.loc[i8_comparison.scope.eq('discovery')]
supported_confirmation_weather=i8_weather_by_cluster.loc[
    i8_weather_by_cluster.scope.eq('confirmation')&i8_weather_by_cluster.weather_event_rows.gt(0)]
assert len(supported_confirmation_weather)>0
i8_worst_confirmation_weather=float(supported_confirmation_weather.f1.min())
I8_PROMOTION_GATES={
    'tune_fault_constraints':I8_FAULT_TUNE_STATUS=='constraints_met',
    'tune_weather_constraints':I8_WEATHER_TUNE_STATUS=='constraints_met',
    'discovery_precision':bool(discovery_rows.candidate_precision.ge(.80).all()),
    'confirmation_precision':bool(confirmation_rows.candidate_precision.ge(.80).all()),
    'discovery_false_alarm':bool(discovery_rows.candidate_false_alarm.le(.02).all()),
    'confirmation_false_alarm':bool(confirmation_rows.candidate_false_alarm.le(.02).all()),
    'no_discovery_point_f1_regression':bool(discovery_rows.point_f1_delta.ge(0).all()),
    'no_confirmation_point_f1_regression':bool(confirmation_rows.point_f1_delta.ge(0).all()),
    'no_confirmation_event_f1_regression':bool(confirmation_rows.event_f1_delta.ge(0).all()),
    'positive_confirmation_weak_recall':bool(confirmation_rows.weak_recall_delta.ge(0).all() and confirmation_rows.weak_recall_delta.mean()>0),
    'confirmation_weather_f1_at_least_075':bool(confirmation_rows.candidate_weather_f1.ge(.75).all()),
    'worst_climate_weather_f1_at_least_065':bool(i8_worst_confirmation_weather>=.65),
    'dwd_holdout_confirmation_present':bool((confirmation_rows.domain=='dwd_holdout').any()),
    'fault_to_weather_at_most_001':bool(confirmation_rows.candidate_fault_to_weather.le(.01).all()),
}
I8_PROMOTED=all(I8_PROMOTION_GATES.values())
I8_STATUS='eligible_for_locked_dwd_2024_confirmation' if I8_PROMOTED else 'not_eligible_keep_iteration5_development_reference'

i8_feature_contract={
    'observation_inputs':['temperature','pressure','relative_humidity'],
    'model_features':FEATURES,'model_feature_count':len(FEATURES),
    'unsupervised_features':I8_UNSUP_FEATURES,'unsupervised_feature_count':len(I8_UNSUP_FEATURES),
    'unsupervised_window':I8_SEQUENCE_WINDOW,'unsupervised_fit':'2022 non-fault rows only',
    'new_source':'DWD CDC official 10-minute station observations',
    'dwd_training_year':2022,'dwd_validation_year':2023,
    'dwd_2024_opened':False,'any_2025_opened':False,
    'weather_families':list(WEATHER_FAMILIES),'fault_families':list(FAULT_FAMILIES),
    'causality':'current and previously emitted station/neighbor observations only',
    'forbidden':['dew_point','future_observation','2024_labels','2025_observations_or_labels'],
}
(ITER8_ROOT/'iteration8_feature_contract.json').write_text(json.dumps(i8_feature_contract,indent=2))

i8_receipt={
    'development_bundle_sha256':ITER8_EXPECTED_SHA,'original_bundle_sha256':OLD_EXPECTED_SHA,
    'source_validation_status':i8_source_report['status'],
    'dwd_2024_opened':False,'noaa_2024_opened':False,'any_2025_opened':False,
    'train_curriculum':i8_train_audit,'validation_curriculum':i8_val_audit,
    'pseudo_unseen_stations':I8_DWD_HOLDOUTS,
    'future_features_used':False,'dew_point_used':False,
    'isolation_forest_fit_on_evaluation':False,'lstm_bidirectional':False,
}
(ITER8_ROOT/'iteration8_data_curriculum_receipt.json').write_text(json.dumps(i8_receipt,indent=2))

result8={
    'iteration':'08_multiclimate_data_curriculum','status':I8_STATUS,'promoted':I8_PROMOTED,
    'device':DEVICE,'gpu':torch.cuda.get_device_name(0),
    'new_data':{'provider':'DWD','stations':16,'clusters':4,'cadence_minutes':10,
                'development_years':[2022,2023],'locked_2024_opened':False,'any_2025_opened':False},
    'curriculum':{'training':i8_train_audit,'validation':i8_val_audit},
    'models':['CatBoost fault','LightGBM fault seeds 17/41','Isolation Forest clean-only novelty',
              'causal LSTM reconstruction autoencoder','CatBoost weather','LightGBM weather seeds 17/41','CatBoost root cause'],
    'selected_fault_variant':I8_FAULT_VARIANT,
    'thresholds':{'fault':I8_FAULT_THRESHOLD,'fault_score_col':I8_FAULT_SCORE_COL,'weather':I8_WEATHER_THRESHOLD},
    'tune_status':{'fault':I8_FAULT_TUNE_STATUS,'weather':I8_WEATHER_TUNE_STATUS},
    'promotion_gates':I8_PROMOTION_GATES,
    'root_cause_accuracy_on_detected_confirmation_fault_rows':I8_ROOT_ACCURACY,
    'multidomain_confirmation':i8_comparison.to_dict('records'),
    'worst_confirmation_climate_weather_f1':i8_worst_confirmation_weather,
    'next_decision':'run one locked DWD 2024 confirmation' if I8_PROMOTED else 'retain Iteration 5 development reference',
}
(ITER8_ROOT/'iteration8_result_block.json').write_text(json.dumps(result8,indent=2,default=float))

print(json.dumps(result8,indent=2,default=float))
print('\nHISTORICAL ITERATION 8 CHECKPOINT - continue through Iteration 9:')
for filename in [
    'iteration8_result_block.json','iteration8_data_curriculum_receipt.json',
    'iteration8_feature_contract.json','iteration8_training_history.csv',
    'iteration8_unsupervised_training_history.csv',
    'iteration8_multidomain_confirmation.csv','iteration8_fault_episode_recall.csv',
    'iteration8_weather_by_cluster.csv','iteration8_feature_importance.csv',
    'iteration8_weather_policy_frontier.csv','iteration8_fault_policy_frontier.csv',
]: print(ITER8_ROOT/filename)


{
  "iteration": "08_multiclimate_data_curriculum",
  "status": "not_eligible_keep_iteration5_development_reference",
  "promoted": false,
  "device": "cuda",
  "gpu": "Tesla T4",
  "new_data": {
    "provider": "DWD",
    "stations": 16,
    "clusters": 4,
    "cadence_minutes": 10,
    "development_years": [
      2022,
      2023
    ],
    "locked_2024_opened": false,
    "any_2025_opened": false
  },
  "curriculum": {
    "training": {
      "status": "PASS",
      "rows": 139684,
      "event_rows": 11498,
      "event_row_fraction": 0.0823143667134389,
      "weather_events": 96,
      "fault_events": 160,
      "weather_families": [
        "dry_intrusion",
        "humidity_surge",
        "multivariate_storm",
        "pressure_front",
        "regional_cold_surge",
        "regional_heatwave"
      ],
      "fault_families": [
        "bias",
        "drift",
        "frozen_sensor",
        "noise",
        "spike"
      ],
      "errors": []
    },
    "validation": {
    

## Historical Iteration 8 checkpoint - continue to Iteration 9

Return the eleven files printed above. Do not open the locked DWD 2024 confirmation bundle or any 2025 benchmark. We will audit whether new-data gains transfer simultaneously to India, DWD climates and pseudo-unseen stations. Only a fully passing candidate can proceed to one final locked confirmation.


# Iteration 9 controlled development phase

The accepted Iteration 5 deployment and the non-promoted Iteration 8 challenger remain immutable. This section uses only the already-approved 2022 training observations and 2023 development observations.


In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.preprocessing import StandardScaler

ITER9_ROOT=DRIVE_ROOT/'experiments'/'iteration_09_domain_invariant_calibration'
ITER9_ROOT.mkdir(parents=True,exist_ok=True)
I9_BASE_RESULT_PATH=ITER8_ROOT/'iteration8_result_block.json'
assert I9_BASE_RESULT_PATH.exists(),'Run the Iteration 8 cells first or restore its Drive checkpoint.'
i9_base_result=json.loads(I9_BASE_RESULT_PATH.read_text())
assert i9_base_result['new_data']['locked_2024_opened'] is False
assert i9_base_result['new_data']['any_2025_opened'] is False
assert UNLOCK_FINAL_TESTS is False

RUN_ITER9_STRESS=True
I9_BASE_STRESS_SEED=8023
I9_NEW_STRESS_SEEDS=[9029,9049]
I9_ALL_STRESS_SEEDS=[I9_BASE_STRESS_SEED]+I9_NEW_STRESS_SEEDS
I9_MIN_PRECISION=.80
I9_MAX_FALSE_ALARM=.02
I9_MAX_FAULT_TO_WEATHER=.01

print({
    'iteration8_status':i9_base_result['status'],
    'iteration8_passed_gates':sum(bool(value) for value in i9_base_result['promotion_gates'].values()),
    'iteration8_total_gates':len(i9_base_result['promotion_gates']),
    'iteration9_root':str(ITER9_ROOT),
    'stress_seeds':I9_ALL_STRESS_SEEDS,
    'locked_2024_opened':False,
    'any_2025_opened':False,
})


{'iteration8_status': 'not_eligible_keep_iteration5_development_reference', 'iteration8_passed_gates': 8, 'iteration8_total_gates': 14, 'iteration9_root': '/content/drive/MyDrive/SkyGuard_AI_GPU/experiments/iteration_09_domain_invariant_calibration', 'stress_seeds': [8023, 9029, 9049], 'locked_2024_opened': False, 'any_2025_opened': False}


## 42. Freeze a domain-invariant residual feature contract

The new ML challenger is not allowed to use absolute temperature/pressure/humidity levels, station spacing, absolute rolling climate levels, absolute neighbour means or station/domain labels. Those fields were useful in-domain but can encode geography and caused brittle cross-climate transfer. Physical range checks remain separate deterministic safety rules.


In [48]:
I9_EXCLUDED_SHORTCUT_FEATURES={
    'temperature_value','pressure_value','humidity_value',
    'temperature_humidity_interaction','pressure_temperature_ratio',
    'nearest_neighbor_km',
}
for sensor in ['temperature','pressure','humidity']:
    I9_EXCLUDED_SHORTCUT_FEATURES.update({
        f'{sensor}_lag1',f'{sensor}_rolling_median_24h',f'{sensor}_ewma_prior',
        f'neighbor_{sensor}_weighted_mean',f'neighbor_{sensor}_median',
    })

I9_RESIDUAL_FEATURES=[feature for feature in FEATURES if feature not in I9_EXCLUDED_SHORTCUT_FEATURES]
assert len(I9_RESIDUAL_FEATURES)>=80
assert not (I9_EXCLUDED_SHORTCUT_FEATURES&set(I9_RESIDUAL_FEATURES))
assert not ({'station_id','i8_domain','cluster','latitude','longitude','elevation_m'}&set(I9_RESIDUAL_FEATURES))
assert {'temperature_slope_3h','temperature_slope_6h','temperature_slope_12h',
        'pressure_cusum_positive','humidity_cusum_negative',
        'regional_agreement_mean','regional_trend_disagreement_mean'}<=set(I9_RESIDUAL_FEATURES)

i9_contract_rows=[]
for feature in FEATURES:
    i9_contract_rows.append({
        'feature':feature,
        'residual_model_allowed':feature in I9_RESIDUAL_FEATURES,
        'exclusion_reason':'absolute climate/domain shortcut' if feature in I9_EXCLUDED_SHORTCUT_FEATURES else '',
    })
i9_feature_ablation=pd.DataFrame(i9_contract_rows)
i9_feature_ablation.to_csv(ITER9_ROOT/'iteration9_feature_ablation_contract.csv',index=False)
print({'full_feature_count':len(FEATURES),'residual_feature_count':len(I9_RESIDUAL_FEATURES),
       'excluded_shortcuts':len(I9_EXCLUDED_SHORTCUT_FEATURES)})
display(i9_feature_ablation.loc[~i9_feature_ablation.residual_model_allowed])


{'full_feature_count': 108, 'residual_feature_count': 87, 'excluded_shortcuts': 21}


,feature,residual_model_allowed,exclusion_reason
0,temperature_value,False,absolute climate/domain shortcut
1,pressure_value,False,absolute climate/domain shortcut
2,humidity_value,False,absolute climate/domain shortcut
7,temperature_humidity_interaction,False,absolute climate/domain shortcut
8,pressure_temperature_ratio,False,absolute climate/domain shortcut
12,nearest_neighbor_km,False,absolute climate/domain shortcut
14,temperature_lag1,False,absolute climate/domain shortcut
18,temperature_rolling_median_24h,False,absolute climate/domain shortcut
21,temperature_ewma_prior,False,absolute climate/domain shortcut
25,pressure_lag1,False,absolute climate/domain shortcut


## 43. Chronological calibration/policy split

The first half of each domain's January–April tune window is used for early stopping and L2 calibration. The later half is used once for threshold selection. Pseudo-unseen DWD stations are excluded from both halves.


In [49]:
def i9_non_holdout_tune(frame):
    mask=frame.i8_scope.eq('tune')
    mask&=~(frame.i8_domain.eq('dwd')&frame.station_id.isin(I8_DWD_HOLDOUTS))
    return frame.loc[mask].copy()

def i9_assign_tune_stage(frame):
    result=frame.copy(); result['i9_tune_stage']=''
    for domain,indices in result.groupby('i8_domain',sort=True).groups.items():
        timestamps=pd.to_datetime(result.loc[indices,'emitted_timestamp_utc'],utc=True)
        cutoff=timestamps.quantile(.50)
        result.loc[indices,'i9_tune_stage']=np.where(timestamps.le(cutoff),'calibration','policy')
    return result

i9_tune=i9_assign_tune_stage(i9_non_holdout_tune(i8_validation))
i9_early=i9_tune.loc[i9_tune.i9_tune_stage.eq('calibration')].copy()
i9_policy=i9_tune.loc[i9_tune.i9_tune_stage.eq('policy')].copy()
assert set(i9_tune.i9_tune_stage)=={'calibration','policy'}
assert not (i9_tune.i8_domain.eq('dwd')&i9_tune.station_id.isin(I8_DWD_HOLDOUTS)).any()
assert i9_early.is_anomaly.sum()>0 and i9_policy.is_anomaly.sum()>0
assert i9_early.is_weather_event.sum()>0 and i9_policy.is_weather_event.sum()>0
display(i9_tune.groupby(['i8_domain','i9_tune_stage'])[['is_anomaly','is_weather_event']].agg(['size','sum']))


is_anomaly      is_weather_event     
                              size  sum             size  sum
i8_domain i9_tune_stage                                      
dwd       calibration        17220  192            17220  684
          policy             17213  208            17213  873
india     calibration        28950  251            28950    0
          policy             28944  194            28944  122

## 44. Train residual-only, domain-balanced LightGBM models

Three regularized seeds are trained for fault and genuine-weather targets. The training weights equalize India/DWD contribution and episode mass. No station ID, domain label or excluded shortcut reaches these models.


In [51]:
I9_LGB_SEEDS=[17,41,67]
i9_models={}; i9_model_history=[]
i9_early_features=i8_enforce_numeric_features(i9_early.copy(),'Iteration 9 early-stop table')

for target in ['fault','weather']:
    y_fit=i8_target(i8_fit,target); y_early=i8_target(i9_early_features,target)
    weights=i8_domain_episode_weights(i8_fit,y_fit)
    models=[]
    for seed in I9_LGB_SEEDS:
        path=ITER9_ROOT/f'iteration9_{target}_residual_lightgbm_seed{seed}.joblib'
        if REUSE_SAVED_MODELS and path.exists():
            model=joblib.load(path)
        else:
            model=LGBMClassifier(
                objective='binary',n_estimators=2400,learning_rate=.018,num_leaves=31,
                min_child_samples=90,subsample=.85,colsample_bytree=.82,
                reg_alpha=2.0,reg_lambda=20.0,random_state=seed,n_jobs=-1,
                verbosity=-1,force_col_wise=True)
            model.fit(
                i8_fit[I9_RESIDUAL_FEATURES],y_fit,sample_weight=weights,
                eval_set=[(i9_early_features[I9_RESIDUAL_FEATURES],y_early)],
                eval_metric='average_precision',
                callbacks=[early_stopping(180,verbose=False),log_evaluation(0)])
            joblib.dump(model,path,compress=3)
        models.append(model)
        i9_model_history.append({
            'target':target,'seed':seed,'features':len(I9_RESIDUAL_FEATURES),
            'fit_rows':len(i8_fit),'fit_positives':int(y_fit.sum()),
            'early_rows':len(i9_early_features),'early_positives':int(y_early.sum()),
            'best_iteration':int(getattr(model,'best_iteration_',0)),
        })
    i9_models[target]=models

pd.DataFrame(i9_model_history).to_csv(ITER9_ROOT/'iteration9_model_training_history.csv',index=False)
display(pd.DataFrame(i9_model_history))


Iteration 9 early-stop table numeric feature contract: PASS | features: 108


,target,seed,features,fit_rows,fit_positives,early_rows,early_positives,best_iteration
0,fault,17,87,286952,5228,46170,443,2217
1,fault,41,87,286952,5228,46170,443,1257
2,fault,67,87,286952,5228,46170,443,2390
3,weather,17,87,286952,6341,46170,684,1350
4,weather,41,87,286952,6341,46170,684,1575
5,weather,67,87,286952,6341,46170,684,1853


## 45. Causal per-station normalization

Each residual score is converted to a robust z-like value relative to that station's previous 30 days. The current/future value never enters its own baseline. New stations fall back to a training-only global median and IQR.


In [52]:
def i9_logit_probability(values):
    clipped=np.clip(np.asarray(values,dtype=float),1e-6,1-1e-6)
    return np.log(clipped/(1-clipped))

def i9_add_residual_scores(frame):
    frame=i8_enforce_numeric_features(frame,'Iteration 9 residual scoring table')
    X=frame[I9_RESIDUAL_FEATURES]
    for target,models in i9_models.items():
        probabilities=np.mean([model.predict_proba(X)[:,1] for model in models],axis=0)
        frame[f'i9_{target}_residual_score']=np.clip(probabilities,0,1).astype(np.float32)
    return frame

i9_fit_reference=i9_add_residual_scores(i8_fit.copy())
I9_SCORE_REFERENCE={}
for target in ['fault','weather']:
    score=f'i9_{target}_residual_score'
    if target=='fault': clean=i9_fit_reference.is_anomaly.eq(0)
    else: clean=i9_fit_reference.is_anomaly.eq(0)&i9_fit_reference.is_weather_event.eq(0)
    values=i9_logit_probability(i9_fit_reference.loc[clean,score])
    median=float(np.nanmedian(values)); q25,q75=np.nanpercentile(values,[25,75])
    scale=float(max((q75-q25)/1.349,1e-3))
    I9_SCORE_REFERENCE[target]={'median':median,'scale':scale}

def i9_causal_station_z(frame,score_col,reference,window=720,min_periods=72):
    result=pd.Series(np.nan,index=frame.index,dtype=float)
    for _,group in frame.groupby('station_id',sort=False):
        ordered=group.sort_values('emitted_timestamp_utc',kind='stable')
        logits=pd.Series(i9_logit_probability(ordered[score_col]),index=ordered.index)
        past=logits.shift(1)
        rolling=past.rolling(window=window,min_periods=min_periods)
        median=rolling.median(); q25=rolling.quantile(.25); q75=rolling.quantile(.75)
        scale=((q75-q25)/1.349).clip(lower=1e-3)
        z=(logits-median)/scale
        fallback=(logits-reference['median'])/reference['scale']
        result.loc[ordered.index]=z.where(z.notna(),fallback).clip(-12,12)
    return result.astype(np.float32)

i8_validation=i9_add_residual_scores(i8_validation)
for target in ['fault','weather']:
    i8_validation[f'i9_{target}_residual_z']=i9_causal_station_z(
        i8_validation,f'i9_{target}_residual_score',I9_SCORE_REFERENCE[target])

display(i8_validation.groupby(['i8_domain','i8_scope'])[
    ['i9_fault_residual_score','i9_fault_residual_z','i9_weather_residual_score','i9_weather_residual_z']
].quantile(.99))
del i9_fit_reference


Iteration 9 residual scoring table numeric feature contract: PASS | features: 108
Iteration 9 residual scoring table numeric feature contract: PASS | features: 108


i9_fault_residual_score  i9_fault_residual_z  \
i8_domain i8_scope                                                     
dwd       confirmation                  0.34824              2.76661   
          discovery                     0.28665              2.73352   
          tune                          0.36383              2.98543   
india     confirmation                  0.23432              4.18840   
          discovery                     0.22211              3.91089   
          tune                          0.20533              3.65374   

                        i9_weather_residual_score  i9_weather_residual_z  
i8_domain i8_scope                                                        
dwd       confirmation                    0.99496                7.46108  
          discovery                       0.98600                6.23384  
          tune                            0.99575                7.27473  
india     confirmation                    0.03151                3.59941  
          discovery                       0.02200                3.30470  
          tune                            0.03105                2.93507

## 46. L2-regularized incident score calibration

The calibrators receive model scores and normalized temporal/spatial consistency signals only. They cannot see domain, station, coordinates, climate cluster or labels at inference. This is calibrated fusion, not max-score inflation.


In [53]:
I9_FAULT_META_FEATURES=[
    'i8_fault_score','i9_fault_residual_score','i8_if_score','i8_lstm_ae_score',
    'i9_fault_residual_z','regional_standardized_disagreement_max',
    'regional_trend_disagreement_mean','regional_agreement_mean',
]
I9_WEATHER_META_FEATURES=[
    'i8_weather_score','i9_weather_residual_score','i9_weather_residual_z',
    'regional_agreement_mean','regional_agreement_min','regional_agreeing_sensor_count',
    'regional_standardized_disagreement_max','regional_trend_disagreement_mean',
    'i9_fault_residual_score',
]
assert not ({'i8_domain','station_id','cluster','latitude','longitude','nearest_neighbor_km'}&
            set(I9_FAULT_META_FEATURES+I9_WEATHER_META_FEATURES))

def i9_fit_calibrator(frame,features,target):
    X=frame[features].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    medians=X.median().fillna(0.0)
    X=X.fillna(medians).to_numpy(np.float64)
    scaler=StandardScaler().fit(X)
    y=i8_target(frame,target)
    weights=i8_domain_episode_weights(frame,y)
    model=LogisticRegression(
        penalty='l2',C=.35,solver='lbfgs',max_iter=2000,random_state=9101)
    model.fit(scaler.transform(X),y,sample_weight=weights)
    return {'features':features,'medians':medians.to_dict(),'scaler':scaler,'model':model}

def i9_calibrated_score(frame,package):
    features=package['features']
    X=frame[features].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    X=X.fillna(pd.Series(package['medians'])).to_numpy(np.float64)
    return package['model'].predict_proba(package['scaler'].transform(X))[:,1].astype(np.float32)

i9_calibration=i9_assign_tune_stage(i9_non_holdout_tune(i8_validation))
i9_calibration=i9_calibration.loc[i9_calibration.i9_tune_stage.eq('calibration')].copy()
i9_calibrators={
    'fault':i9_fit_calibrator(i9_calibration,I9_FAULT_META_FEATURES,'fault'),
    'weather':i9_fit_calibrator(i9_calibration,I9_WEATHER_META_FEATURES,'weather'),
}
joblib.dump(i9_calibrators,ITER9_ROOT/'iteration9_l2_calibrators.joblib',compress=3)

def i9_add_calibrated_scores(frame):
    frame['i9_fault_stack_score']=i9_calibrated_score(frame,i9_calibrators['fault'])
    frame['i9_weather_stack_score']=i9_calibrated_score(frame,i9_calibrators['weather'])
    frame['i9_fault_consensus_score']=np.sqrt(
        np.clip(frame.i8_fault_score,0,1)*np.clip(frame.i9_fault_residual_score,0,1)).astype(np.float32)
    frame['i9_weather_consensus_score']=np.sqrt(
        np.clip(frame.i8_weather_score,0,1)*np.clip(frame.i9_weather_residual_score,0,1)).astype(np.float32)
    return frame

i8_validation=i9_add_calibrated_scores(i8_validation)
i9_coefficients=[]
for target,package in i9_calibrators.items():
    for feature,coefficient in zip(package['features'],package['model'].coef_[0]):
        i9_coefficients.append({'target':target,'feature':feature,'coefficient':float(coefficient)})
i9_coefficients=pd.DataFrame(i9_coefficients)
i9_coefficients.to_csv(ITER9_ROOT/'iteration9_calibrator_coefficients.csv',index=False)
display(i9_coefficients.sort_values(['target','coefficient'],ascending=[True,False]))


,target,feature,coefficient
4,fault,i9_fault_residual_z,0.70204
7,fault,regional_agreement_mean,0.51980
2,fault,i8_if_score,0.51310
1,fault,i9_fault_residual_score,0.15563
0,fault,i8_fault_score,0.15488
5,fault,regional_standardized_disagreement_max,0.03765
6,fault,regional_trend_disagreement_mean,-0.04925
3,fault,i8_lstm_ae_score,-0.22990
10,weather,i9_weather_residual_z,1.32560
9,weather,i9_weather_residual_score,0.81045


## 47. Freeze weather and fault policies on the policy half only

All variants are retained in the frontier. Selection requires minimum precision and false-alarm constraints in both India and DWD; no discovery or confirmation row is searched.


In [54]:
I9_WEATHER_VARIANTS={
    'residual_only':'i9_weather_residual_score',
    'full_residual_consensus':'i9_weather_consensus_score',
    'l2_calibrated_stack':'i9_weather_stack_score',
}
I9_FAULT_VARIANTS={
    'residual_only':'i9_fault_residual_score',
    'full_residual_consensus':'i9_fault_consensus_score',
    'l2_calibrated_stack':'i9_fault_stack_score',
}

def i9_weather_prediction(frame,score_col,threshold):
    agreement=pd.to_numeric(frame.regional_agreement_mean,errors='coerce').fillna(0)
    agreeing=pd.to_numeric(frame.regional_agreeing_sensor_count,errors='coerce').fillna(0)
    hard=frame.hard_rule.astype(bool) if 'hard_rule' in frame else pd.Series(False,index=frame.index)
    return frame[score_col].ge(float(threshold))&agreement.ge(.50)&agreeing.ge(1)&~hard

def i9_weather_metrics(frame,pred,score_col):
    y=frame.is_weather_event.astype(int)
    return {
        'precision':float(precision_score(y,pred,zero_division=0)),
        'recall':float(recall_score(y,pred,zero_division=0)),
        'f1':float(f1_score(y,pred,zero_division=0)),
        'auprc':float(average_precision_score(y,frame[score_col])) if y.sum()>0 else float('nan'),
        'weather_event_rows':int(y.sum()),
        'weather_event_episodes':int(frame.loc[y.eq(1),'episode_id'].replace('',np.nan).nunique()),
        'fault_to_weather_rate':float(pred.loc[frame.is_anomaly.eq(1)].mean()) if frame.is_anomaly.eq(1).any() else 0.0,
    }

i9_policy=i9_assign_tune_stage(i9_non_holdout_tune(i8_validation))
i9_policy=i9_policy.loc[i9_policy.i9_tune_stage.eq('policy')].copy()
i9_weather_frontier=[]
for variant,score_col in I9_WEATHER_VARIANTS.items():
  for threshold in np.linspace(.03,.97,48):
    rows=[]
    for domain in ['india','dwd']:
        part=i9_policy.loc[i9_policy.i8_domain.eq(domain)].copy()
        pred=i9_weather_prediction(part,score_col,threshold)
        rows.append({'domain':domain,**i9_weather_metrics(part,pred,score_col)})
    i9_weather_frontier.append({
        'variant':variant,'score_col':score_col,'threshold':float(threshold),
        'min_precision':min(row['precision'] for row in rows),
        'min_f1':min(row['f1'] for row in rows),
        'mean_f1':float(np.mean([row['f1'] for row in rows])),
        'max_fault_to_weather':max(row['fault_to_weather_rate'] for row in rows),
    })
i9_weather_frontier=pd.DataFrame(i9_weather_frontier)
i9_weather_feasible=i9_weather_frontier.loc[
    i9_weather_frontier.min_precision.ge(I9_MIN_PRECISION)&
    i9_weather_frontier.max_fault_to_weather.le(I9_MAX_FAULT_TO_WEATHER)]
if len(i9_weather_feasible):
    i9_weather_selected=i9_weather_feasible.sort_values(
        ['min_f1','mean_f1','min_precision'],ascending=False).iloc[0]
    I9_WEATHER_TUNE_STATUS='constraints_met'
else:
    i9_weather_frontier['violation']=np.maximum(0,I9_MIN_PRECISION-i9_weather_frontier.min_precision)+10*np.maximum(
        0,i9_weather_frontier.max_fault_to_weather-I9_MAX_FAULT_TO_WEATHER)
    i9_weather_selected=i9_weather_frontier.sort_values(
        ['violation','min_f1','mean_f1'],ascending=[True,False,False]).iloc[0]
    I9_WEATHER_TUNE_STATUS='pareto_fallback_not_promotable'
I9_WEATHER_VARIANT=str(i9_weather_selected.variant)
I9_WEATHER_SCORE_COL=str(i9_weather_selected.score_col)
I9_WEATHER_THRESHOLD=float(i9_weather_selected.threshold)
i9_weather_frontier.to_csv(ITER9_ROOT/'iteration9_weather_policy_frontier.csv',index=False)
display(i9_weather_selected.to_frame('selected_weather'))


,selected_weather
variant,l2_calibrated_stack
score_col,i9_weather_stack_score
threshold,0.69000
min_precision,0.96023
min_f1,0.80000
mean_f1,0.87512
max_fault_to_weather,0.00962


In [55]:
def i9_fault_prediction(frame,score_col,threshold):
    raw=hysteresis(frame,score_col,float(threshold),max(.01,float(threshold)-.06))
    weather=i9_weather_prediction(frame,I9_WEATHER_SCORE_COL,I9_WEATHER_THRESHOLD)
    disagreement=pd.to_numeric(frame.regional_standardized_disagreement_max,errors='coerce').fillna(0)
    override=frame[score_col].ge(min(.995,float(threshold)+.16))&disagreement.ge(2.5)
    return (raw&(~weather|override))|frame.hard_rule.astype(bool)

i9_fault_frontier=[]
for variant,score_col in I9_FAULT_VARIANTS.items():
  for threshold in np.linspace(.08,.97,46):
    rows=[]
    for domain in ['india','dwd']:
        part=i9_policy.loc[i9_policy.i8_domain.eq(domain)].copy()
        part['candidate']=i9_fault_prediction(part,score_col,threshold)
        metric=evaluate(part,score_col,'candidate')
        rows.append({'domain':domain,**metric})
    i9_fault_frontier.append({
        'variant':variant,'score_col':score_col,'threshold':float(threshold),
        'min_precision':min(row['precision'] for row in rows),
        'max_false_alarm':max(row['false_alarm_episodes_per_station_day'] for row in rows),
        'min_point_f1':min(row['f1'] for row in rows),
        'mean_point_f1':float(np.mean([row['f1'] for row in rows])),
        'min_event_f1':min(row['event_f1'] for row in rows),
        'mean_event_f1':float(np.mean([row['event_f1'] for row in rows])),
        'min_event_recall':min(row['event_recall'] for row in rows),
    })
i9_fault_frontier=pd.DataFrame(i9_fault_frontier)
i9_fault_feasible=i9_fault_frontier.loc[
    i9_fault_frontier.min_precision.ge(.82)&i9_fault_frontier.max_false_alarm.le(I9_MAX_FALSE_ALARM)]
if len(i9_fault_feasible):
    i9_fault_selected=i9_fault_feasible.sort_values(
        ['min_event_recall','min_event_f1','mean_point_f1'],ascending=False).iloc[0]
    I9_FAULT_TUNE_STATUS='constraints_met'
else:
    i9_fault_frontier['violation']=np.maximum(0,.82-i9_fault_frontier.min_precision)+10*np.maximum(
        0,i9_fault_frontier.max_false_alarm-I9_MAX_FALSE_ALARM)
    i9_fault_selected=i9_fault_frontier.sort_values(
        ['violation','min_event_recall','mean_event_f1'],ascending=[True,False,False]).iloc[0]
    I9_FAULT_TUNE_STATUS='pareto_fallback_not_promotable'
I9_FAULT_VARIANT=str(i9_fault_selected.variant)
I9_FAULT_SCORE_COL=str(i9_fault_selected.score_col)
I9_FAULT_THRESHOLD=float(i9_fault_selected.threshold)
i9_fault_frontier.to_csv(ITER9_ROOT/'iteration9_fault_policy_frontier.csv',index=False)
display(i9_fault_selected.to_frame('selected_fault'))


,selected_fault
variant,residual_only
score_col,i9_fault_residual_score
threshold,0.89089
min_precision,0.94737
max_false_alarm,0.01385
min_point_f1,0.15111
mean_point_f1,0.16006
min_event_f1,0.33333
mean_event_f1,0.54902
min_event_recall,0.40000


## 48. Main India/DWD confirmation ablation

Iteration 9 is compared with both the deployed Iteration 5 reference and the rejected Iteration 8 challenger. Promotion cannot hide a regression behind an aggregate average.


In [56]:
def i9_metric_bundle(frame,pred,score_col):
    part=frame.copy(); part['candidate']=np.asarray(pred,bool)
    return evaluate(part,score_col,'candidate')

def i9_weak_mean(metric):
    values=[metric['per_fault_episode_recall'].get(name,np.nan) for name in ['bias','drift','frozen_sensor']]
    values=[value for value in values if not pd.isna(value)]
    return float(np.mean(values)) if values else float('nan')

i9_main_rows=[]; i9_family_rows=[]; i9_weather_cluster_rows=[]
for scope in ['discovery','confirmation']:
  domain_masks={
      'india':i8_validation.i8_domain.eq('india'),
      'dwd_all':i8_validation.i8_domain.eq('dwd'),
      'dwd_holdout':i8_validation.i8_domain.eq('dwd')&i8_validation.station_id.isin(I8_DWD_HOLDOUTS),
  }
  for domain,domain_mask in domain_masks.items():
    part=i8_validation.loc[i8_validation.i8_scope.eq(scope)&domain_mask].copy()
    i5_pred=part.i8_baseline_pred.astype(bool)
    i8_pred=i8_candidate_fault_prediction(part,I8_FAULT_SCORE_COL,I8_FAULT_THRESHOLD)
    i9_pred=i9_fault_prediction(part,I9_FAULT_SCORE_COL,I9_FAULT_THRESHOLD)
    i5=i9_metric_bundle(part,i5_pred,'base_score')
    i8m=i9_metric_bundle(part,i8_pred,I8_FAULT_SCORE_COL)
    i9m=i9_metric_bundle(part,i9_pred,I9_FAULT_SCORE_COL)
    i8_weather=i8_weather_metrics(part,i8_weather_prediction(part,I8_WEATHER_THRESHOLD))
    i9_weather=i9_weather_metrics(
        part,i9_weather_prediction(part,I9_WEATHER_SCORE_COL,I9_WEATHER_THRESHOLD),I9_WEATHER_SCORE_COL)
    i9_main_rows.append({
        'scope':scope,'domain':domain,'rows':len(part),
        'iteration5_precision':i5['precision'],'iteration8_precision':i8m['precision'],'iteration9_precision':i9m['precision'],
        'iteration5_point_f1':i5['f1'],'iteration8_point_f1':i8m['f1'],'iteration9_point_f1':i9m['f1'],
        'point_f1_delta_vs_i8':i9m['f1']-i8m['f1'],
        'iteration5_event_f1':i5['event_f1'],'iteration8_event_f1':i8m['event_f1'],'iteration9_event_f1':i9m['event_f1'],
        'event_f1_delta_vs_i8':i9m['event_f1']-i8m['event_f1'],
        'iteration8_weak_recall':i9_weak_mean(i8m),'iteration9_weak_recall':i9_weak_mean(i9m),
        'weak_recall_delta_vs_i8':i9_weak_mean(i9m)-i9_weak_mean(i8m),
        'iteration9_false_alarm':i9m['false_alarm_episodes_per_station_day'],
        'iteration8_weather_f1':i8_weather['f1'],'iteration9_weather_f1':i9_weather['f1'],
        'iteration9_fault_to_weather':i9_weather['fault_to_weather_rate'],
    })
    for family,group in part.loc[part.is_anomaly.eq(1)&part.episode_id.ne('')].groupby('anomaly_type'):
        episode_ids=group.episode_id.unique()
        i8_hits=[]; i9_hits=[]
        for episode_id in episode_ids:
            event=part.episode_id.eq(episode_id)
            i8_hits.append(bool(i8_pred.loc[event].any()))
            i9_hits.append(bool(i9_pred.loc[event].any()))
        i9_family_rows.append({
            'scope':scope,'domain':domain,'anomaly_type':family,'episodes':len(episode_ids),
            'iteration8_episode_recall':float(np.mean(i8_hits)),
            'iteration9_episode_recall':float(np.mean(i9_hits)),
        })
    for cluster,group in part.groupby('cluster'):
        weather=i9_weather_metrics(
            group,i9_weather_prediction(group,I9_WEATHER_SCORE_COL,I9_WEATHER_THRESHOLD),I9_WEATHER_SCORE_COL)
        i9_weather_cluster_rows.append({'scope':scope,'domain':domain,'cluster':cluster,**weather})

i9_main=pd.DataFrame(i9_main_rows)
i9_family=pd.DataFrame(i9_family_rows)
i9_weather_clusters=pd.DataFrame(i9_weather_cluster_rows)
i9_main.to_csv(ITER9_ROOT/'iteration9_multidomain_confirmation.csv',index=False)
i9_family.to_csv(ITER9_ROOT/'iteration9_fault_episode_recall.csv',index=False)
i9_weather_clusters.to_csv(ITER9_ROOT/'iteration9_weather_by_cluster.csv',index=False)
display(i9_main); display(i9_family); display(i9_weather_clusters)


,scope,domain,rows,iteration5_precision,iteration8_precision,iteration9_precision,iteration5_point_f1,iteration8_point_f1,iteration9_point_f1,point_f1_delta_vs_i8,iteration5_event_f1,iteration8_event_f1,iteration9_event_f1,event_f1_delta_vs_i8,iteration8_weak_recall,iteration9_weak_recall,weak_recall_delta_vs_i8,iteration9_false_alarm,iteration8_weather_f1,iteration9_weather_f1,iteration9_fault_to_weather
0,discovery,india,75815,0.83410,0.86957,0.89634,0.43353,0.39900,0.37596,-0.02304,0.61947,0.64000,0.66667,0.02667,0.16667,0.22222,0.05556,0.00425,0.78195,0.70189,0.00809
1,discovery,dwd_all,47121,0.09036,0.80000,0.96000,0.07702,0.32740,0.19316,-0.13424,0.09756,0.37037,0.52941,0.15904,0.41667,0.33333,-0.08333,0.00254,0.85714,0.88571,0.02013
2,discovery,dwd_holdout,11808,0.19318,0.87500,1.00000,0.13386,0.39252,0.38049,-0.01204,0.06897,0.35294,0.60000,0.24706,0.33333,0.33333,0.00000,0.00407,0.86359,0.88937,0.01807
3,confirmation,india,47599,0.83262,0.84211,0.85646,0.55114,0.54936,0.52647,-0.02289,0.59794,0.65263,0.62791,-0.02472,0.55556,0.33333,-0.22222,0.01252,0.34000,0.37821,0.03397
4,confirmation,dwd_all,46667,0.03776,0.66667,0.91176,0.04344,0.22449,0.22262,-0.00187,0.06867,0.30508,0.40000,0.09492,0.41667,0.41667,0.00000,0.00615,0.90836,0.94093,0.01431
5,confirmation,dwd_holdout,11617,0.03587,0.76562,0.98000,0.04255,0.45161,0.48276,0.03115,0.07895,0.42105,0.61538,0.19433,0.55556,0.55556,0.00000,0.00615,0.90385,0.93891,0.00654


,scope,domain,anomaly_type,episodes,iteration8_episode_recall,iteration9_episode_recall
0,discovery,india,bias,3,0.00000,0.33333
1,discovery,india,communication_corruption,2,1.00000,1.00000
2,discovery,india,drift,6,0.16667,0.00000
3,discovery,india,duplicate_packet,5,0.00000,0.00000
4,discovery,india,frozen_sensor,3,0.33333,0.33333
5,discovery,india,multi_sensor_failure,3,1.00000,1.00000
6,discovery,india,noise,4,0.75000,0.75000
7,discovery,india,scaling_error,5,1.00000,1.00000
8,discovery,india,spike,8,1.00000,1.00000
9,discovery,india,sudden_drop,3,1.00000,0.66667


,scope,domain,cluster,precision,recall,f1,auprc,weather_event_rows,weather_event_episodes,fault_to_weather_rate
0,discovery,india,bengaluru,0.00000,0.00000,0.00000,NaN,0,0,0.01481
1,discovery,india,chennai,0.96875,0.34444,0.50820,0.62844,90,1,0.00000
2,discovery,india,delhi,0.95385,0.88571,0.91852,0.94895,70,1,0.00602
3,discovery,india,hyderabad,0.00000,0.00000,0.00000,NaN,0,0,0.00851
4,discovery,dwd_all,dwd_east_continental,0.89655,0.90374,0.90013,0.95708,374,6,0.01613
5,discovery,dwd_all,dwd_north_coastal,0.93911,0.78627,0.85592,0.95213,510,6,0.01852
6,discovery,dwd_all,dwd_south_upland,0.84231,0.85547,0.84884,0.94707,512,6,0.05051
7,discovery,dwd_all,dwd_west_lowland,0.94094,0.94286,0.94190,0.98318,490,6,0.00000
8,discovery,dwd_holdout,dwd_east_continental,0.93333,0.91304,0.92308,0.97113,92,6,0.02020
9,discovery,dwd_holdout,dwd_north_coastal,0.96226,0.79688,0.87179,0.95430,128,6,0.02941


## 49. Residual-only root-cause diagnosis

The diagnosis model uses the same shortcut-free contract. We report macro F1 and per-family recall, not only row accuracy on the easiest detected incidents.


In [57]:
from sklearn.metrics import accuracy_score

I9_ROOT_MODEL_PATH=ITER9_ROOT/'iteration9_residual_root_cause_catboost.cbm'
i9_root_train=i8_fit.loc[i8_fit.is_anomaly.eq(1)].copy()

def i9_root_episode_class_weights(frame):
    keys=frame.episode_id.fillna('').astype(str)
    keys=keys.where(keys.ne(''),'single_'+frame.row_id.astype(str))
    lengths=keys.value_counts()
    weights=keys.map(lambda key:1.0/max(lengths.loc[key],1)).to_numpy(float)
    labels=frame.anomaly_type.astype(str)
    for _,indices in labels.groupby(labels,sort=True).groups.items():
        positions=frame.index.get_indexer(indices)
        positions=positions[positions>=0]
        weights[positions]*=len(frame)/(labels.nunique()*max(weights[positions].sum(),1e-12))
    for _,indices in frame.groupby('i8_domain',sort=True).groups.items():
        positions=frame.index.get_indexer(indices)
        positions=positions[positions>=0]
        weights[positions]*=len(frame)/(frame.i8_domain.nunique()*max(weights[positions].sum(),1e-12))
    return weights/np.mean(weights)

i9_root_weights=i9_root_episode_class_weights(i9_root_train)
i9_root_model=CatBoostClassifier(
    iterations=1200,depth=7,learning_rate=.035,loss_function='MultiClass',
    eval_metric='TotalF1',l2_leaf_reg=16,random_seed=91,task_type='GPU',devices='0',
    verbose=150,allow_writing_files=False)
if REUSE_SAVED_MODELS and I9_ROOT_MODEL_PATH.exists():
    i9_root_model.load_model(I9_ROOT_MODEL_PATH)
else:
    i9_root_model.fit(i9_root_train[I9_RESIDUAL_FEATURES],i9_root_train.anomaly_type.astype(str),
                      sample_weight=i9_root_weights)
    i9_root_model.save_model(I9_ROOT_MODEL_PATH)

i9_root_rows=[]; i9_root_class_rows=[]
for domain in ['india','dwd']:
    part=i8_validation.loc[i8_validation.i8_scope.eq('confirmation')&i8_validation.i8_domain.eq(domain)].copy()
    detected=i9_fault_prediction(part,I9_FAULT_SCORE_COL,I9_FAULT_THRESHOLD)
    for evaluation_slice,mask in {
        'all_fault_rows':part.is_anomaly.eq(1),
        'detected_fault_rows':part.is_anomaly.eq(1)&detected,
    }.items():
        sample=part.loc[mask].copy()
        if not len(sample): continue
        truth=sample.anomaly_type.astype(str).to_numpy()
        prediction=np.asarray(i9_root_model.predict(sample[I9_RESIDUAL_FEATURES])).reshape(-1).astype(str)
        i9_root_rows.append({
            'domain':domain,'slice':evaluation_slice,'rows':len(sample),
            'accuracy':float(accuracy_score(truth,prediction)),
            'macro_f1':float(f1_score(truth,prediction,average='macro',zero_division=0)),
        })
        report=classification_report(truth,prediction,output_dict=True,zero_division=0)
        for label,values in report.items():
            if isinstance(values,dict) and label not in {'accuracy','macro avg','weighted avg'}:
                i9_root_class_rows.append({
                    'domain':domain,'slice':evaluation_slice,'anomaly_type':label,
                    'precision':values['precision'],'recall':values['recall'],'f1':values['f1-score'],
                    'support':values['support'],
                })

i9_root_metrics=pd.DataFrame(i9_root_rows)
i9_root_classes=pd.DataFrame(i9_root_class_rows)
i9_root_metrics.to_csv(ITER9_ROOT/'iteration9_root_cause_metrics.csv',index=False)
i9_root_classes.to_csv(ITER9_ROOT/'iteration9_root_cause_per_class.csv',index=False)
display(i9_root_metrics); display(i9_root_classes)


0:	learn: 0.4928858	total: 310ms	remaining: 6m 11s
150:	learn: 0.8533988	total: 4.44s	remaining: 30.8s
300:	learn: 0.9121313	total: 6.41s	remaining: 19.1s
450:	learn: 0.9368609	total: 8.32s	remaining: 13.8s
600:	learn: 0.9533741	total: 10.2s	remaining: 10.2s
750:	learn: 0.9641762	total: 12.1s	remaining: 7.25s
900:	learn: 0.9727159	total: 15.1s	remaining: 5s
1050:	learn: 0.9777578	total: 18.9s	remaining: 2.68s
1199:	learn: 0.9803344	total: 20.8s	remaining: 0us


,domain,slice,rows,accuracy,macro_f1
0,india,all_fault_rows,471,0.47134,0.46230
1,india,detected_fault_rows,179,0.76536,0.61997
2,dwd,all_fault_rows,489,0.48466,0.53455
3,dwd,detected_fault_rows,62,0.85484,0.53333


,domain,slice,anomaly_type,precision,recall,f1,support
0,india,all_fault_rows,bias,0.70940,0.56463,0.62879,147.00000
1,india,all_fault_rows,communication_corruption,1.00000,0.85714,0.92308,7.00000
2,india,all_fault_rows,drift,0.45238,0.16964,0.24675,112.00000
3,india,all_fault_rows,duplicate_packet,0.00000,0.00000,0.00000,0.00000
4,india,all_fault_rows,frozen_sensor,0.26087,0.24490,0.25263,49.00000
5,india,all_fault_rows,multi_sensor_failure,0.75862,0.92958,0.83544,71.00000
6,india,all_fault_rows,noise,0.20000,0.37879,0.26178,66.00000
7,india,all_fault_rows,scaling_error,0.66667,0.66667,0.66667,3.00000
8,india,all_fault_rows,spike,0.00000,0.00000,0.00000,2.00000
9,india,all_fault_rows,sudden_drop,0.21429,0.60000,0.31579,5.00000


## 50. Three-seed DWD confirmation stress test

The original 8023 validation seed is joined by two new independent seeds. Each replica keeps the same event prevalence and the same chronological scopes; thresholds and calibrators remain frozen. Feature tables are cached one seed at a time.


In [58]:
def i9_prepare_scored_frame(frame):
    frame=i8_enforce_numeric_features(frame,'Iteration 9 stress feature table')
    frame['station_id']=frame.station_id.astype(str)
    frame['emitted_timestamp_utc']=pd.to_datetime(frame.emitted_timestamp_utc,utc=True)
    frame['episode_id']=frame.episode_id.fillna('').astype(str)
    frame['available_to_detector']=pd.to_numeric(frame.available_to_detector).fillna(0).astype(int)
    frame['i8_domain']='dwd'
    frame['i8_scope']=np.select(
        [frame.emitted_timestamp_utc<'2023-05-01',frame.emitted_timestamp_utc<'2023-09-01'],
        ['tune','discovery'],default='confirmation')
    frame=i8_old_scores(frame)
    frame=i8_add_candidate_scores(frame)
    frame['i8_if_score']=i8_isolation_scores(frame)
    frame['i8_lstm_ae_score']=i8_lstm_scores(frame)
    frame['i8_tree_if_score']=np.maximum(
        frame.i8_fault_score,np.sqrt(frame.i8_fault_score*frame.i8_if_score))
    frame['i8_tree_lstm_score']=np.maximum(
        frame.i8_fault_score,np.sqrt(frame.i8_fault_score*frame.i8_lstm_ae_score))
    frame['i8_three_model_score']=np.maximum(
        frame.i8_fault_score,np.cbrt(frame.i8_fault_score*frame.i8_if_score*frame.i8_lstm_ae_score))
    frame=i9_add_residual_scores(frame)
    for target in ['fault','weather']:
        frame[f'i9_{target}_residual_z']=i9_causal_station_z(
            frame,f'i9_{target}_residual_score',I9_SCORE_REFERENCE[target])
    frame['hard_rule']=add_hard_rules(frame).hard_rule.to_numpy(bool)
    return i9_add_calibrated_scores(frame)

def i9_build_stress_seed(seed):
    cache=ITER9_ROOT/f'iteration9_dwd_stress_seed{seed}_features.csv.gz'
    if REUSE_SAVED_MODELS and cache.exists():
        frame=pd.read_csv(cache,low_memory=False)
        print('Reused stress cache',seed)
    else:
        raw=load_dwd(2023)
        curriculum=MultiClimateCurriculum(raw,f'i9_stress_seed{seed}',seed=int(seed))
        curriculum.build_validation(i8_validation_scopes)
        audit=curriculum.validate()
        assert audit['status']=='PASS' and audit['weather_events']==72 and audit['fault_events']==60
        base=i8_base_features(curriculum.frame,f'Iteration 9 stress seed {seed}')
        frame=add_phase10_features(base,i8_profiles)
        frame.to_csv(cache,index=False,compression={'method':'gzip','compresslevel':6})
        del raw,curriculum,base
    return i9_prepare_scored_frame(frame)

def i9_summarize_seed(frame,seed):
    summary=[]; family=[]; clusters=[]
    for scope in ['discovery','confirmation']:
      for domain,mask in {
          'dwd_all':pd.Series(True,index=frame.index),
          'dwd_holdout':frame.station_id.isin(I8_DWD_HOLDOUTS),
      }.items():
        part=frame.loc[frame.i8_scope.eq(scope)&mask].copy()
        i8_pred=i8_candidate_fault_prediction(part,I8_FAULT_SCORE_COL,I8_FAULT_THRESHOLD)
        i9_pred=i9_fault_prediction(part,I9_FAULT_SCORE_COL,I9_FAULT_THRESHOLD)
        i8m=i9_metric_bundle(part,i8_pred,I8_FAULT_SCORE_COL)
        i9m=i9_metric_bundle(part,i9_pred,I9_FAULT_SCORE_COL)
        weather=i9_weather_metrics(
            part,i9_weather_prediction(part,I9_WEATHER_SCORE_COL,I9_WEATHER_THRESHOLD),I9_WEATHER_SCORE_COL)
        summary.append({
            'seed':seed,'scope':scope,'domain':domain,'rows':len(part),
            'iteration8_precision':i8m['precision'],'iteration9_precision':i9m['precision'],
            'iteration8_point_f1':i8m['f1'],'iteration9_point_f1':i9m['f1'],
            'point_f1_delta_vs_i8':i9m['f1']-i8m['f1'],
            'iteration8_event_f1':i8m['event_f1'],'iteration9_event_f1':i9m['event_f1'],
            'event_f1_delta_vs_i8':i9m['event_f1']-i8m['event_f1'],
            'iteration8_weak_recall':i9_weak_mean(i8m),'iteration9_weak_recall':i9_weak_mean(i9m),
            'iteration9_false_alarm':i9m['false_alarm_episodes_per_station_day'],
            'iteration9_weather_f1':weather['f1'],
            'iteration9_fault_to_weather':weather['fault_to_weather_rate'],
            'iteration9_drift_episode_recall':float(i9m['per_fault_episode_recall'].get('drift',np.nan)),
        })
        for anomaly_type,value in i9m['per_fault_episode_recall'].items():
            family.append({'seed':seed,'scope':scope,'domain':domain,'anomaly_type':anomaly_type,
                           'iteration9_episode_recall':value})
        for cluster,group in part.groupby('cluster'):
            metric=i9_weather_metrics(
                group,i9_weather_prediction(group,I9_WEATHER_SCORE_COL,I9_WEATHER_THRESHOLD),I9_WEATHER_SCORE_COL)
            clusters.append({'seed':seed,'scope':scope,'domain':domain,'cluster':cluster,**metric})
    return summary,family,clusters

i9_stress_summary=[]; i9_stress_family=[]; i9_stress_clusters=[]
base_stress=i8_validation.loc[i8_validation.i8_domain.eq('dwd')].copy()
rows,families,clusters=i9_summarize_seed(base_stress,I9_BASE_STRESS_SEED)
i9_stress_summary+=rows; i9_stress_family+=families; i9_stress_clusters+=clusters

if RUN_ITER9_STRESS:
    for seed in I9_NEW_STRESS_SEEDS:
        stress=i9_build_stress_seed(seed)
        rows,families,clusters=i9_summarize_seed(stress,seed)
        i9_stress_summary+=rows; i9_stress_family+=families; i9_stress_clusters+=clusters
        del stress

i9_stress_summary=pd.DataFrame(i9_stress_summary)
i9_stress_family=pd.DataFrame(i9_stress_family)
i9_stress_clusters=pd.DataFrame(i9_stress_clusters)
i9_stress_summary.to_csv(ITER9_ROOT/'iteration9_multiseed_stress.csv',index=False)
i9_stress_family.to_csv(ITER9_ROOT/'iteration9_multiseed_fault_recall.csv',index=False)
i9_stress_clusters.to_csv(ITER9_ROOT/'iteration9_multiseed_weather_by_cluster.csv',index=False)
display(i9_stress_summary); display(i9_stress_family); display(i9_stress_clusters)


Iteration 9 stress seed 9029 50000 / 139716
Iteration 9 stress seed 9029 100000 / 139716
Iteration 9 stress feature table numeric feature contract: PASS | features: 108
Candidate scoring table numeric feature contract: PASS | features: 108
Iteration 9 residual scoring table numeric feature contract: PASS | features: 108
Iteration 9 stress seed 9049 50000 / 139716
Iteration 9 stress seed 9049 100000 / 139716
Iteration 9 stress feature table numeric feature contract: PASS | features: 108
Candidate scoring table numeric feature contract: PASS | features: 108
Iteration 9 residual scoring table numeric feature contract: PASS | features: 108


,seed,scope,domain,rows,iteration8_precision,iteration9_precision,iteration8_point_f1,iteration9_point_f1,point_f1_delta_vs_i8,iteration8_event_f1,iteration9_event_f1,event_f1_delta_vs_i8,iteration8_weak_recall,iteration9_weak_recall,iteration9_false_alarm,iteration9_weather_f1,iteration9_fault_to_weather,iteration9_drift_episode_recall
0,8023,discovery,dwd_all,47121,0.80000,0.96000,0.32740,0.19316,-0.13424,0.37037,0.52941,0.15904,0.41667,0.33333,0.00254,0.88571,0.02013,0.25000
1,8023,discovery,dwd_holdout,11808,0.87500,1.00000,0.39252,0.38049,-0.01204,0.35294,0.60000,0.24706,0.33333,0.33333,0.00407,0.88937,0.01807,0.00000
2,8023,confirmation,dwd_all,46667,0.66667,0.91176,0.22449,0.22262,-0.00187,0.30508,0.40000,0.09492,0.41667,0.41667,0.00615,0.94093,0.01431,0.00000
3,8023,confirmation,dwd_holdout,11617,0.76562,0.98000,0.45161,0.48276,0.03115,0.42105,0.61538,0.19433,0.55556,0.55556,0.00615,0.93891,0.00654,0.00000
4,9029,discovery,dwd_all,47121,0.88889,1.00000,0.24000,0.21955,-0.02045,0.46512,0.46667,0.00155,0.41667,0.25000,0.00152,0.91729,0.01156,0.25000
5,9029,discovery,dwd_holdout,11808,0.94737,1.00000,0.36000,0.31250,-0.04750,0.66667,0.57143,-0.09524,0.33333,0.33333,0.00000,0.91187,0.01235,0.00000
6,9029,confirmation,dwd_all,46667,0.70755,0.91304,0.27778,0.25050,-0.02728,0.31373,0.34286,0.02913,0.16667,0.16667,0.00461,0.93741,0.01382,0.00000
7,9029,confirmation,dwd_holdout,11617,0.76786,0.97619,0.46995,0.48521,0.01526,0.47059,0.50000,0.02941,0.16667,0.16667,0.00205,0.93688,0.00000,0.00000
8,9049,discovery,dwd_all,47121,0.75000,0.94737,0.25920,0.24283,-0.01637,0.32143,0.47059,0.14916,0.33333,0.25000,0.00305,0.89818,0.01934,0.00000
9,9049,discovery,dwd_holdout,11808,0.80000,0.92857,0.68293,0.69333,0.01041,0.40000,0.66667,0.26667,0.50000,0.50000,0.00203,0.88711,0.04255,NaN


,seed,scope,domain,anomaly_type,iteration9_episode_recall
0,8023,discovery,dwd_all,bias,0.00000
1,8023,discovery,dwd_all,drift,0.25000
2,8023,discovery,dwd_all,frozen_sensor,0.75000
3,8023,discovery,dwd_all,noise,0.50000
4,8023,discovery,dwd_all,spike,0.75000
5,8023,discovery,dwd_holdout,bias,0.00000
6,8023,discovery,dwd_holdout,drift,0.00000
7,8023,discovery,dwd_holdout,frozen_sensor,1.00000
8,8023,discovery,dwd_holdout,noise,1.00000
9,8023,confirmation,dwd_all,bias,0.50000


,seed,scope,domain,cluster,precision,recall,f1,auprc,weather_event_rows,weather_event_episodes,fault_to_weather_rate
0,8023,discovery,dwd_all,dwd_east_continental,0.89655,0.90374,0.90013,0.95708,374,6,0.01613
1,8023,discovery,dwd_all,dwd_north_coastal,0.93911,0.78627,0.85592,0.95213,510,6,0.01852
2,8023,discovery,dwd_all,dwd_south_upland,0.84231,0.85547,0.84884,0.94707,512,6,0.05051
3,8023,discovery,dwd_all,dwd_west_lowland,0.94094,0.94286,0.94190,0.98318,490,6,0.00000
4,8023,discovery,dwd_holdout,dwd_east_continental,0.93333,0.91304,0.92308,0.97113,92,6,0.02020
5,8023,discovery,dwd_holdout,dwd_north_coastal,0.96226,0.79688,0.87179,0.95430,128,6,0.02941
6,8023,discovery,dwd_holdout,dwd_south_upland,0.79851,0.83594,0.81679,0.92366,128,6,0.00000
7,8023,discovery,dwd_holdout,dwd_west_lowland,0.95122,0.96694,0.95902,0.98706,121,6,0.00000
8,8023,confirmation,dwd_all,dwd_east_continental,0.90991,0.94393,0.92661,0.97777,428,6,0.00000
9,8023,confirmation,dwd_all,dwd_north_coastal,0.94209,0.94843,0.94525,0.98373,446,6,0.03150


## 51. Seed-level confidence intervals

Intervals are bootstrapped across independent injection seeds, not across correlated rows. Three seeds are still a small sample, so the interval is evidence of stability—not a claim of final population certainty.


In [59]:
def i9_seed_interval(values,seed=9191,repetitions=5000):
    values=np.asarray(values,dtype=float); values=values[np.isfinite(values)]
    if not len(values): return {'mean':float('nan'),'ci_lower':float('nan'),'ci_upper':float('nan'),'seeds':0}
    rng=np.random.default_rng(seed)
    draws=rng.choice(values,size=(repetitions,len(values)),replace=True).mean(axis=1)
    return {'mean':float(values.mean()),'ci_lower':float(np.quantile(draws,.025)),
            'ci_upper':float(np.quantile(draws,.975)),'seeds':len(values)}

i9_ci_rows=[]
metrics=['iteration9_precision','iteration9_point_f1','iteration9_event_f1',
         'iteration9_weak_recall','iteration9_false_alarm','iteration9_weather_f1',
         'iteration9_fault_to_weather','iteration9_drift_episode_recall']
for (scope,domain),group in i9_stress_summary.groupby(['scope','domain']):
    for metric in metrics:
        i9_ci_rows.append({'scope':scope,'domain':domain,'metric':metric,
                           **i9_seed_interval(group[metric])})
i9_confidence=pd.DataFrame(i9_ci_rows)
i9_confidence.to_csv(ITER9_ROOT/'iteration9_seed_confidence_intervals.csv',index=False)
display(i9_confidence)


,scope,domain,metric,mean,ci_lower,ci_upper,seeds
0,confirmation,dwd_all,iteration9_precision,0.92748,0.91176,0.95763,3
1,confirmation,dwd_all,iteration9_point_f1,0.28410,0.22262,0.37919,3
2,confirmation,dwd_all,iteration9_event_f1,0.45275,0.34286,0.61538,3
3,confirmation,dwd_all,iteration9_weak_recall,0.41667,0.16667,0.66667,3
4,confirmation,dwd_all,iteration9_false_alarm,0.00478,0.00359,0.00615,3
5,confirmation,dwd_all,iteration9_weather_f1,0.93543,0.92796,0.94093,3
6,confirmation,dwd_all,iteration9_fault_to_weather,0.01705,0.01382,0.02301,3
7,confirmation,dwd_all,iteration9_drift_episode_recall,0.16667,0.00000,0.50000,3
8,confirmation,dwd_holdout,iteration9_precision,0.98540,0.97619,1.00000,3
9,confirmation,dwd_holdout,iteration9_point_f1,0.46154,0.41667,0.48521,3


## 52. Strict promotion decision and SIH impact receipt

Passing means “eligible for one locked 2024 confirmation,” not automatically deployed. Any failed gate keeps Iteration 5 as the accepted project model.


In [60]:
i9_confirmation=i9_main.loc[i9_main.scope.eq('confirmation')].copy()
i9_supported_weather=i9_weather_clusters.loc[
    i9_weather_clusters.scope.eq('confirmation')&i9_weather_clusters.weather_event_rows.gt(0)].copy()
i9_stress_confirmation=i9_stress_summary.loc[i9_stress_summary.scope.eq('confirmation')].copy()
i9_stress_supported_weather=i9_stress_clusters.loc[
    i9_stress_clusters.scope.eq('confirmation')&i9_stress_clusters.weather_event_rows.gt(0)].copy()
i9_root_confirmation=i9_root_metrics.loc[i9_root_metrics.slice.eq('detected_fault_rows')]

I9_PROMOTION_GATES={
    'policy_fault_constraints':I9_FAULT_TUNE_STATUS=='constraints_met',
    'policy_weather_constraints':I9_WEATHER_TUNE_STATUS=='constraints_met',
    'main_confirmation_precision':bool(i9_confirmation.iteration9_precision.ge(I9_MIN_PRECISION).all()),
    'main_confirmation_false_alarm':bool(i9_confirmation.iteration9_false_alarm.le(I9_MAX_FALSE_ALARM).all()),
    'no_main_point_f1_regression_vs_i8':bool(i9_confirmation.point_f1_delta_vs_i8.ge(0).all()),
    'no_main_event_f1_regression_vs_i8':bool(i9_confirmation.event_f1_delta_vs_i8.ge(0).all()),
    'no_main_weak_recall_regression_vs_i8':bool(i9_confirmation.weak_recall_delta_vs_i8.ge(0).all()),
    'india_confirmation_weather_f1_at_least_065':bool(
        i9_confirmation.loc[i9_confirmation.domain.eq('india'),'iteration9_weather_f1'].ge(.65).all()),
    'dwd_confirmation_weather_f1_at_least_075':bool(
        i9_confirmation.loc[i9_confirmation.domain.str.startswith('dwd'),'iteration9_weather_f1'].ge(.75).all()),
    'worst_supported_climate_weather_f1_at_least_065':bool(
        len(i9_supported_weather)>0 and i9_supported_weather.f1.min()>=.65),
    'main_fault_to_weather_at_most_001':bool(
        i9_confirmation.iteration9_fault_to_weather.le(I9_MAX_FAULT_TO_WEATHER).all()),
    'three_stress_seeds_completed':bool(set(i9_stress_summary.seed.astype(int))==set(I9_ALL_STRESS_SEEDS)),
    'stress_confirmation_precision':bool(i9_stress_confirmation.iteration9_precision.ge(I9_MIN_PRECISION).all()),
    'stress_confirmation_false_alarm':bool(i9_stress_confirmation.iteration9_false_alarm.le(I9_MAX_FALSE_ALARM).all()),
    'stress_no_point_f1_regression_vs_i8':bool(i9_stress_confirmation.point_f1_delta_vs_i8.ge(0).all()),
    'stress_no_event_f1_regression_vs_i8':bool(i9_stress_confirmation.event_f1_delta_vs_i8.ge(0).all()),
    'stress_weather_f1_at_least_075':bool(i9_stress_confirmation.iteration9_weather_f1.ge(.75).all()),
    'stress_fault_to_weather_at_most_001':bool(
        i9_stress_confirmation.iteration9_fault_to_weather.le(I9_MAX_FAULT_TO_WEATHER).all()),
    'stress_drift_episode_recall_at_least_050':bool(
        i9_stress_confirmation.iteration9_drift_episode_recall.fillna(0).mean()>=.50),
    'root_cause_detected_macro_f1_at_least_070':bool(
        len(i9_root_confirmation)>0 and i9_root_confirmation.macro_f1.ge(.70).all()),
    'root_cause_detected_accuracy_at_least_080':bool(
        len(i9_root_confirmation)>0 and i9_root_confirmation.accuracy.ge(.80).all()),
    'shortcut_contract_clean':bool(not (I9_EXCLUDED_SHORTCUT_FEATURES&set(I9_RESIDUAL_FEATURES))),
    'locked_years_sealed':True,
}
I9_PROMOTED=all(I9_PROMOTION_GATES.values())
I9_STATUS='eligible_for_one_locked_2024_confirmation' if I9_PROMOTED else 'not_eligible_keep_iteration5_deployment'

i9_feature_importance=pd.DataFrame({
    'feature':I9_RESIDUAL_FEATURES,
    'importance':np.mean([model.feature_importances_ for model in i9_models['fault']],axis=0),
}).sort_values('importance',ascending=False).reset_index(drop=True)
i9_feature_importance.to_csv(ITER9_ROOT/'iteration9_residual_feature_importance.csv',index=False)

i9_feature_contract={
    'observation_inputs':['temperature','pressure','relative_humidity'],
    'full_feature_count':len(FEATURES),'residual_feature_count':len(I9_RESIDUAL_FEATURES),
    'residual_features':I9_RESIDUAL_FEATURES,
    'excluded_shortcuts':sorted(I9_EXCLUDED_SHORTCUT_FEATURES),
    'calibrator_features':{'fault':I9_FAULT_META_FEATURES,'weather':I9_WEATHER_META_FEATURES},
    'causal_score_normalization':{'window_rows':720,'min_history_rows':72,'uses_current_or_future_in_baseline':False},
    'development_years':[2022,2023],'dwd_2024_opened':False,'any_2025_opened':False,
    'forbidden':['domain','station_id','climate_cluster','coordinates','future_observation','locked_year_labels'],
}
(ITER9_ROOT/'iteration9_feature_contract.json').write_text(json.dumps(i9_feature_contract,indent=2))

i9_integrity={
    'iteration8_result_sha256':sha256_file(I9_BASE_RESULT_PATH),
    'development_years':[2022,2023],'dwd_2024_opened':False,'noaa_2024_opened':False,'any_2025_opened':False,
    'threshold_selection_scope':'later half of non-holdout tune rows only',
    'calibration_scope':'earlier half of non-holdout tune rows only',
    'discovery_or_confirmation_used_for_selection':False,
    'stress_seeds':I9_ALL_STRESS_SEEDS,'stress_event_prevalence_held_constant':True,
    'domain_or_station_identifier_used_by_model':False,
}
(ITER9_ROOT/'iteration9_integrity_receipt.json').write_text(json.dumps(i9_integrity,indent=2))

result9={
    'iteration':'09_domain_invariant_calibration_multiseed_stress',
    'status':I9_STATUS,'promoted':I9_PROMOTED,'device':DEVICE,
    'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'base_deployment':'Iteration 5','iteration8_promoted':bool(i9_base_result['promoted']),
    'development_years':[2022,2023],'locked_2024_opened':False,'any_2025_opened':False,
    'models':['residual-only LightGBM seeds 17/41/67','L2 fault calibrator','L2 weather calibrator',
              'clean-only Isolation Forest (Iteration 8 frozen)','causal LSTM autoencoder (Iteration 8 frozen)',
              'residual-only CatBoost root-cause classifier'],
    'selected_policy':{
        'fault_variant':I9_FAULT_VARIANT,'fault_score_col':I9_FAULT_SCORE_COL,'fault_threshold':I9_FAULT_THRESHOLD,
        'weather_variant':I9_WEATHER_VARIANT,'weather_score_col':I9_WEATHER_SCORE_COL,
        'weather_threshold':I9_WEATHER_THRESHOLD,
    },
    'feature_contract':{'full':len(FEATURES),'residual':len(I9_RESIDUAL_FEATURES),
                        'excluded_shortcuts':sorted(I9_EXCLUDED_SHORTCUT_FEATURES)},
    'stress_seeds':I9_ALL_STRESS_SEEDS,
    'promotion_gates':I9_PROMOTION_GATES,
    'passed_gates':sum(bool(value) for value in I9_PROMOTION_GATES.values()),
    'total_gates':len(I9_PROMOTION_GATES),
    'multidomain_confirmation':i9_main.to_dict('records'),
    'root_cause_metrics':i9_root_metrics.to_dict('records'),
    'stress_confirmation':i9_stress_confirmation.to_dict('records'),
    'next_decision':'audit then run one locked DWD 2024 confirmation' if I9_PROMOTED else 'retain Iteration 5 and inspect failed Iteration 9 gates',
}
(ITER9_ROOT/'iteration9_result_block.json').write_text(json.dumps(result9,indent=2,default=float))

i9_problem_coverage={
    'problem_statement_id':'SIH26073',
    'observation_inputs':['temperature','pressure','relative_humidity'],
    'iteration9_directly_validates':[
        'fault anomaly detection','genuine weather versus sensor fault separation',
        'spike detection','frozen sensor detection','bias and drift detection',
        'multivariate and neighbour consistency','false alarm control',
        'root-cause classification','confidence calibration','causal real-time inference',
        'unseen station and multi-climate transfer',
    ],
    'existing_end_to_end_modules_preserved':[
        'streaming API and replay','communication gap and duplicate detection',
        'alert severity and explanation','sensor health and maintenance recommendation',
        'advisory corrected value','offline dashboard and incident reports',
    ],
    'promotion_requires_all_functional_gates':True,
    'locked_test_required_after_development_pass':True,
}
(ITER9_ROOT/'iteration9_problem_coverage.json').write_text(json.dumps(i9_problem_coverage,indent=2))

print(json.dumps(result9,indent=2,default=float))
print('\nSEND BACK THESE ITERATION 9 FILES:')
for filename in [
    'iteration9_result_block.json','iteration9_integrity_receipt.json','iteration9_feature_contract.json',
    'iteration9_problem_coverage.json','iteration9_model_training_history.csv',
    'iteration9_multidomain_confirmation.csv','iteration9_fault_episode_recall.csv',
    'iteration9_weather_by_cluster.csv','iteration9_root_cause_metrics.csv',
    'iteration9_root_cause_per_class.csv','iteration9_multiseed_stress.csv',
    'iteration9_multiseed_fault_recall.csv','iteration9_multiseed_weather_by_cluster.csv',
    'iteration9_seed_confidence_intervals.csv','iteration9_calibrator_coefficients.csv',
    'iteration9_residual_feature_importance.csv','iteration9_weather_policy_frontier.csv',
    'iteration9_fault_policy_frontier.csv',
]: print(ITER9_ROOT/filename)


{
  "iteration": "09_domain_invariant_calibration_multiseed_stress",
  "status": "not_eligible_keep_iteration5_deployment",
  "promoted": false,
  "device": "cuda",
  "gpu": "Tesla T4",
  "base_deployment": "Iteration 5",
  "iteration8_promoted": false,
  "development_years": [
    2022,
    2023
  ],
  "locked_2024_opened": false,
  "any_2025_opened": false,
  "models": [
    "residual-only LightGBM seeds 17/41/67",
    "L2 fault calibrator",
    "L2 weather calibrator",
    "clean-only Isolation Forest (Iteration 8 frozen)",
    "causal LSTM autoencoder (Iteration 8 frozen)",
    "residual-only CatBoost root-cause classifier"
  ],
  "selected_policy": {
    "fault_variant": "residual_only",
    "fault_score_col": "i9_fault_residual_score",
    "fault_threshold": 0.890888888888889,
    "weather_variant": "l2_calibrated_stack",
    "weather_score_col": "i9_weather_stack_score",
    "weather_threshold": 0.6900000000000001
  },
  "feature_contract": {
    "full": 108,
    "residual": 87,

## Iteration 9 stop rule

Return the eighteen result files printed above. Do not open DWD/NOAA 2024 or any 2025 file even if the notebook reports eligibility. A separate audit must first reconcile JSON, CSV, seed counts, root-cause metrics and all promotion gates.
